# 01. 원본 데이터 품질 점검 및 전처리

이 노트북은 서울시 공공자전거 원본 데이터의 구조와 품질을 점검하고, 후속 분석에 사용할 정제 데이터를 생성한다.

주요 수행 내용은 다음과 같다.

1. `config.yaml`에서 분석 기간과 데이터 경로를 불러온다.
2. `data/raw/bike_usage/`의 공공자전거 이용정보 파일 목록과 수집 기간을 확인한다.
3. `data/raw/station/`의 대여소 마스터 데이터 구조를 확인한다.
4. 파일별 행 수, 컬럼 구성, 데이터 유형 및 인코딩을 점검한다.
5. 시간 변수와 대여소 ID를 분석 가능한 형식으로 표준화한다.
6. 결측치, 중복값, 비정상 시간값 및 비정상 GPS 좌표를 점검한다.
7. 공공자전거 이용정보와 대여소 마스터 간 대여소 ID 정합성을 확인한다.
8. 후속 공간 군집화에 사용할 정제 데이터를 `data/processed/`에 저장한다.

본 노트북에서는 K-Means 클러스터링과 수요예측 모델 학습을 수행하지 않는다. 공간 군집화는 `02_spatial_clustering_analysis.ipynb`, 모델 학습과 평가는 `03_modeling_evaluation.ipynb`에서 수행한다.


## 0. 패키지 불러오기 및 프로젝트 경로 설정

In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import yaml


def find_project_root(start: Path | None = None) -> Path:
    """config.yaml이 위치한 프로젝트 루트 경로를 찾는다."""
    
    current_path = (start or Path.cwd()).resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "config.yaml을 찾지 못했다. "
        "VS Code에서 프로젝트 루트 폴더를 연 뒤 다시 실행해야 한다."
    )


ROOT = find_project_root()

print(f"PROJECT ROOT: {ROOT}")


PROJECT ROOT: C:\Users\user\Desktop\주진호\05.ICT 인턴십\02.공공자전거


## 1. 설정 파일 불러오기

프로젝트 전반에서 사용하는 분석 기간, 데이터 경로 및 모델 설정은 `config.yaml`에서 통합 관리한다.


In [2]:
def load_config() -> Dict:
    """프로젝트 설정 파일을 불러온다."""

    config_path = ROOT / "config.yaml"

    if not config_path.exists():
        raise FileNotFoundError(
            f"설정 파일이 존재하지 않는다: {config_path}"
        )

    with open(config_path, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if not isinstance(config, dict):
        raise ValueError("config.yaml의 설정 구조가 올바르지 않다.")

    return config


cfg = load_config()

print("[분석 기간]")
print(f"{cfg['period']['start']} ~ {cfg['period']['end']}")

print("\n[데이터 경로]")
print(f"공공자전거 이용정보: {cfg['data']['bike_usage_dir']}")
print(f"대여소 마스터: {cfg['data']['station_dir']}")
print(f"전처리 데이터 저장 경로: {cfg['data']['processed_dir']}")


[분석 기간]
2023-01-01 ~ 2024-12-31

[데이터 경로]
공공자전거 이용정보: data/raw/bike_usage
대여소 마스터: data/raw/station
전처리 데이터 저장 경로: data/processed


## 2. 원천 데이터 읽기 및 컬럼 표준화 함수 정의

공공자전거 원천 데이터는 CSV와 Excel 형식이 혼재할 수 있으며, CSV 파일의 문자 인코딩도 파일별로 다를 수 있다. 이에 지원 가능한 원천 파일을 탐색하고, 여러 인코딩을 순차적으로 적용하여 데이터를 안정적으로 불러오는 함수를 정의한다.

또한 파일마다 다르게 표현된 컬럼명과 대여소 ID를 공통 형식으로 표준화하여 후속 품질 점검과 데이터 결합에 활용한다.


In [3]:
SUPPORTED_SUFFIXES = {".csv", ".xlsx", ".xls"}
CSV_ENCODINGS = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]


def get_data_files(folder: Path) -> List[Path]:
    """폴더에서 분석 가능한 원천 데이터 파일 목록을 반환한다."""

    if not folder.exists():
        raise FileNotFoundError(f"데이터 폴더가 존재하지 않는다: {folder}")

    files = sorted(
        path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )

    if not files:
        raise FileNotFoundError(
            f"분석 가능한 CSV 또는 Excel 파일이 존재하지 않는다: {folder}"
        )

    return files


def read_table(path: Path, nrows: int | None = None) -> pd.DataFrame:
    """CSV 또는 Excel 파일을 불러온다."""

    suffix = path.suffix.lower()

    if suffix == ".csv":
        for encoding in CSV_ENCODINGS:
            try:
                df = pd.read_csv(
                    path,
                    encoding=encoding,
                    low_memory=False,
                    nrows=nrows,
                )
                df.attrs["source_encoding"] = encoding
                return df

            except UnicodeDecodeError:
                continue

        raise UnicodeDecodeError(
            "unknown",
            b"",
            0,
            1,
            f"지원하는 인코딩으로 파일을 읽지 못했다: {path.name}",
        )

    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, nrows=nrows)
        df.attrs["source_encoding"] = "excel"
        return df

    raise ValueError(f"지원하지 않는 파일 형식이다: {path.suffix}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """컬럼명의 공백과 줄바꿈을 제거하고 표준화한다."""

    rename_map = {}

    for column in df.columns:
        normalized = str(column).strip()
        normalized = normalized.replace("\n", "").replace("\r", "")
        normalized = re.sub(r"\s+", "_", normalized)
        rename_map[column] = normalized

    return df.rename(columns=rename_map)


def find_column(
    df: pd.DataFrame,
    candidates: List[str],
    required: bool = True,
) -> Optional[str]:
    """후보 이름과 일치하는 실제 컬럼명을 탐색한다."""

    columns = list(df.columns)
    lowercase_map = {
        str(column).lower(): column
        for column in columns
    }

    for candidate in candidates:
        key = candidate.lower()

        if key in lowercase_map:
            return lowercase_map[key]

    for candidate in candidates:
        for column in columns:
            if candidate.lower() in str(column).lower():
                return column

    if required:
        raise KeyError(
            "필수 컬럼을 찾지 못했다.\n"
            f"후보 컬럼: {candidates}\n"
            f"실제 컬럼: {columns[:50]}"
        )

    return None


def clean_station_id(series: pd.Series) -> pd.Series:
    """대여소 ID를 숫자로 구성된 문자열 형식으로 표준화한다."""

    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.extract(r"(\d+)", expand=False)
    )

## 3. 원천 데이터 파일 목록 및 분석 기간 점검

공공자전거 이용정보와 대여소 마스터 파일의 목록, 파일 형식 및 용량을 확인한다.

공공자전거 이용정보는 월별 파일명에서 수집 연월을 추출한 뒤, `config.yaml`에 설정된 분석 기간과 비교한다. 이를 통해 누락 월, 중복 월 및 분석 범위 밖의 파일이 존재하는지 점검하고 정식 분석 데이터의 버전을 확정한다.


In [4]:
def extract_year_month(path: Path) -> Optional[pd.Period]:
    """파일명에서 연월 정보를 추출한다."""

    filename = path.stem

    four_digit_match = re.search(
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
        filename,
    )

    if four_digit_match:
        year, month = four_digit_match.groups()
        return pd.Period(f"{year}-{month}", freq="M")

    two_digit_match = re.search(
        r"(?<!\d)(\d{2})(0[1-9]|1[0-2])(?!\d)",
        filename,
    )

    if two_digit_match:
        year, month = two_digit_match.groups()
        return pd.Period(f"20{year}-{month}", freq="M")

    return None


def build_file_inventory(
    files: List[Path],
    data_type: str,
) -> pd.DataFrame:
    """원천 데이터 파일 목록을 표 형태로 정리한다."""

    records = []

    for path in files:
        records.append(
            {
                "data_type": data_type,
                "file_name": path.name,
                "file_extension": path.suffix.lower(),
                "file_size_mb": round(
                    path.stat().st_size / (1024**2),
                    2,
                ),
                "year_month": extract_year_month(path),
            }
        )

    return pd.DataFrame(records)


bike_usage_dir = ROOT / cfg["data"]["bike_usage_dir"]
station_dir = ROOT / cfg["data"]["station_dir"]
processed_dir = ROOT / cfg["data"]["processed_dir"]

bike_usage_files = get_data_files(bike_usage_dir)
station_files = get_data_files(station_dir)

bike_file_inventory = build_file_inventory(
    bike_usage_files,
    data_type="bike_usage",
)

station_file_inventory = build_file_inventory(
    station_files,
    data_type="station",
)

display(bike_file_inventory)
display(station_file_inventory)


expected_months = pd.period_range(
    start=pd.Timestamp(cfg["period"]["start"]).to_period("M"),
    end=pd.Timestamp(cfg["period"]["end"]).to_period("M"),
    freq="M",
)

observed_month_values = (
    bike_file_inventory["year_month"]
    .dropna()
    .tolist()
)

observed_months = pd.PeriodIndex(
    sorted(set(observed_month_values)),
    freq="M",
)

missing_months = expected_months.difference(observed_months)
outside_months = observed_months.difference(expected_months)

duplicate_months = (
    bike_file_inventory
    .dropna(subset=["year_month"])
    .groupby("year_month")
    .size()
)

duplicate_months = duplicate_months[
    duplicate_months > 1
]


print("[원천 파일 현황]")
print(f"공공자전거 이용정보 파일 수: {len(bike_usage_files):,}개")
print(f"대여소 마스터 파일 수: {len(station_files):,}개")
print(
    "공공자전거 이용정보 전체 용량: "
    f"{bike_file_inventory['file_size_mb'].sum():,.2f} MB"
)

print("\n[분석 기간 정합성]")
print(
    f"설정 기간: {expected_months.min()} ~ "
    f"{expected_months.max()}"
)
print(
    f"파일명 기준 기간: {observed_months.min()} ~ "
    f"{observed_months.max()}"
)
print(f"예상 월 수: {len(expected_months):,}개월")
print(f"확인된 고유 월 수: {len(observed_months):,}개월")
print(f"누락 월: {list(missing_months.astype(str))}")
print(f"중복 월: {list(duplicate_months.index.astype(str))}")
print(f"분석 범위 밖의 월: {list(outside_months.astype(str))}")

,data_type,file_name,file_extension,file_size_mb,year_month
0,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202311.csv,.csv,311.73,2023-11
1,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202401.csv,.csv,187.09,2024-01
2,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202402.csv,.csv,191.25,2024-02
3,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202403.csv,.csv,290.63,2024-03
4,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202404.csv,.csv,416.20,2024-04
5,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202405.csv,.csv,430.54,2024-05
6,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202406.csv,.csv,436.86,2024-06
7,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202407.csv,.csv,339.93,2024-07
8,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202408.csv,.csv,364.77,2024-08
9,bike_usage,서울특별시 공공자전거 이용정보(시간대별)_202409.csv,.csv,383.75,2024-09


,data_type,file_name,file_extension,file_size_mb,year_month
0,station,서울시 공공자전거 따릉이 대여소 마스터 정보.csv,.csv,0.27,None


[원천 파일 현황]
공공자전거 이용정보 파일 수: 24개
대여소 마스터 파일 수: 1개
공공자전거 이용정보 전체 용량: 8,027.94 MB

[분석 기간 정합성]
설정 기간: 2023-01 ~ 2024-12
파일명 기준 기간: 2023-01 ~ 2024-12
예상 월 수: 24개월
확인된 고유 월 수: 24개월
누락 월: []
중복 월: []
분석 범위 밖의 월: []


## 4. 원천 데이터 스키마 및 인코딩 점검

공공자전거 이용정보는 총 24개의 월별 CSV 파일로 구성되어 있다. 파일 전체를 결합하기 전에 각 파일의 문자 인코딩, 컬럼 수 및 컬럼 구성이 동일한지 확인한다.

월별 파일 간 스키마가 다를 경우 동일한 전처리 함수를 일괄 적용할 수 없으므로, 기준 파일과 비교하여 누락 컬럼과 추가 컬럼을 점검한다. 대여소 마스터 파일도 일부 행을 불러와 컬럼 구성과 인코딩을 함께 확인한다.


In [5]:
def inspect_file_schema(
    path: Path,
    reference_columns: List[str] | None = None,
    sample_rows: int = 5,
) -> Dict:
    """원천 파일의 인코딩과 컬럼 구조를 점검한다."""

    sample_raw = read_table(path, nrows=sample_rows)
    source_encoding = sample_raw.attrs.get(
        "source_encoding",
        "unknown",
    )

    sample = normalize_columns(sample_raw)
    columns = list(sample.columns)

    if reference_columns is None:
        missing_columns = []
        extra_columns = []
        schema_matches_reference = True
    else:
        missing_columns = sorted(
            set(reference_columns) - set(columns)
        )
        extra_columns = sorted(
            set(columns) - set(reference_columns)
        )
        schema_matches_reference = (
            columns == reference_columns
        )

    return {
        "file_name": path.name,
        "year_month": extract_year_month(path),
        "encoding": source_encoding,
        "sample_row_count": len(sample),
        "column_count": len(columns),
        "duplicate_column_count": int(
            pd.Index(columns).duplicated().sum()
        ),
        "schema_matches_reference": schema_matches_reference,
        "missing_columns": missing_columns,
        "extra_columns": extra_columns,
        "schema_signature": tuple(columns),
    }


# 첫 번째 파일을 기준 스키마로 설정한다.
reference_sample_raw = read_table(
    bike_usage_files[0],
    nrows=5,
)

reference_sample = normalize_columns(
    reference_sample_raw
)

reference_columns = list(reference_sample.columns)


# 24개 월별 파일의 스키마를 점검한다.
bike_schema_records = []

for path in bike_usage_files:
    record = inspect_file_schema(
        path=path,
        reference_columns=reference_columns,
        sample_rows=5,
    )
    bike_schema_records.append(record)


bike_schema_summary = (
    pd.DataFrame(bike_schema_records)
    .sort_values("year_month")
    .reset_index(drop=True)
)


display(
    bike_schema_summary[
        [
            "year_month",
            "file_name",
            "encoding",
            "column_count",
            "duplicate_column_count",
            "schema_matches_reference",
        ]
    ]
)


schema_mismatch = bike_schema_summary.loc[
    ~bike_schema_summary["schema_matches_reference"],
    [
        "year_month",
        "file_name",
        "missing_columns",
        "extra_columns",
    ],
]


print("[공공자전거 이용정보 스키마 점검]")
print(f"점검 파일 수: {len(bike_schema_summary):,}개")
print(
    "확인된 스키마 버전 수: "
    f"{bike_schema_summary['schema_signature'].nunique():,}개"
)
print(
    "기준 스키마와 불일치한 파일 수: "
    f"{len(schema_mismatch):,}개"
)
print(
    "중복 컬럼이 존재하는 파일 수: "
    f"{(bike_schema_summary['duplicate_column_count'] > 0).sum():,}개"
)

print("\n[기준 파일]")
print(bike_usage_files[0].name)

print("\n[기준 컬럼]")
for index, column in enumerate(reference_columns, start=1):
    print(f"{index:>2}. {column}")


if not schema_mismatch.empty:
    print("\n[스키마 불일치 파일]")
    display(schema_mismatch)


# 대여소 마스터 파일의 기본 구조를 확인한다.
station_preview_raw = read_table(
    station_files[0],
    nrows=5,
)

station_encoding = station_preview_raw.attrs.get(
    "source_encoding",
    "unknown",
)

station_preview = normalize_columns(
    station_preview_raw
)


print("\n[대여소 마스터 파일]")
print(f"파일명: {station_files[0].name}")
print(f"인코딩: {station_encoding}")
print(f"컬럼 수: {station_preview.shape[1]:,}개")

print("\n[대여소 마스터 컬럼]")
for index, column in enumerate(
    station_preview.columns,
    start=1,
):
    print(f"{index:>2}. {column}")


display(station_preview)

,year_month,file_name,encoding,column_count,duplicate_column_count,schema_matches_reference
0,2023-01,서울특별시 공공자전거 이용정보(시간대별)_2301.csv,cp949,12,0,True
1,2023-02,서울특별시 공공자전거 이용정보(시간대별)_2302.csv,cp949,12,0,True
2,2023-03,서울특별시 공공자전거 이용정보(시간대별)_2303.csv,cp949,12,0,True
3,2023-04,서울특별시 공공자전거 이용정보(시간대별)_2304.csv,cp949,12,0,True
4,2023-05,서울특별시 공공자전거 이용정보(시간대별)_2305.csv,cp949,12,0,True
5,2023-06,서울특별시 공공자전거 이용정보(시간대별)_2306.csv,cp949,12,0,True
6,2023-07,서울특별시 공공자전거 이용정보(시간대별)_2307.csv,cp949,12,0,True
7,2023-08,서울특별시 공공자전거 이용정보(시간대별)_2308.csv,cp949,12,0,True
8,2023-09,서울특별시 공공자전거 이용정보(시간대별)_2309.csv,cp949,12,0,True
9,2023-10,서울특별시 공공자전거 이용정보(시간대별)_2310.csv,cp949,12,0,True


[공공자전거 이용정보 스키마 점검]
점검 파일 수: 24개
확인된 스키마 버전 수: 1개
기준 스키마와 불일치한 파일 수: 0개
중복 컬럼이 존재하는 파일 수: 0개

[기준 파일]
서울특별시 공공자전거 이용정보(시간대별)_202311.csv

[기준 컬럼]
 1. 대여일자
 2. 대여시간
 3. 대여소번호
 4. 대여소명
 5. 대여구분코드
 6. 성별
 7. 연령대코드
 8. 이용건수
 9. 운동량
10. 탄소량
11. 이동거리(M)
12. 이용시간(분)

[대여소 마스터 파일]
파일명: 서울시 공공자전거 따릉이 대여소 마스터 정보.csv
인코딩: cp949
컬럼 수: 5개

[대여소 마스터 컬럼]
 1. 대여소_ID
 2. 주소1
 3. 주소2
 4. 위도
 5. 경도


,대여소_ID,주소1,주소2,위도,경도
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.000000,0.000000
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000
2,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
4,ST-995,서울특별시 양천구 중앙로 153 공중화장실,NaN,37.510597,126.857323


## 5. 분석 대상 컬럼 정의 및 표본 품질 점검

공공자전거 수요예측에 필요한 핵심 변수는 대여일자, 대여시간, 대여소번호, 대여소명 및 이용건수이다. 원천 데이터의 한 행은 개별 이용자 집단의 시간대별 이용정보를 나타내므로, 동일한 날짜·시간·대여소 조합에 여러 행이 존재할 수 있다.

따라서 대여일자와 대여시간을 결합하여 시간 변수를 생성하고, 대여소별 이용건수를 합산하여 시간당 대여 수요를 산출한다.

전체 약 8GB의 데이터를 결합하기 전에 각 월별 파일의 일부 행을 불러와 다음 항목을 점검한다.

* 대여일자의 날짜형 변환 가능 여부
* 대여시간의 숫자형 변환 여부와 0~23 범위 충족 여부
* 대여소번호의 결측 및 표준화 가능 여부
* 이용건수의 숫자형 변환 여부와 음수 존재 여부
* 대여소명의 결측 여부


In [6]:
BIKE_COLUMN_MAP = {
    "대여일자": "rental_date",
    "대여시간": "rental_hour",
    "대여소번호": "station_id",
    "대여소명": "station_name",
    "이용건수": "rental_count",
}

REQUIRED_BIKE_COLUMNS = list(BIKE_COLUMN_MAP.keys())


def inspect_bike_sample(
    path: Path,
    sample_rows: int = 1_000,
) -> Dict:
    """월별 이용정보 표본의 핵심 변수 품질을 점검한다."""

    sample = read_table(
        path,
        nrows=sample_rows,
    )

    sample = normalize_columns(sample)

    missing_required_columns = sorted(
        set(REQUIRED_BIKE_COLUMNS) - set(sample.columns)
    )

    if missing_required_columns:
        raise KeyError(
            f"{path.name}에 필수 컬럼이 존재하지 않는다: "
            f"{missing_required_columns}"
        )

    sample = (
        sample[REQUIRED_BIKE_COLUMNS]
        .rename(columns=BIKE_COLUMN_MAP)
        .copy()
    )

    rental_date = pd.to_datetime(
        sample["rental_date"],
        errors="coerce",
    )

    rental_hour = pd.to_numeric(
        sample["rental_hour"],
        errors="coerce",
    )

    station_id = clean_station_id(
        sample["station_id"]
    )

    rental_count = pd.to_numeric(
        sample["rental_count"],
        errors="coerce",
    )

    station_name = (
        sample["station_name"]
        .astype("string")
        .str.strip()
    )

    return {
        "year_month": extract_year_month(path),
        "file_name": path.name,
        "sample_rows": len(sample),
        "date_parse_failure": int(rental_date.isna().sum()),
        "hour_parse_failure": int(rental_hour.isna().sum()),
        "hour_out_of_range": int(
            (~rental_hour.between(0, 23))
            .fillna(False)
            .sum()
        ),
        "station_id_missing": int(station_id.isna().sum()),
        "station_name_missing": int(
            station_name.isna().sum()
            + station_name.eq("").sum()
        ),
        "rental_count_parse_failure": int(
            rental_count.isna().sum()
        ),
        "negative_rental_count": int(
            rental_count.lt(0).fillna(False).sum()
        ),
        "zero_rental_count": int(
            rental_count.eq(0).fillna(False).sum()
        ),
        "rental_count_min": rental_count.min(),
        "rental_count_max": rental_count.max(),
    }


sample_quality_records = []

for path in bike_usage_files:
    sample_quality_records.append(
        inspect_bike_sample(
            path=path,
            sample_rows=1_000,
        )
    )


bike_sample_quality = (
    pd.DataFrame(sample_quality_records)
    .sort_values("year_month")
    .reset_index(drop=True)
)


display(bike_sample_quality)


quality_columns = [
    "date_parse_failure",
    "hour_parse_failure",
    "hour_out_of_range",
    "station_id_missing",
    "station_name_missing",
    "rental_count_parse_failure",
    "negative_rental_count",
    "zero_rental_count",
]

quality_totals = (
    bike_sample_quality[quality_columns]
    .sum()
    .astype(int)
)


print("[표본 품질 점검 결과]")
print(
    f"점검 파일 수: "
    f"{len(bike_sample_quality):,}개"
)
print(
    f"점검 행 수: "
    f"{bike_sample_quality['sample_rows'].sum():,}행"
)

for column, value in quality_totals.items():
    print(f"{column}: {value:,}건")


print("\n[이용건수 표본 범위]")
print(
    "최솟값: "
    f"{bike_sample_quality['rental_count_min'].min()}"
)
print(
    "최댓값: "
    f"{bike_sample_quality['rental_count_max'].max()}"
)


,year_month,file_name,sample_rows,date_parse_failure,hour_parse_failure,hour_out_of_range,station_id_missing,station_name_missing,rental_count_parse_failure,negative_rental_count,zero_rental_count,rental_count_min,rental_count_max
0,2023-01,서울특별시 공공자전거 이용정보(시간대별)_2301.csv,1000,0,0,0,0,0,0,0,0,1,5
1,2023-02,서울특별시 공공자전거 이용정보(시간대별)_2302.csv,1000,0,0,0,0,0,0,0,0,1,3
2,2023-03,서울특별시 공공자전거 이용정보(시간대별)_2303.csv,1000,0,0,0,0,0,0,0,0,1,3
3,2023-04,서울특별시 공공자전거 이용정보(시간대별)_2304.csv,1000,0,0,0,0,0,0,0,0,1,4
4,2023-05,서울특별시 공공자전거 이용정보(시간대별)_2305.csv,1000,0,0,0,0,0,0,0,0,1,4
5,2023-06,서울특별시 공공자전거 이용정보(시간대별)_2306.csv,1000,0,0,0,0,0,0,0,0,1,5
6,2023-07,서울특별시 공공자전거 이용정보(시간대별)_2307.csv,1000,0,0,0,0,0,0,0,0,1,3
7,2023-08,서울특별시 공공자전거 이용정보(시간대별)_2308.csv,1000,0,0,0,0,0,0,0,0,1,4
8,2023-09,서울특별시 공공자전거 이용정보(시간대별)_2309.csv,1000,0,0,0,0,0,0,0,0,1,4
9,2023-10,서울특별시 공공자전거 이용정보(시간대별)_2310.csv,1000,0,0,0,0,0,0,0,0,1,5


[표본 품질 점검 결과]
점검 파일 수: 24개
점검 행 수: 24,000행
date_parse_failure: 0건
hour_parse_failure: 0건
hour_out_of_range: 0건
station_id_missing: 0건
station_name_missing: 0건
rental_count_parse_failure: 0건
negative_rental_count: 0건
zero_rental_count: 0건

[이용건수 표본 범위]
최솟값: 1
최댓값: 5


## 6. 대여소 마스터 데이터 품질 점검

대여소 마스터 데이터는 공공자전거 이용정보의 대여소번호와 결합하고, 후속 공간 군집화에 사용할 위도와 경도를 제공한다.

대여소 마스터 파일 전체를 불러와 다음 항목을 점검한다.

* 대여소 ID의 결측 및 중복 여부
* 주소 정보의 결측 여부
* 위도와 경도의 숫자형 변환 가능 여부
* 서울시 범위를 벗어난 비정상 GPS 좌표
* 동일한 대여소 ID에 서로 다른 좌표가 연결된 경우

대여소명에 해당하는 `주소2`가 결측인 경우에는 `주소1`을 보조 표기로 사용하되, 원본 주소 컬럼은 별도로 보존한다.


In [7]:
STATION_COLUMN_MAP = {
    "대여소_ID": "station_id",
    "주소1": "address_primary",
    "주소2": "address_detail",
    "위도": "latitude",
    "경도": "longitude",
}

REQUIRED_STATION_COLUMNS = list(STATION_COLUMN_MAP.keys())


station_raw = read_table(station_files[0])
station_raw = normalize_columns(station_raw)

missing_station_columns = sorted(
    set(REQUIRED_STATION_COLUMNS) - set(station_raw.columns)
)

if missing_station_columns:
    raise KeyError(
        "대여소 마스터에 필수 컬럼이 존재하지 않는다: "
        f"{missing_station_columns}"
    )


station_data = (
    station_raw[REQUIRED_STATION_COLUMNS]
    .rename(columns=STATION_COLUMN_MAP)
    .copy()
)


# 대여소 ID를 공통 문자열 형식으로 변환한다.
station_data["station_id"] = clean_station_id(
    station_data["station_id"]
)


# 주소 문자열의 앞뒤 공백과 빈 문자열을 정리한다.
for column in ["address_primary", "address_detail"]:
    station_data[column] = (
        station_data[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


# 주소2를 우선 사용하고, 결측인 경우 주소1을 보조 표기로 사용한다.
station_data["station_name"] = (
    station_data["address_detail"]
    .fillna(station_data["address_primary"])
)


# GPS 좌표를 숫자형으로 변환한다.
station_data["latitude"] = pd.to_numeric(
    station_data["latitude"],
    errors="coerce",
)

station_data["longitude"] = pd.to_numeric(
    station_data["longitude"],
    errors="coerce",
)


# 서울시의 대략적인 위도·경도 범위를 적용한다.
station_data["valid_gps"] = (
    station_data["latitude"].between(37.3, 37.8)
    & station_data["longitude"].between(126.7, 127.3)
)


# 동일한 대여소 ID가 두 번 이상 등장하는 행을 식별한다.
duplicate_station_rows = station_data.loc[
    station_data["station_id"].notna()
    & station_data["station_id"].duplicated(keep=False)
].copy()


# 동일 ID에 서로 다른 좌표가 연결되었는지 확인한다.
coordinate_nunique = (
    station_data
    .dropna(subset=["station_id"])
    .groupby("station_id")[["latitude", "longitude"]]
    .nunique(dropna=True)
)

conflicting_coordinate_ids = coordinate_nunique.loc[
    (coordinate_nunique["latitude"] > 1)
    | (coordinate_nunique["longitude"] > 1)
].index


station_quality_summary = pd.DataFrame(
    {
        "quality_item": [
            "전체 행 수",
            "대여소 ID 결측",
            "고유 대여소 ID 수",
            "중복 대여소 ID 수",
            "주소1 결측",
            "주소2 결측",
            "최종 대여소 표기 결측",
            "위도 결측 또는 변환 실패",
            "경도 결측 또는 변환 실패",
            "서울 범위 밖 GPS",
            "좌표가 충돌하는 대여소 ID",
            "완전 중복 행",
        ],
        "count": [
            len(station_data),
            station_data["station_id"].isna().sum(),
            station_data["station_id"].nunique(),
            duplicate_station_rows["station_id"].nunique(),
            station_data["address_primary"].isna().sum(),
            station_data["address_detail"].isna().sum(),
            station_data["station_name"].isna().sum(),
            station_data["latitude"].isna().sum(),
            station_data["longitude"].isna().sum(),
            (~station_data["valid_gps"]).sum(),
            len(conflicting_coordinate_ids),
            station_data.duplicated().sum(),
        ],
    }
)


display(station_quality_summary)


print("[대여소 마스터 품질 점검]")
for row in station_quality_summary.itertuples(index=False):
    print(f"{row.quality_item}: {row.count:,}건")


print("\n[GPS 좌표 범위]")
print(
    f"위도: {station_data['latitude'].min()} "
    f"~ {station_data['latitude'].max()}"
)
print(
    f"경도: {station_data['longitude'].min()} "
    f"~ {station_data['longitude'].max()}"
)


if not duplicate_station_rows.empty:
    print("\n[중복 대여소 ID 표본]")
    display(
        duplicate_station_rows
        .sort_values("station_id")
        .head(20)
    )


invalid_gps_rows = station_data.loc[
    ~station_data["valid_gps"]
]

if not invalid_gps_rows.empty:
    print("\n[비정상 GPS 표본]")
    display(invalid_gps_rows.head(20))


if len(conflicting_coordinate_ids) > 0:
    print("\n[좌표가 충돌하는 대여소 ID 표본]")
    display(
        station_data.loc[
            station_data["station_id"].isin(
                conflicting_coordinate_ids
            )
        ]
        .sort_values("station_id")
        .head(20)
    )

,quality_item,count
0,전체 행 수,3418
1,대여소 ID 결측,0
2,고유 대여소 ID 수,3418
3,중복 대여소 ID 수,0
4,주소1 결측,0
5,주소2 결측,1995
6,최종 대여소 표기 결측,0
7,위도 결측 또는 변환 실패,0
8,경도 결측 또는 변환 실패,0
9,서울 범위 밖 GPS,77


[대여소 마스터 품질 점검]
전체 행 수: 3,418건
대여소 ID 결측: 0건
고유 대여소 ID 수: 3,418건
중복 대여소 ID 수: 0건
주소1 결측: 0건
주소2 결측: 1,995건
최종 대여소 표기 결측: 0건
위도 결측 또는 변환 실패: 0건
경도 결측 또는 변환 실패: 0건
서울 범위 밖 GPS: 77건
좌표가 충돌하는 대여소 ID: 0건
완전 중복 행: 0건

[GPS 좌표 범위]
위도: 0.0 ~ 37.692322
경도: 0.0 ~ 127.180756

[비정상 GPS 표본]


,station_id,address_primary,address_detail,latitude,longitude,station_name,valid_gps
0,999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.0,0.0,목동아파트 8단지 상가동,False
1,998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.0,0.0,목동아파트 4단지 상가동,False
11,989,서울특별시 마포구 월드컵로5길 11,합정동 주민센터,0.0,0.0,합정동 주민센터,False
156,858,서울특별시 강북구 덕릉로40길 48,번동 619-53,0.0,0.0,번동 619-53,False
171,844,서울특별시 성북구 인촌로 16,성북구 안암동2가 140-1,0.0,0.0,성북구 안암동2가 140-1,False
439,601,서울특별시 노원구 월계로 252,노원구 월계동 860-16,0.0,0.0,노원구 월계동 860-16,False
564,489,서울특별시 강동구 고덕로 362,상일동 146,0.0,0.0,상일동 146,False
587,468,서울특별시 은평구 녹번동 116-3,서울혁신파크 건너편,0.0,0.0,서울혁신파크 건너편,False
598,458,서울특별시 은평구 신사동 35-23,새절역 사거리,0.0,0.0,새절역 사거리,False
635,423,서울특별시 영등포구 여의나루로 96,MBC,0.0,0.0,MBC,False


## 7. 비정상 GPS 좌표의 원인 확인

대여소 마스터 3,418개 중 77개 대여소의 위도 또는 경도가 서울시의 정상 범위를 벗어난 것으로 확인되었다.

공간 군집화에서는 위도와 경도가 필수이므로 비정상 좌표를 그대로 사용하면 K-Means 군집 중심이 왜곡될 수 있다. 따라서 비정상 좌표가 `(0, 0)`과 같은 미입력 값인지, 일부 좌표만 0인지, 또는 서울시 범위를 벗어난 실제 좌표인지 유형별로 구분한다.

비정상 GPS 대여소는 원본 데이터에서 삭제하지 않고 별도로 기록한다. 이후 공간 군집화 단계에서는 정상 좌표를 보유한 대여소만 사용한다.


In [8]:
invalid_gps_detail = (
    station_data.loc[
        ~station_data["valid_gps"],
        [
            "station_id",
            "station_name",
            "address_primary",
            "address_detail",
            "latitude",
            "longitude",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)


conditions = [
    (
        invalid_gps_detail["latitude"].eq(0)
        & invalid_gps_detail["longitude"].eq(0)
    ),
    (
        invalid_gps_detail["latitude"].eq(0)
        & invalid_gps_detail["longitude"].ne(0)
    ),
    (
        invalid_gps_detail["latitude"].ne(0)
        & invalid_gps_detail["longitude"].eq(0)
    ),
    (
        invalid_gps_detail["latitude"].isna()
        | invalid_gps_detail["longitude"].isna()
    ),
]

choices = [
    "위도·경도 모두 0",
    "위도만 0",
    "경도만 0",
    "좌표 결측",
]

invalid_gps_detail["gps_issue"] = np.select(
    conditions,
    choices,
    default="서울시 범위 밖 좌표",
)


invalid_gps_type_summary = (
    invalid_gps_detail
    .groupby("gps_issue", dropna=False)
    .size()
    .rename("station_count")
    .reset_index()
    .sort_values(
        "station_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


invalid_coordinate_summary = (
    invalid_gps_detail
    .groupby(
        ["latitude", "longitude", "gps_issue"],
        dropna=False,
    )
    .size()
    .rename("station_count")
    .reset_index()
    .sort_values(
        "station_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(invalid_gps_type_summary)
display(invalid_coordinate_summary)
display(invalid_gps_detail.head(20))


print("[비정상 GPS 유형]")
for row in invalid_gps_type_summary.itertuples(index=False):
    print(f"{row.gps_issue}: {row.station_count:,}개")


print("\n[대여소 수 비교]")
print(f"전체 대여소: {len(station_data):,}개")
print(f"비정상 GPS 대여소: {len(invalid_gps_detail):,}개")
print(
    "정상 GPS 대여소: "
    f"{station_data['valid_gps'].sum():,}개"
)


,gps_issue,station_count
0,위도·경도 모두 0,77


,latitude,longitude,gps_issue,station_count
0,0.0,0.0,위도·경도 모두 0,77


,station_id,station_name,address_primary,address_detail,latitude,longitude,gps_issue
0,999,목동아파트 8단지 상가동,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.0,0.0,위도·경도 모두 0
1,998,목동아파트 4단지 상가동,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.0,0.0,위도·경도 모두 0
2,989,합정동 주민센터,서울특별시 마포구 월드컵로5길 11,합정동 주민센터,0.0,0.0,위도·경도 모두 0
3,858,번동 619-53,서울특별시 강북구 덕릉로40길 48,번동 619-53,0.0,0.0,위도·경도 모두 0
4,844,성북구 안암동2가 140-1,서울특별시 성북구 인촌로 16,성북구 안암동2가 140-1,0.0,0.0,위도·경도 모두 0
5,601,노원구 월계동 860-16,서울특별시 노원구 월계로 252,노원구 월계동 860-16,0.0,0.0,위도·경도 모두 0
6,489,상일동 146,서울특별시 강동구 고덕로 362,상일동 146,0.0,0.0,위도·경도 모두 0
7,468,서울혁신파크 건너편,서울특별시 은평구 녹번동 116-3,서울혁신파크 건너편,0.0,0.0,위도·경도 모두 0
8,458,새절역 사거리,서울특별시 은평구 신사동 35-23,새절역 사거리,0.0,0.0,위도·경도 모두 0
9,423,MBC,서울특별시 영등포구 여의나루로 96,MBC,0.0,0.0,위도·경도 모두 0


[비정상 GPS 유형]
위도·경도 모두 0: 77개

[대여소 수 비교]
전체 대여소: 3,418개
비정상 GPS 대여소: 77개
정상 GPS 대여소: 3,341개


## 8. 대여소 마스터 정제 및 품질 이슈 분리

비정상 GPS 대여소 77개는 위도와 경도가 모두 0으로 기록된 좌표 미입력 사례이다. `(0, 0)`은 실제 대여소 위치를 의미하지 않으며, 이를 그대로 공간 군집화에 사용하면 군집 중심이 왜곡된다.

좌표를 평균값이나 임의의 위치로 대체할 경우 실제 공간적 특성과 다른 정보가 생성되므로 별도의 좌표 보정은 수행하지 않는다. 원본 데이터는 유지하되, 정상 GPS 대여소와 비정상 GPS 대여소를 분리하여 각각 저장한다.

* 정상 GPS 대여소 3,341개는 후속 공간 군집화에 사용한다.
* 비정상 GPS 대여소 77개는 데이터 품질 이슈 목록으로 별도 보존한다.
* 후속 이용정보 정합성 점검에서 비정상 GPS 대여소가 차지하는 이용건수 비중을 확인한다.


In [9]:
processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)


station_master_clean = (
    station_data.loc[
        station_data["valid_gps"],
        [
            "station_id",
            "station_name",
            "address_primary",
            "address_detail",
            "latitude",
            "longitude",
        ],
    ]
    .copy()
    .sort_values("station_id")
    .reset_index(drop=True)
)


station_master_invalid_gps = (
    invalid_gps_detail
    .copy()
    .sort_values("station_id")
    .reset_index(drop=True)
)


clean_station_path = (
    processed_dir / "station_master_clean.csv"
)

invalid_station_path = (
    processed_dir / "station_master_invalid_gps.csv"
)


station_master_clean.to_csv(
    clean_station_path,
    index=False,
    encoding="utf-8-sig",
)

station_master_invalid_gps.to_csv(
    invalid_station_path,
    index=False,
    encoding="utf-8-sig",
)


print("[대여소 마스터 정제 결과]")
print(
    f"원본 대여소: "
    f"{len(station_data):,}개"
)
print(
    f"정상 GPS 대여소: "
    f"{len(station_master_clean):,}개"
)
print(
    f"비정상 GPS 대여소: "
    f"{len(station_master_invalid_gps):,}개"
)

print("\n[저장 파일]")
print(
    f"{clean_station_path.relative_to(ROOT)} "
    f"- {clean_station_path.exists()}"
)
print(
    f"{invalid_station_path.relative_to(ROOT)} "
    f"- {invalid_station_path.exists()}"
)


if (
    len(station_master_clean)
    + len(station_master_invalid_gps)
    != len(station_data)
):
    raise ValueError(
        "정상 GPS와 비정상 GPS 데이터의 행 수 합계가 "
        "원본 대여소 수와 일치하지 않는다."
    )

if station_master_clean["station_id"].duplicated().any():
    raise ValueError(
        "정제된 대여소 마스터에 중복 대여소 ID가 존재한다."
    )

if not station_master_clean[
    ["latitude", "longitude"]
].notna().all().all():
    raise ValueError(
        "정제된 대여소 마스터에 GPS 결측값이 존재한다."
    )

[대여소 마스터 정제 결과]
원본 대여소: 3,418개
정상 GPS 대여소: 3,341개
비정상 GPS 대여소: 77개

[저장 파일]
data\processed\station_master_clean.csv - True
data\processed\station_master_invalid_gps.csv - True


## 9. 청크 단위 공공자전거 이용정보 집계 함수 정의

공공자전거 이용정보 24개 파일의 전체 용량은 약 8.03GB이므로, 모든 원천 데이터를 한 번에 메모리에 적재하면 불필요한 메모리 사용과 실행 안정성 저하가 발생할 수 있다.

이에 각 월별 CSV 파일을 일정한 행 수의 청크로 나누어 읽고, 각 청크를 `시간 × 대여소` 단위로 먼저 집계한다. 이후 동일 월의 청크별 집계 결과를 다시 합산하여 월별 대여소 시간당 수요 데이터를 생성한다.

집계 과정에서는 다음 품질 항목을 함께 기록한다.

* 날짜 및 시간 변환 실패
* 0~23 범위를 벗어난 대여시간
* 정수가 아닌 대여시간
* 대여소 ID 결측
* 이용건수 숫자형 변환 실패
* 음수 및 정수가 아닌 이용건수
* 설정된 분석 기간 밖의 행
* 파일명 연월과 실제 대여일자의 불일치
* 전처리 전후 이용건수 합계 보존 여부

원천 데이터의 동일한 날짜·시간·대여소 조합에 여러 행이 존재하는 것은 성별, 연령대 및 대여구분별 집계 구조에 따른 정상적인 현상이다. 따라서 해당 행을 중복으로 제거하지 않고 `이용건수`를 합산한다.


In [10]:
CHUNK_SIZE = 250_000


def detect_csv_encoding(path: Path) -> str:
    """CSV 파일을 읽을 수 있는 문자 인코딩을 확인한다."""

    sample = read_table(path, nrows=1)
    encoding = sample.attrs.get("source_encoding")

    if encoding in {None, "unknown", "excel"}:
        raise ValueError(
            f"CSV 문자 인코딩을 확인하지 못했다: {path.name}"
        )

    return encoding


def process_bike_chunk(
    chunk: pd.DataFrame,
    source_file: str,
    file_year_month: Optional[pd.Period],
    period_start: pd.Timestamp,
    period_end_exclusive: pd.Timestamp,
) -> tuple[pd.DataFrame, Dict]:
    """공공자전거 이용정보 청크를 정제하고 시간·대여소별로 집계한다."""

    chunk = normalize_columns(chunk)

    missing_columns = sorted(
        set(REQUIRED_BIKE_COLUMNS) - set(chunk.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{source_file}에 필수 컬럼이 존재하지 않는다: "
            f"{missing_columns}"
        )

    data = (
        chunk[REQUIRED_BIKE_COLUMNS]
        .rename(columns=BIKE_COLUMN_MAP)
        .copy()
    )

    raw_row_count = len(data)

    rental_date = pd.to_datetime(
        data["rental_date"],
        errors="coerce",
    ).dt.normalize()

    rental_hour = pd.to_numeric(
        data["rental_hour"],
        errors="coerce",
    )

    station_id = clean_station_id(
        data["station_id"]
    )

    station_name = (
        data["station_name"]
        .astype("string")
        .str.strip()
    )

    rental_count = pd.to_numeric(
        data["rental_count"],
        errors="coerce",
    )


    # 시간 변수의 품질을 점검한다.
    date_parse_failure = rental_date.isna()

    hour_parse_failure = rental_hour.isna()

    hour_out_of_range = (
        rental_hour.notna()
        & ~rental_hour.between(0, 23)
    )

    hour_non_integer = (
        rental_hour.notna()
        & rental_hour.mod(1).ne(0)
    )


    # 대여소 및 이용건수 변수의 품질을 점검한다.
    station_id_missing = station_id.isna()

    station_name_missing = (
        station_name.isna()
        | station_name.eq("")
    )

    rental_count_parse_failure = rental_count.isna()

    negative_rental_count = (
        rental_count.notna()
        & rental_count.lt(0)
    )

    rental_count_non_integer = (
        rental_count.notna()
        & rental_count.mod(1).ne(0)
    )

    zero_rental_count = (
        rental_count.notna()
        & rental_count.eq(0)
    )


    # 날짜와 시간이 모두 정상인 행에 대해 시간 변수를 생성한다.
    valid_datetime_component = (
        ~date_parse_failure
        & ~hour_parse_failure
        & ~hour_out_of_range
        & ~hour_non_integer
    )

    datetime = pd.Series(
        pd.NaT,
        index=data.index,
        dtype="datetime64[ns]",
    )

    datetime.loc[valid_datetime_component] = (
        rental_date.loc[valid_datetime_component]
        + pd.to_timedelta(
            rental_hour.loc[
                valid_datetime_component
            ].astype(int),
            unit="h",
        )
    )


    # 설정된 전체 분석 기간 밖의 행을 식별한다.
    outside_period = (
        datetime.notna()
        & (
            datetime.lt(period_start)
            | datetime.ge(period_end_exclusive)
        )
    )


    # 파일명 연월과 실제 대여일자의 연월이 다른 행을 식별한다.
    if file_year_month is None:
        file_month_mismatch = pd.Series(
            False,
            index=data.index,
        )
    else:
        file_month_mismatch = (
            rental_date.notna()
            & rental_date.dt.to_period("M").ne(
                file_year_month
            )
        )


    # 집계에 사용할 유효 행을 정의한다.
    valid_row = (
        datetime.notna()
        & ~outside_period
        & station_id.notna()
        & rental_count.notna()
        & ~negative_rental_count
        & ~rental_count_non_integer
    )


    valid_data = pd.DataFrame(
        {
            "datetime": datetime.loc[valid_row],
            "station_id": station_id.loc[valid_row],
            "rental_count": rental_count.loc[valid_row],
        }
    )


    chunk_hourly = (
        valid_data
        .groupby(
            ["datetime", "station_id"],
            as_index=False,
        )["rental_count"]
        .sum()
    )


    input_rental_count_sum = float(
        valid_data["rental_count"].sum()
    )

    output_rental_count_sum = float(
        chunk_hourly["rental_count"].sum()
    )


    if not np.isclose(
        input_rental_count_sum,
        output_rental_count_sum,
    ):
        raise ValueError(
            f"{source_file} 청크 집계 전후의 이용건수 합계가 "
            "일치하지 않는다."
        )


    quality_record = {
        "source_file": source_file,
        "raw_row_count": raw_row_count,
        "valid_row_count": int(valid_row.sum()),
        "excluded_row_count": int((~valid_row).sum()),
        "date_parse_failure": int(
            date_parse_failure.sum()
        ),
        "hour_parse_failure": int(
            hour_parse_failure.sum()
        ),
        "hour_out_of_range": int(
            hour_out_of_range.sum()
        ),
        "hour_non_integer": int(
            hour_non_integer.sum()
        ),
        "station_id_missing": int(
            station_id_missing.sum()
        ),
        "station_name_missing": int(
            station_name_missing.sum()
        ),
        "rental_count_parse_failure": int(
            rental_count_parse_failure.sum()
        ),
        "negative_rental_count": int(
            negative_rental_count.sum()
        ),
        "rental_count_non_integer": int(
            rental_count_non_integer.sum()
        ),
        "zero_rental_count": int(
            zero_rental_count.sum()
        ),
        "outside_period": int(
            outside_period.sum()
        ),
        "file_month_mismatch": int(
            file_month_mismatch.sum()
        ),
        "input_rental_count_sum": input_rental_count_sum,
        "output_rental_count_sum": output_rental_count_sum,
        "aggregated_row_count": len(chunk_hourly),
    }

    return chunk_hourly, quality_record


def process_bike_file(
    path: Path,
    period_start: pd.Timestamp,
    period_end_exclusive: pd.Timestamp,
    chunk_size: int = CHUNK_SIZE,
) -> tuple[pd.DataFrame, Dict]:
    """월별 CSV 파일을 청크 단위로 읽어 시간·대여소별로 집계한다."""

    if path.suffix.lower() != ".csv":
        raise ValueError(
            f"청크 처리는 CSV 파일만 지원한다: {path.name}"
        )

    encoding = detect_csv_encoding(path)
    file_year_month = extract_year_month(path)

    chunk_results = []
    chunk_quality_records = []


    # 파일 중간의 해석 불가능한 문자는 대체문자로 변환한다.
    # 이후 핵심 변수에 대체문자가 포함되었는지 별도로 검증한다.
    reader = pd.read_csv(
        path,
        encoding=encoding,
        encoding_errors="replace",
        usecols=REQUIRED_BIKE_COLUMNS,
        dtype={
            column: "string"
            for column in REQUIRED_BIKE_COLUMNS
        },
        chunksize=chunk_size,
        low_memory=False,
    )


    key_source_columns = [
        "대여일자",
        "대여시간",
        "대여소번호",
        "이용건수",
    ]


    for chunk_number, chunk in enumerate(
        reader,
        start=1,
    ):
        string_chunk = chunk.astype("string")


        # 전체 핵심 컬럼에서 유니코드 대체문자 발생 건수를 계산한다.
        decode_replacement_count = int(
            string_chunk
            .apply(
                lambda series: (
                    series
                    .str.count("\ufffd")
                    .fillna(0)
                    .sum()
                )
            )
            .sum()
        )


        # 수요 집계에 직접 사용되는 핵심 변수의 대체문자를 계산한다.
        decode_replacement_in_key_columns = int(
            string_chunk[key_source_columns]
            .apply(
                lambda series: (
                    series
                    .str.count("\ufffd")
                    .fillna(0)
                    .sum()
                )
            )
            .sum()
        )


        chunk_hourly, quality_record = (
            process_bike_chunk(
                chunk=chunk,
                source_file=path.name,
                file_year_month=file_year_month,
                period_start=period_start,
                period_end_exclusive=period_end_exclusive,
            )
        )


        quality_record[
            "decode_replacement_count"
        ] = decode_replacement_count

        quality_record[
            "decode_replacement_in_key_columns"
        ] = decode_replacement_in_key_columns


        chunk_results.append(chunk_hourly)
        chunk_quality_records.append(quality_record)


        replacement_message = ""

        if decode_replacement_count > 0:
            replacement_message = (
                f", 디코딩 대체문자 "
                f"{decode_replacement_count:,}건"
            )


        print(
            f"  청크 {chunk_number:>3}: "
            f"{quality_record['raw_row_count']:,}행 → "
            f"{quality_record['aggregated_row_count']:,}행"
            f"{replacement_message}"
        )


    if not chunk_results:
        raise ValueError(
            f"처리할 데이터가 존재하지 않는다: {path.name}"
        )


    monthly_hourly = (
        pd.concat(
            chunk_results,
            ignore_index=True,
        )
        .groupby(
            ["datetime", "station_id"],
            as_index=False,
        )["rental_count"]
        .sum()
        .sort_values(
            ["datetime", "station_id"]
        )
        .reset_index(drop=True)
    )


    chunk_quality = pd.DataFrame(
        chunk_quality_records
    )


    file_quality_record = {
        "year_month": file_year_month,
        "source_file": path.name,
        "encoding": encoding,
        "chunk_count": len(chunk_quality),
        "raw_row_count": int(
            chunk_quality["raw_row_count"].sum()
        ),
        "valid_row_count": int(
            chunk_quality["valid_row_count"].sum()
        ),
        "excluded_row_count": int(
            chunk_quality["excluded_row_count"].sum()
        ),
        "date_parse_failure": int(
            chunk_quality["date_parse_failure"].sum()
        ),
        "hour_parse_failure": int(
            chunk_quality["hour_parse_failure"].sum()
        ),
        "hour_out_of_range": int(
            chunk_quality["hour_out_of_range"].sum()
        ),
        "hour_non_integer": int(
            chunk_quality["hour_non_integer"].sum()
        ),
        "station_id_missing": int(
            chunk_quality["station_id_missing"].sum()
        ),
        "station_name_missing": int(
            chunk_quality["station_name_missing"].sum()
        ),
        "rental_count_parse_failure": int(
            chunk_quality[
                "rental_count_parse_failure"
            ].sum()
        ),
        "negative_rental_count": int(
            chunk_quality[
                "negative_rental_count"
            ].sum()
        ),
        "rental_count_non_integer": int(
            chunk_quality[
                "rental_count_non_integer"
            ].sum()
        ),
        "zero_rental_count": int(
            chunk_quality["zero_rental_count"].sum()
        ),
        "outside_period": int(
            chunk_quality["outside_period"].sum()
        ),
        "file_month_mismatch": int(
            chunk_quality["file_month_mismatch"].sum()
        ),
        "decode_replacement_count": int(
            chunk_quality[
                "decode_replacement_count"
            ].sum()
        ),
        "decode_replacement_in_key_columns": int(
            chunk_quality[
                "decode_replacement_in_key_columns"
            ].sum()
        ),
        "input_rental_count_sum": float(
            chunk_quality[
                "input_rental_count_sum"
            ].sum()
        ),
        "output_rental_count_sum": float(
            monthly_hourly["rental_count"].sum()
        ),
        "aggregated_row_count": len(monthly_hourly),
        "unique_station_count": int(
            monthly_hourly["station_id"].nunique()
        ),
        "datetime_min": monthly_hourly["datetime"].min(),
        "datetime_max": monthly_hourly["datetime"].max(),
    }


    # 집계 핵심 변수에서 손상 문자가 발견되면 처리를 중단한다.
    if (
        file_quality_record[
            "decode_replacement_in_key_columns"
        ]
        > 0
    ):
        raise ValueError(
            f"{path.name}의 대여일자·대여시간·대여소번호·"
            "이용건수에서 디코딩 손상 문자가 발견되었다."
        )


    if not np.isclose(
        file_quality_record["input_rental_count_sum"],
        file_quality_record["output_rental_count_sum"],
    ):
        raise ValueError(
            f"{path.name}의 월별 집계 전후 이용건수 합계가 "
            "일치하지 않는다."
        )


    return monthly_hourly, file_quality_record

## 10. 단일 월 파일 시험 집계

전체 24개 파일을 처리하기 전에 용량이 가장 작은 월별 파일 1개를 대상으로 청크 집계를 시험한다.

시험 처리에서는 다음 사항을 확인한다.

- 원천 파일이 청크 단위로 정상적으로 읽히는지 확인한다.
- 날짜, 시간, 대여소 ID 및 이용건수가 정상적으로 변환되는지 확인한다.
- 분석에서 제외된 행과 품질 이상 항목을 확인한다.
- 집계 전후 이용건수 합계가 보존되는지 확인한다.
- 집계 결과의 시간 범위와 대여소 수가 합리적인지 확인한다.

시험 결과는 최종 전처리 파일로 저장하지 않는다.

In [11]:
period_start = pd.Timestamp(
    cfg["period"]["start"]
)

period_end_exclusive = (
    pd.Timestamp(cfg["period"]["end"])
    + pd.Timedelta(days=1)
)


# 파일 크기가 가장 작은 월별 파일을 시험 대상으로 선택한다.
test_file = min(
    bike_usage_files,
    key=lambda path: path.stat().st_size,
)


print("[시험 대상 파일]")
print(f"파일명: {test_file.name}")
print(
    "파일 크기: "
    f"{test_file.stat().st_size / (1024**2):,.2f} MB"
)


test_hourly, test_quality = process_bike_file(
    path=test_file,
    period_start=period_start,
    period_end_exclusive=period_end_exclusive,
    chunk_size=CHUNK_SIZE,
)


test_quality_df = pd.DataFrame(
    [test_quality]
)

display(test_quality_df)
display(test_hourly.head())
display(test_hourly.tail())


quality_error_columns = [
    "date_parse_failure",
    "hour_parse_failure",
    "hour_out_of_range",
    "hour_non_integer",
    "station_id_missing",
    "rental_count_parse_failure",
    "negative_rental_count",
    "rental_count_non_integer",
    "outside_period",
    "file_month_mismatch",
]


print("\n[시험 집계 결과]")
print(
    f"원천 행 수: "
    f"{test_quality['raw_row_count']:,}행"
)
print(
    f"유효 행 수: "
    f"{test_quality['valid_row_count']:,}행"
)
print(
    f"제외 행 수: "
    f"{test_quality['excluded_row_count']:,}행"
)
print(
    f"집계 후 행 수: "
    f"{test_quality['aggregated_row_count']:,}행"
)
print(
    f"고유 대여소 수: "
    f"{test_quality['unique_station_count']:,}개"
)
print(
    f"시간 범위: "
    f"{test_quality['datetime_min']} "
    f"~ {test_quality['datetime_max']}"
)


print("\n[품질 이상 항목]")
for column in quality_error_columns:
    print(
        f"{column}: "
        f"{test_quality[column]:,}건"
    )


print("\n[이용건수 합계 보존]")
print(
    "집계 전 합계: "
    f"{test_quality['input_rental_count_sum']:,.0f}"
)
print(
    "집계 후 합계: "
    f"{test_quality['output_rental_count_sum']:,.0f}"
)
print(
    "합계 일치 여부: "
    f"{np.isclose(
        test_quality['input_rental_count_sum'],
        test_quality['output_rental_count_sum'],
    )}"
)


if test_quality["excluded_row_count"] != 0:
    print(
        "\n주의: 시험 파일에서 제외된 행이 확인되었다. "
        "품질 항목별 원인을 검토해야 한다."
    )

if not np.isclose(
    test_quality["input_rental_count_sum"],
    test_quality["output_rental_count_sum"],
):
    raise ValueError(
        "시험 집계 전후 이용건수 합계가 일치하지 않는다."
    )

[시험 대상 파일]
파일명: 서울특별시 공공자전거 이용정보(시간대별)_2301.csv
파일 크기: 148.09 MB
  청크   1: 250,000행 → 126,479행
  청크   2: 250,000행 → 123,117행
  청크   3: 250,000행 → 115,757행
  청크   4: 250,000행 → 125,889행
  청크   5: 250,000행 → 150,923행
  청크   6: 223,431행 → 115,790행


,year_month,source_file,encoding,chunk_count,raw_row_count,valid_row_count,excluded_row_count,date_parse_failure,hour_parse_failure,hour_out_of_range,...,outside_period,file_month_mismatch,decode_replacement_count,decode_replacement_in_key_columns,input_rental_count_sum,output_rental_count_sum,aggregated_row_count,unique_station_count,datetime_min,datetime_max
0,2023-01,서울특별시 공공자전거 이용정보(시간대별)_2301.csv,cp949,6,1473431,1473431,0,0,0,0,...,0,0,0,0,1571434.0,1571434.0,755977,2709,2023-01-01,2023-01-31 23:00:00


,datetime,station_id,rental_count
0,2023-01-01,00102,3
1,2023-01-01,00103,1
2,2023-01-01,00104,1
3,2023-01-01,00105,1
4,2023-01-01,00106,3


,datetime,station_id,rental_count
755972,2023-01-31 23:00:00,05854,1
755973,2023-01-31 23:00:00,05858,5
755974,2023-01-31 23:00:00,05859,1
755975,2023-01-31 23:00:00,05862,2
755976,2023-01-31 23:00:00,05865,1



[시험 집계 결과]
원천 행 수: 1,473,431행
유효 행 수: 1,473,431행
제외 행 수: 0행
집계 후 행 수: 755,977행
고유 대여소 수: 2,709개
시간 범위: 2023-01-01 00:00:00 ~ 2023-01-31 23:00:00

[품질 이상 항목]
date_parse_failure: 0건
hour_parse_failure: 0건
hour_out_of_range: 0건
hour_non_integer: 0건
station_id_missing: 0건
rental_count_parse_failure: 0건
negative_rental_count: 0건
rental_count_non_integer: 0건
outside_period: 0건
file_month_mismatch: 0건

[이용건수 합계 보존]
집계 전 합계: 1,571,434
집계 후 합계: 1,571,434
합계 일치 여부: True


## 11. 전체 월별 이용정보 집계 및 중간 결과 저장

단일 월 시험 집계를 통해 청크 처리와 이용건수 합계 보존이 정상적으로 수행되는 것을 확인했다. 동일한 전처리 함수를 2023년 1월부터 2024년 12월까지의 24개 월별 파일 전체에 적용한다.

월별 집계 결과는 하나의 대용량 파일로 즉시 결합하지 않고 `data/processed/station_hourly_monthly/`에 월별 CSV 파일로 나누어 저장한다. 이 구조는 실행 중 메모리 사용량을 제한하고, 특정 월의 오류 발생 시 전체 전처리를 다시 수행하지 않고 해당 월을 개별적으로 점검할 수 있게 한다.

각 월의 처리 결과에서는 다음 항목을 검증한다.

- 원천 행 수와 유효 행 수
- 품질 문제로 제외된 행 수
- 날짜·시간·대여소 ID·이용건수 오류
- 월별 시간 범위
- 고유 대여소 수
- 집계 전후 이용건수 합계
- 월별 중간 파일의 생성 여부

이 단계에서 생성하는 데이터는 대여소별 시간당 수요이다. 공간 군집화와 클러스터별 수요 집계는 `02_spatial_clustering_analysis.ipynb`에서 수행한다.

In [12]:
import gc


monthly_output_dir = (
    processed_dir / "station_hourly_monthly"
)

monthly_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

monthly_quality_path = (
    processed_dir / "bike_usage_quality_by_month.csv"
)


processing_files = sorted(
    bike_usage_files,
    key=lambda path: extract_year_month(path),
)


monthly_quality_records = []
expected_output_paths = []


for file_index, path in enumerate(
    processing_files,
    start=1,
):
    year_month = extract_year_month(path)

    if year_month is None:
        raise ValueError(
            f"파일명에서 연월을 추출하지 못했다: {path.name}"
        )

    output_path = (
        monthly_output_dir
        / f"station_hourly_{year_month.strftime('%Y%m')}.csv"
    )

    expected_output_paths.append(output_path)


    print("\n" + "=" * 80)
    print(
        f"[{file_index:02d}/{len(processing_files):02d}] "
        f"{year_month} 처리"
    )
    print(f"원천 파일: {path.name}")
    print("=" * 80)


    monthly_hourly, file_quality = process_bike_file(
        path=path,
        period_start=period_start,
        period_end_exclusive=period_end_exclusive,
        chunk_size=CHUNK_SIZE,
    )


    # 월별 결과에 다른 연월의 데이터가 포함되었는지 재검증한다.
    result_months = (
        monthly_hourly["datetime"]
        .dt.to_period("M")
        .unique()
    )

    if (
        len(result_months) != 1
        or result_months[0] != year_month
    ):
        raise ValueError(
            f"{path.name}의 집계 결과에 "
            "다른 연월의 데이터가 포함되어 있다."
        )


    # 이용건수의 정수성을 확인한 뒤 정수형으로 저장한다.
    rental_count_non_integer = (
        monthly_hourly["rental_count"]
        .mod(1)
        .ne(0)
        .sum()
    )

    if rental_count_non_integer > 0:
        raise ValueError(
            f"{path.name}의 집계 결과에 "
            "정수가 아닌 이용건수가 존재한다."
        )


    monthly_hourly["station_id"] = (
        monthly_hourly["station_id"]
        .astype("string")
    )

    monthly_hourly["rental_count"] = (
        monthly_hourly["rental_count"]
        .astype("int64")
    )


    monthly_hourly.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )


    if not output_path.exists():
        raise FileNotFoundError(
            f"월별 집계 파일이 생성되지 않았다: {output_path}"
        )


    file_quality["output_file"] = str(
        output_path.relative_to(ROOT)
    )

    file_quality["output_file_size_mb"] = round(
        output_path.stat().st_size / (1024**2),
        2,
    )

    monthly_quality_records.append(file_quality)


    # 실행이 중단되더라도 완료된 월의 품질 기록을 보존한다.
    quality_checkpoint = pd.DataFrame(
        monthly_quality_records
    ).copy()

    quality_checkpoint["year_month"] = (
        quality_checkpoint["year_month"]
        .astype(str)
    )

    quality_checkpoint.to_csv(
        monthly_quality_path,
        index=False,
        encoding="utf-8-sig",
    )


    print("\n[월별 처리 완료]")
    print(
        f"원천 행 수: "
        f"{file_quality['raw_row_count']:,}행"
    )
    print(
        f"집계 후 행 수: "
        f"{file_quality['aggregated_row_count']:,}행"
    )
    print(
        f"고유 대여소 수: "
        f"{file_quality['unique_station_count']:,}개"
    )
    print(
        f"이용건수 합계: "
        f"{file_quality['output_rental_count_sum']:,.0f}건"
    )
    print(
        f"저장 파일: "
        f"{output_path.relative_to(ROOT)}"
    )


    del monthly_hourly
    gc.collect()


bike_usage_quality_by_month = (
    pd.DataFrame(monthly_quality_records)
    .sort_values("year_month")
    .reset_index(drop=True)
)


quality_issue_columns = [
    "date_parse_failure",
    "hour_parse_failure",
    "hour_out_of_range",
    "hour_non_integer",
    "station_id_missing",
    "rental_count_parse_failure",
    "negative_rental_count",
    "rental_count_non_integer",
    "outside_period",
    "file_month_mismatch",
]


quality_issue_totals = (
    bike_usage_quality_by_month[
        quality_issue_columns
    ]
    .sum()
    .astype(int)
)


missing_output_files = [
    path
    for path in expected_output_paths
    if not path.exists()
]


display(
    bike_usage_quality_by_month[
        [
            "year_month",
            "raw_row_count",
            "valid_row_count",
            "excluded_row_count",
            "aggregated_row_count",
            "unique_station_count",
            "input_rental_count_sum",
            "output_rental_count_sum",
            "output_file_size_mb",
        ]
    ]
)


print("\n[전체 월별 집계 결과]")
print(
    f"처리 파일 수: "
    f"{len(bike_usage_quality_by_month):,}개"
)
print(
    f"원천 행 수: "
    f"{bike_usage_quality_by_month['raw_row_count'].sum():,}행"
)
print(
    f"유효 행 수: "
    f"{bike_usage_quality_by_month['valid_row_count'].sum():,}행"
)
print(
    f"제외 행 수: "
    f"{bike_usage_quality_by_month['excluded_row_count'].sum():,}행"
)
print(
    f"집계 후 행 수 합계: "
    f"{bike_usage_quality_by_month['aggregated_row_count'].sum():,}행"
)


print("\n[전체 품질 이상 항목]")
for column, value in quality_issue_totals.items():
    print(f"{column}: {value:,}건")


total_input_count = (
    bike_usage_quality_by_month[
        "input_rental_count_sum"
    ].sum()
)

total_output_count = (
    bike_usage_quality_by_month[
        "output_rental_count_sum"
    ].sum()
)


print("\n[전체 이용건수 합계 보존]")
print(f"집계 전 합계: {total_input_count:,.0f}")
print(f"집계 후 합계: {total_output_count:,.0f}")
print(
    "합계 일치 여부: "
    f"{np.isclose(total_input_count, total_output_count)}"
)


print("\n[중간 파일 저장]")
print(
    f"생성 예정 파일 수: "
    f"{len(expected_output_paths):,}개"
)
print(
    f"누락 파일 수: "
    f"{len(missing_output_files):,}개"
)
print(
    f"월별 품질 보고서: "
    f"{monthly_quality_path.relative_to(ROOT)}"
)


if len(bike_usage_quality_by_month) != len(expected_months):
    raise ValueError(
        "처리된 월별 파일 수가 예상 월 수와 일치하지 않는다."
    )

if quality_issue_totals.sum() != 0:
    print(
        "\n주의: 전체 데이터에서 품질 이상 항목이 확인되었다. "
        "다음 단계에서 항목별 원인을 검토해야 한다."
    )

if not np.isclose(
    total_input_count,
    total_output_count,
):
    raise ValueError(
        "전체 집계 전후 이용건수 합계가 일치하지 않는다."
    )

if missing_output_files:
    raise FileNotFoundError(
        "생성되지 않은 월별 집계 파일이 존재한다."
    )


[01/24] 2023-01 처리
원천 파일: 서울특별시 공공자전거 이용정보(시간대별)_2301.csv
  청크   1: 250,000행 → 126,479행
  청크   2: 250,000행 → 123,117행
  청크   3: 250,000행 → 115,757행
  청크   4: 250,000행 → 125,889행
  청크   5: 250,000행 → 150,923행
  청크   6: 223,431행 → 115,790행

[월별 처리 완료]
원천 행 수: 1,473,431행
집계 후 행 수: 755,977행
고유 대여소 수: 2,709개
이용건수 합계: 1,571,434건
저장 파일: data\processed\station_hourly_monthly\station_hourly_202301.csv

[02/24] 2023-02 처리
원천 파일: 서울특별시 공공자전거 이용정보(시간대별)_2302.csv
  청크   1: 250,000행 → 117,900행
  청크   2: 250,000행 → 113,295행
  청크   3: 250,000행 → 107,865행
  청크   4: 250,000행 → 109,144행
  청크   5: 250,000행 → 104,908행
  청크   6: 250,000행 → 112,813행
  청크   7: 250,000행 → 101,560행
  청크   8: 250,000행 → 109,654행
  청크   9: 61,454행 → 22,015행

[월별 처리 완료]
원천 행 수: 2,061,454행
집계 후 행 수: 897,582행
고유 대여소 수: 2,715개
이용건수 합계: 2,231,457건
저장 파일: data\processed\station_hourly_monthly\station_hourly_202302.csv

[03/24] 2023-03 처리
원천 파일: 서울특별시 공공자전거 이용정보(시간대별)_2303.csv
  청크   1: 250,000행 → 100,718행
  청크   2: 250,000행 → 97,723행


,year_month,raw_row_count,valid_row_count,excluded_row_count,aggregated_row_count,unique_station_count,input_rental_count_sum,output_rental_count_sum,output_file_size_mb
0,2023-01,1473431,1473431,0,755977,2709,1571434.0,1571434.0,20.92
1,2023-02,2061454,2061454,0,897582,2715,2231457.0,2231457.0,24.84
2,2023-03,3461757,3461757,0,1162346,2723,3885426.0,3885426.0,32.20
3,2023-04,3601047,3601047,0,1129551,2721,4081995.0,4081995.0,31.31
4,2023-05,4303099,4303099,0,1201550,2724,4952798.0,4952798.0,33.33
5,2023-06,4322767,4322767,0,1224866,2717,4933508.0,4933508.0,33.97
6,2023-07,3508728,3508728,0,1128307,2733,3931894.0,3931894.0,31.27
7,2023-08,3529592,3529592,0,1153894,2729,3935325.0,3935325.0,31.97
8,2023-09,4014342,4014342,0,1168616,2729,4548903.0,4548903.0,32.41
9,2023-10,4662766,4662766,0,1303271,2734,5293068.0,5293068.0,36.15



[전체 월별 집계 결과]
처리 파일 수: 24개
원천 행 수: 79,670,647행
유효 행 수: 79,670,641행
제외 행 수: 6행
집계 후 행 수 합계: 26,145,337행

[전체 품질 이상 항목]
date_parse_failure: 0건
hour_parse_failure: 0건
hour_out_of_range: 0건
hour_non_integer: 0건
station_id_missing: 0건
rental_count_parse_failure: 0건
negative_rental_count: 0건
rental_count_non_integer: 6건
outside_period: 0건
file_month_mismatch: 0건

[전체 이용건수 합계 보존]
집계 전 합계: 88,750,382
집계 후 합계: 88,750,382
합계 일치 여부: True

[중간 파일 저장]
생성 예정 파일 수: 24개
누락 파일 수: 0개
월별 품질 보고서: data\processed\bike_usage_quality_by_month.csv

주의: 전체 데이터에서 품질 이상 항목이 확인되었다. 다음 단계에서 항목별 원인을 검토해야 한다.


## 12. 2023년 12월 품질 이상 행 상세 점검

전체 원천 데이터 79,670,647행 중 2023년 12월 파일에서만 정수가 아닌 이용건수 6행이 발견되었다. 공공자전거 이용건수는 건수를 의미하므로 원칙적으로 음이 아닌 정수여야 한다.

해당 행을 임의로 반올림하거나 삭제하기 전에 원본 문자열, 날짜, 시간, 대여소, 이용자 구분 변수 및 숫자 변환 결과를 확인한다.

동일 파일에서 문자 디코딩 대체문자도 발견되었으므로, 대체문자가 발생한 컬럼과 표본을 함께 점검한다. 이를 통해 수요 집계에 사용되는 핵심 변수의 손상 여부와 보조 문자열 변수의 인코딩 문제를 구분한다.


In [13]:
problem_year_month = pd.Period(
    "2023-12",
    freq="M",
)

problem_file = next(
    path
    for path in bike_usage_files
    if extract_year_month(path) == problem_year_month
)

problem_encoding = detect_csv_encoding(
    problem_file
)


issue_row_records = []
replacement_count_by_column = {}
replacement_sample_records = []

row_offset = 0
replacement_sample_limit = 20


reader = pd.read_csv(
    problem_file,
    encoding=problem_encoding,
    encoding_errors="replace",
    dtype="string",
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


for chunk_number, chunk in enumerate(
    reader,
    start=1,
):
    chunk = normalize_columns(chunk)

    source_row_number = pd.Series(
        np.arange(
            row_offset + 2,
            row_offset + len(chunk) + 2,
        ),
        index=chunk.index,
    )


    # 이용건수 원문과 숫자 변환 결과를 비교한다.
    rental_count_raw = (
        chunk["이용건수"]
        .astype("string")
        .str.strip()
    )

    rental_count_numeric = pd.to_numeric(
        rental_count_raw,
        errors="coerce",
    )


    rental_count_parse_failure = (
        rental_count_raw.notna()
        & rental_count_numeric.isna()
    )

    rental_count_non_integer = (
        rental_count_numeric.notna()
        & rental_count_numeric.mod(1).ne(0)
    )

    rental_count_issue = (
        rental_count_parse_failure
        | rental_count_non_integer
    )


    if rental_count_issue.any():
        issue_rows = (
            chunk.loc[rental_count_issue]
            .copy()
        )

        issue_rows.insert(
            0,
            "source_row_number",
            source_row_number.loc[
                rental_count_issue
            ].to_numpy(),
        )

        issue_rows.insert(
            1,
            "chunk_number",
            chunk_number,
        )

        issue_rows["이용건수_원문"] = (
            rental_count_raw.loc[
                rental_count_issue
            ].to_numpy()
        )

        issue_rows["이용건수_숫자변환"] = (
            rental_count_numeric.loc[
                rental_count_issue
            ].to_numpy()
        )

        issue_rows["quality_issue"] = np.where(
            rental_count_non_integer.loc[
                rental_count_issue
            ],
            "비정수 이용건수",
            "이용건수 숫자 변환 실패",
        )

        issue_row_records.append(issue_rows)


    # 디코딩 대체문자의 컬럼별 발생 건수를 계산한다.
    string_chunk = chunk.astype("string")

    for column in string_chunk.columns:
        replacement_mask = (
            string_chunk[column]
            .str.contains(
                "\ufffd",
                regex=False,
                na=False,
            )
        )

        replacement_count = int(
            string_chunk[column]
            .str.count("\ufffd")
            .fillna(0)
            .sum()
        )

        if replacement_count == 0:
            continue

        replacement_count_by_column[column] = (
            replacement_count_by_column.get(
                column,
                0,
            )
            + replacement_count
        )


        remaining_sample_count = (
            replacement_sample_limit
            - len(replacement_sample_records)
        )

        if remaining_sample_count > 0:
            sample_indices = (
                string_chunk.index[
                    replacement_mask
                ][:remaining_sample_count]
            )

            for index in sample_indices:
                replacement_sample_records.append(
                    {
                        "source_row_number": int(
                            source_row_number.loc[index]
                        ),
                        "chunk_number": chunk_number,
                        "column_name": column,
                        "column_value": (
                            string_chunk
                            .loc[index, column]
                        ),
                        "rental_date": (
                            string_chunk
                            .loc[index, "대여일자"]
                        ),
                        "rental_hour": (
                            string_chunk
                            .loc[index, "대여시간"]
                        ),
                        "station_id": (
                            string_chunk
                            .loc[index, "대여소번호"]
                        ),
                        "station_name": (
                            string_chunk
                            .loc[index, "대여소명"]
                        ),
                        "rental_count": (
                            string_chunk
                            .loc[index, "이용건수"]
                        ),
                    }
                )


    row_offset += len(chunk)

    print(
        f"청크 {chunk_number:>2} 점검 완료: "
        f"{len(chunk):,}행"
    )


if issue_row_records:
    rental_count_issue_rows = (
        pd.concat(
            issue_row_records,
            ignore_index=True,
        )
        .sort_values("source_row_number")
        .reset_index(drop=True)
    )
else:
    rental_count_issue_rows = pd.DataFrame()


replacement_summary = (
    pd.DataFrame(
        [
            {
                "column_name": column,
                "replacement_count": count,
            }
            for column, count
            in replacement_count_by_column.items()
        ]
    )
    .sort_values(
        "replacement_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


replacement_samples = pd.DataFrame(
    replacement_sample_records
)


key_source_columns = [
    "대여일자",
    "대여시간",
    "대여소번호",
    "이용건수",
]

replacement_in_key_columns = sum(
    replacement_count_by_column.get(
        column,
        0,
    )
    for column in key_source_columns
)


print("\n[점검 대상 파일]")
print(f"파일명: {problem_file.name}")
print(f"인코딩: {problem_encoding}")
print(f"전체 점검 행 수: {row_offset:,}행")


print("\n[이용건수 품질 이상]")
print(
    "품질 이상 행 수: "
    f"{len(rental_count_issue_rows):,}행"
)

if not rental_count_issue_rows.empty:
    print(
        "비정수 이용건수: "
        f"{(
            rental_count_issue_rows['quality_issue']
            == '비정수 이용건수'
        ).sum():,}행"
    )

    print(
        "숫자 변환 실패: "
        f"{(
            rental_count_issue_rows['quality_issue']
            == '이용건수 숫자 변환 실패'
        ).sum():,}행"
    )

    print("\n[이용건수 원문별 빈도]")
    display(
        rental_count_issue_rows[
            "이용건수_원문"
        ]
        .value_counts(
            dropna=False,
        )
        .rename_axis(
            "이용건수_원문"
        )
        .reset_index(
            name="row_count"
        )
    )

    print("\n[품질 이상 원본 행]")
    display(rental_count_issue_rows)


print("\n[디코딩 대체문자]")
print(
    "전체 대체문자 수: "
    f"{sum(replacement_count_by_column.values()):,}건"
)
print(
    "핵심 변수 대체문자 수: "
    f"{replacement_in_key_columns:,}건"
)

display(replacement_summary)


if not replacement_samples.empty:
    print("\n[디코딩 대체문자 표본]")
    display(replacement_samples)

청크  1 점검 완료: 250,000행
청크  2 점검 완료: 250,000행
청크  3 점검 완료: 250,000행
청크  4 점검 완료: 250,000행
청크  5 점검 완료: 250,000행
청크  6 점검 완료: 250,000행
청크  7 점검 완료: 250,000행
청크  8 점검 완료: 204,959행

[점검 대상 파일]
파일명: 서울특별시 공공자전거 이용정보(시간대별)_2312.csv
인코딩: cp949
전체 점검 행 수: 1,954,959행

[이용건수 품질 이상]
품질 이상 행 수: 6행
비정수 이용건수: 6행
숫자 변환 실패: 0행

[이용건수 원문별 빈도]


,이용건수_원문,row_count
0,29.73,1
1,6.45,1
2,14.94,1
3,134.22,1
4,68.47,1
5,53.95,1



[품질 이상 원본 행]


,source_row_number,chunk_number,대여일자,대여시간,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분),이용건수_원문,이용건수_숫자변환,quality_issue
0,995058,4,2023-12-12,17,05752,5752. 풍납백제문화공원 옆 인근,정기권,<NA>,"60??,1""",29.73,0.27,1154.98,7,<NA>,29.73,29.73,비정수 이용건수
1,1057203,5,2023-12-13,8,01173,1173. 강서구청사거리(SH타워),"?ㅁ瘦?,M""",30대,1,6.45,0.05,220.00,1,<NA>,6.45,6.45,비정수 이용건수
2,1183970,5,2023-12-15,16,01669,"1669. 중계역 3번출??,정기권""",M,~10대,1,14.94,0.15,628.75,7,<NA>,14.94,14.94,비정수 이용건수
3,1270953,6,2023-12-18,5,04406,"4406. 장곡초등학교 ??,정기권""",M,40대,1,134.22,1.00,4290.52,97,<NA>,134.22,134.22,비정수 이용건수
4,1712693,7,2023-12-27,13,02508,2508. 양재이스타빌 앞,"정기??,M""",50대,1,68.47,0.57,2470.00,13,<NA>,68.47,68.47,비정수 이용건수
5,1911502,8,2023-12-29,23,01447,1447. 면목역 3번출구,"일일??,M""",50대,1,53.95,0.49,2095.98,13,<NA>,53.95,53.95,비정수 이용건수



[디코딩 대체문자]
전체 대체문자 수: 163건
핵심 변수 대체문자 수: 0건


,column_name,replacement_count
0,대여소명,163



[디코딩 대체문자 표본]


,source_row_number,chunk_number,column_name,column_value,rental_date,rental_hour,station_id,station_name,rental_count
0,4488,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,5,03109,3109.북한산더�?106동 앞,1
1,16189,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,8,03109,3109.북한산더�?106동 앞,1
2,16809,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,8,03109,3109.북한산더�?106동 앞,1
3,17755,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,8,03109,3109.북한산더�?106동 앞,1
4,21791,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,8,03109,3109.북한산더�?106동 앞,1
5,51007,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,16,03109,3109.북한산더�?106동 앞,1
6,61375,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,18,03109,3109.북한산더�?106동 앞,1
7,72086,1,대여소명,3109.북한산더�?106동 앞,2023-12-01,19,03109,3109.북한산더�?106동 앞,1
8,89463,1,대여소명,3109.북한산더�?106동 앞,2023-12-02,2,03109,3109.북한산더�?106동 앞,1
9,90790,1,대여소명,3109.북한산더�?106동 앞,2023-12-02,4,03109,3109.북한산더�?106동 앞,1


## 13. 2023년 12월 파싱 오류 행 교정

2023년 12월 원천 파일에서 발견된 비정수 이용건수 6행은 실제 소수형 이용건수가 아니라, 손상된 따옴표와 쉼표로 인해 CSV 컬럼이 한 칸씩 밀린 파싱 오류이다.

원본 행을 검토한 결과 6행 모두 실제 이용건수는 1건이며, 현재 이용건수 컬럼에 기록된 값은 실제 운동량에 해당한다. 이후 운동량, 탄소량, 이동거리 및 이용시간도 순차적으로 한 칸씩 이동한 구조가 확인된다.

오류 행을 삭제하면 실제 대여 수요 6건이 손실되므로, 원본 파일명과 원본 행 번호를 기준으로 이용건수만 1건으로 복원한다. 수정 대상과 수정 전후 값은 별도의 교정 이력 파일에 저장하여 재현성과 추적 가능성을 확보한다.

같은 파일에서 발견된 디코딩 대체문자 163건은 모두 대여소명에서 발생했다. 대여소 ID와 이용건수 등 수요 집계 핵심 변수에는 대체문자가 없으며, 대여소명은 후속 단계에서 대여소 마스터 정보를 기준으로 부여하므로 수요 집계에는 영향을 주지 않는다.

In [14]:
CORRECTION_REASON = (
    "손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 "
    "실제 이용건수 1건으로 복원"
)


# 원문 검토를 통해 확정한 교정 대상이다.
correction_spec = pd.DataFrame(
    {
        "source_row_number": [
            995058,
            1057203,
            1183970,
            1270953,
            1712693,
            1911502,
        ],
        "observed_rental_count_raw": [
            "29.73",
            "6.45",
            "14.94",
            "134.22",
            "68.47",
            "53.95",
        ],
        "corrected_rental_count": [
            1,
            1,
            1,
            1,
            1,
            1,
        ],
    }
)

correction_spec["source_file"] = problem_file.name
correction_spec["correction_reason"] = CORRECTION_REASON

correction_lookup = (
    correction_spec
    .set_index("source_row_number")
)


corrected_chunk_results = []
corrected_chunk_quality_records = []
applied_correction_records = []

row_offset = 0

decode_replacement_count = 0
decode_replacement_in_key_columns = 0

key_source_columns = [
    "대여일자",
    "대여시간",
    "대여소번호",
    "이용건수",
]


reader = pd.read_csv(
    problem_file,
    encoding=problem_encoding,
    encoding_errors="replace",
    dtype="string",
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


for chunk_number, chunk in enumerate(
    reader,
    start=1,
):
    chunk = normalize_columns(chunk)

    source_row_number = pd.Series(
        np.arange(
            row_offset + 2,
            row_offset + len(chunk) + 2,
        ),
        index=chunk.index,
    )


    # 교정 전 디코딩 대체문자의 발생 위치를 기록한다.
    string_chunk = chunk.astype("string")

    chunk_replacement_count = int(
        string_chunk
        .apply(
            lambda series: (
                series
                .str.count("\ufffd")
                .fillna(0)
                .sum()
            )
        )
        .sum()
    )

    chunk_key_replacement_count = int(
        string_chunk[key_source_columns]
        .apply(
            lambda series: (
                series
                .str.count("\ufffd")
                .fillna(0)
                .sum()
            )
        )
        .sum()
    )

    decode_replacement_count += (
        chunk_replacement_count
    )

    decode_replacement_in_key_columns += (
        chunk_key_replacement_count
    )


    # 현재 청크에 포함된 교정 대상 행을 찾는다.
    correction_mask = source_row_number.isin(
        correction_lookup.index
    )

    correction_indices = (
        chunk.index[correction_mask]
    )


    for index in correction_indices:
        current_source_row = int(
            source_row_number.loc[index]
        )

        expected_raw_value = str(
            correction_lookup.loc[
                current_source_row,
                "observed_rental_count_raw",
            ]
        )

        actual_raw_value = str(
            chunk.loc[index, "이용건수"]
        ).strip()

        corrected_value = int(
            correction_lookup.loc[
                current_source_row,
                "corrected_rental_count",
            ]
        )


        # 예상한 원문과 실제 원문이 다르면 잘못된 행을
        # 수정할 위험이 있으므로 실행을 중단한다.
        if actual_raw_value != expected_raw_value:
            raise ValueError(
                f"원본 {current_source_row}행의 이용건수가 "
                "사전 확인값과 일치하지 않는다.\n"
                f"예상값: {expected_raw_value}\n"
                f"실제값: {actual_raw_value}"
            )


        applied_correction_records.append(
            {
                "source_file": problem_file.name,
                "source_row_number": current_source_row,
                "rental_date": chunk.loc[
                    index,
                    "대여일자",
                ],
                "rental_hour": chunk.loc[
                    index,
                    "대여시간",
                ],
                "station_id": chunk.loc[
                    index,
                    "대여소번호",
                ],
                "station_name_raw": chunk.loc[
                    index,
                    "대여소명",
                ],
                "observed_rental_count_raw": (
                    actual_raw_value
                ),
                "corrected_rental_count": (
                    corrected_value
                ),
                "correction_reason": (
                    CORRECTION_REASON
                ),
            }
        )


        # 수요 집계에 사용하는 이용건수만 복원한다.
        chunk.loc[
            index,
            "이용건수",
        ] = str(corrected_value)


    corrected_chunk_hourly, corrected_quality = (
        process_bike_chunk(
            chunk=chunk,
            source_file=problem_file.name,
            file_year_month=problem_year_month,
            period_start=period_start,
            period_end_exclusive=period_end_exclusive,
        )
    )


    corrected_quality[
        "decode_replacement_count"
    ] = chunk_replacement_count

    corrected_quality[
        "decode_replacement_in_key_columns"
    ] = chunk_key_replacement_count


    corrected_chunk_results.append(
        corrected_chunk_hourly
    )

    corrected_chunk_quality_records.append(
        corrected_quality
    )


    print(
        f"청크 {chunk_number:>2}: "
        f"{len(chunk):,}행, "
        f"교정 {len(correction_indices):,}행"
    )


    row_offset += len(chunk)


applied_corrections = (
    pd.DataFrame(applied_correction_records)
    .sort_values("source_row_number")
    .reset_index(drop=True)
)


expected_correction_rows = set(
    correction_spec["source_row_number"]
)

applied_correction_rows = set(
    applied_corrections["source_row_number"]
)


if applied_correction_rows != expected_correction_rows:
    missing_rows = sorted(
        expected_correction_rows
        - applied_correction_rows
    )

    unexpected_rows = sorted(
        applied_correction_rows
        - expected_correction_rows
    )

    raise ValueError(
        "교정 대상 행이 정확히 적용되지 않았다.\n"
        f"누락 행: {missing_rows}\n"
        f"예상하지 않은 행: {unexpected_rows}"
    )


if len(applied_corrections) != 6:
    raise ValueError(
        "교정 행 수가 예상한 6행과 일치하지 않는다."
    )


# 교정된 2023년 12월 데이터를 시간·대여소별로 재집계한다.
corrected_monthly_hourly = (
    pd.concat(
        corrected_chunk_results,
        ignore_index=True,
    )
    .groupby(
        ["datetime", "station_id"],
        as_index=False,
    )["rental_count"]
    .sum()
    .sort_values(
        ["datetime", "station_id"]
    )
    .reset_index(drop=True)
)


corrected_chunk_quality = pd.DataFrame(
    corrected_chunk_quality_records
)


corrected_input_sum = float(
    corrected_chunk_quality[
        "input_rental_count_sum"
    ].sum()
)

corrected_output_sum = float(
    corrected_monthly_hourly[
        "rental_count"
    ].sum()
)


if not np.isclose(
    corrected_input_sum,
    corrected_output_sum,
):
    raise ValueError(
        "교정 후 집계 전후 이용건수 합계가 일치하지 않는다."
    )


if (
    corrected_chunk_quality[
        "rental_count_non_integer"
    ].sum()
    != 0
):
    raise ValueError(
        "교정 후에도 비정수 이용건수가 존재한다."
    )


if (
    corrected_chunk_quality[
        "excluded_row_count"
    ].sum()
    != 0
):
    raise ValueError(
        "교정 후에도 집계에서 제외된 행이 존재한다."
    )


if decode_replacement_in_key_columns != 0:
    raise ValueError(
        "수요 집계 핵심 변수에서 디코딩 대체문자가 발견되었다."
    )


# 저장 전 자료형을 명확하게 설정한다.
corrected_monthly_hourly["station_id"] = (
    corrected_monthly_hourly["station_id"]
    .astype("string")
)

corrected_monthly_hourly["rental_count"] = (
    corrected_monthly_hourly["rental_count"]
    .astype("int64")
)


corrected_output_path = (
    monthly_output_dir
    / "station_hourly_202312.csv"
)

correction_log_path = (
    processed_dir
    / "bike_usage_manual_corrections.csv"
)


# 기존 2023년 12월 중간 결과를 교정 결과로 교체한다.
corrected_monthly_hourly.to_csv(
    corrected_output_path,
    index=False,
    encoding="utf-8-sig",
)

applied_corrections.to_csv(
    correction_log_path,
    index=False,
    encoding="utf-8-sig",
)


# 월별 품질 보고서의 2023년 12월 기록을 갱신한다.
quality_report = pd.read_csv(
    monthly_quality_path,
    encoding="utf-8-sig",
)

quality_report["year_month"] = (
    quality_report["year_month"]
    .astype("string")
)


target_mask = (
    quality_report["year_month"]
    .eq("2023-12")
)


if target_mask.sum() != 1:
    raise ValueError(
        "월별 품질 보고서에서 2023-12 기록을 "
        "정확히 하나 찾지 못했다."
    )


corrected_quality_values = {
    "chunk_count": len(
        corrected_chunk_quality
    ),
    "raw_row_count": int(
        corrected_chunk_quality[
            "raw_row_count"
        ].sum()
    ),
    "valid_row_count": int(
        corrected_chunk_quality[
            "valid_row_count"
        ].sum()
    ),
    "excluded_row_count": int(
        corrected_chunk_quality[
            "excluded_row_count"
        ].sum()
    ),
    "date_parse_failure": int(
        corrected_chunk_quality[
            "date_parse_failure"
        ].sum()
    ),
    "hour_parse_failure": int(
        corrected_chunk_quality[
            "hour_parse_failure"
        ].sum()
    ),
    "hour_out_of_range": int(
        corrected_chunk_quality[
            "hour_out_of_range"
        ].sum()
    ),
    "hour_non_integer": int(
        corrected_chunk_quality[
            "hour_non_integer"
        ].sum()
    ),
    "station_id_missing": int(
        corrected_chunk_quality[
            "station_id_missing"
        ].sum()
    ),
    "station_name_missing": int(
        corrected_chunk_quality[
            "station_name_missing"
        ].sum()
    ),
    "rental_count_parse_failure": int(
        corrected_chunk_quality[
            "rental_count_parse_failure"
        ].sum()
    ),
    "negative_rental_count": int(
        corrected_chunk_quality[
            "negative_rental_count"
        ].sum()
    ),
    "rental_count_non_integer": int(
        corrected_chunk_quality[
            "rental_count_non_integer"
        ].sum()
    ),
    "zero_rental_count": int(
        corrected_chunk_quality[
            "zero_rental_count"
        ].sum()
    ),
    "outside_period": int(
        corrected_chunk_quality[
            "outside_period"
        ].sum()
    ),
    "file_month_mismatch": int(
        corrected_chunk_quality[
            "file_month_mismatch"
        ].sum()
    ),
    "decode_replacement_count": (
        decode_replacement_count
    ),
    "decode_replacement_in_key_columns": (
        decode_replacement_in_key_columns
    ),
    "input_rental_count_sum": (
        corrected_input_sum
    ),
    "output_rental_count_sum": (
        corrected_output_sum
    ),
    "aggregated_row_count": len(
        corrected_monthly_hourly
    ),
    "unique_station_count": int(
        corrected_monthly_hourly[
            "station_id"
        ].nunique()
    ),
    "datetime_min": (
        corrected_monthly_hourly[
            "datetime"
        ].min()
    ),
    "datetime_max": (
        corrected_monthly_hourly[
            "datetime"
        ].max()
    ),
    "output_file": str(
        corrected_output_path.relative_to(ROOT)
    ),
    "output_file_size_mb": round(
        corrected_output_path.stat().st_size
        / (1024**2),
        2,
    ),
    "corrected_row_count": len(
        applied_corrections
    ),
    "correction_method": (
        "원본 행 번호 기반 이용건수 복원"
    ),
}


for column, value in corrected_quality_values.items():
    if column not in quality_report.columns:
        quality_report[column] = pd.NA

    quality_report.loc[
        target_mask,
        column,
    ] = value


quality_report.to_csv(
    monthly_quality_path,
    index=False,
    encoding="utf-8-sig",
)


# 갱신된 전체 기간 합계를 계산한다.
total_input_sum = pd.to_numeric(
    quality_report[
        "input_rental_count_sum"
    ],
    errors="coerce",
).sum()

total_output_sum = pd.to_numeric(
    quality_report[
        "output_rental_count_sum"
    ],
    errors="coerce",
).sum()

total_excluded_rows = pd.to_numeric(
    quality_report[
        "excluded_row_count"
    ],
    errors="coerce",
).sum()

total_non_integer_rows = pd.to_numeric(
    quality_report[
        "rental_count_non_integer"
    ],
    errors="coerce",
).sum()


print("\n[교정 적용 결과]")
display(applied_corrections)

print(
    f"교정 행 수: "
    f"{len(applied_corrections):,}행"
)
print(
    f"교정값 합계: "
    f"{applied_corrections['corrected_rental_count'].sum():,}건"
)


print("\n[2023년 12월 교정 후 결과]")
print(
    f"원천 행 수: "
    f"{corrected_quality_values['raw_row_count']:,}행"
)
print(
    f"유효 행 수: "
    f"{corrected_quality_values['valid_row_count']:,}행"
)
print(
    f"제외 행 수: "
    f"{corrected_quality_values['excluded_row_count']:,}행"
)
print(
    f"비정수 이용건수: "
    f"{corrected_quality_values['rental_count_non_integer']:,}건"
)
print(
    f"집계 후 행 수: "
    f"{corrected_quality_values['aggregated_row_count']:,}행"
)
print(
    f"이용건수 합계: "
    f"{corrected_output_sum:,.0f}건"
)
print(
    f"디코딩 대체문자: "
    f"{decode_replacement_count:,}건"
)
print(
    f"핵심 변수 대체문자: "
    f"{decode_replacement_in_key_columns:,}건"
)


print("\n[전체 기간 갱신 결과]")
print(
    f"전체 집계 전 합계: "
    f"{total_input_sum:,.0f}건"
)
print(
    f"전체 집계 후 합계: "
    f"{total_output_sum:,.0f}건"
)
print(
    "전체 합계 일치 여부: "
    f"{np.isclose(total_input_sum, total_output_sum)}"
)
print(
    f"전체 제외 행 수: "
    f"{int(total_excluded_rows):,}행"
)
print(
    f"전체 비정수 이용건수: "
    f"{int(total_non_integer_rows):,}건"
)


print("\n[저장 파일]")
print(
    corrected_output_path.relative_to(ROOT)
)
print(
    correction_log_path.relative_to(ROOT)
)
print(
    monthly_quality_path.relative_to(ROOT)
)


if not np.isclose(
    total_output_sum,
    88_750_388,
):
    raise ValueError(
        "교정 후 전체 이용건수 합계가 예상값 "
        "88,750,388건과 일치하지 않는다."
    )

청크  1: 250,000행, 교정 0행
청크  2: 250,000행, 교정 0행
청크  3: 250,000행, 교정 0행
청크  4: 250,000행, 교정 1행
청크  5: 250,000행, 교정 2행
청크  6: 250,000행, 교정 1행
청크  7: 250,000행, 교정 1행
청크  8: 204,959행, 교정 1행

[교정 적용 결과]


,source_file,source_row_number,rental_date,rental_hour,station_id,station_name_raw,observed_rental_count_raw,corrected_rental_count,correction_reason
0,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,995058,2023-12-12,17,05752,5752. 풍납백제문화공원 옆 인근,29.73,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원
1,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,1057203,2023-12-13,8,01173,1173. 강서구청사거리(SH타워),6.45,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원
2,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,1183970,2023-12-15,16,01669,"1669. 중계역 3번출??,정기권""",14.94,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원
3,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,1270953,2023-12-18,5,04406,"4406. 장곡초등학교 ??,정기권""",134.22,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원
4,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,1712693,2023-12-27,13,02508,2508. 양재이스타빌 앞,68.47,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원
5,서울특별시 공공자전거 이용정보(시간대별)_2312.csv,1911502,2023-12-29,23,01447,1447. 면목역 3번출구,53.95,1,손상된 따옴표와 쉼표로 인한 CSV 컬럼 이동을 확인하고 실제 이용건수 1건으로 복원


교정 행 수: 6행
교정값 합계: 6건

[2023년 12월 교정 후 결과]
원천 행 수: 1,954,959행
유효 행 수: 1,954,959행
제외 행 수: 0행
비정수 이용건수: 0건
집계 후 행 수: 850,806행
이용건수 합계: 2,104,399건
디코딩 대체문자: 163건
핵심 변수 대체문자: 0건

[전체 기간 갱신 결과]
전체 집계 전 합계: 88,750,388건
전체 집계 후 합계: 88,750,388건
전체 합계 일치 여부: True
전체 제외 행 수: 0행
전체 비정수 이용건수: 0건

[저장 파일]
data\processed\station_hourly_monthly\station_hourly_202312.csv
data\processed\bike_usage_manual_corrections.csv
data\processed\bike_usage_quality_by_month.csv


## 14. 공공자전거 이용정보와 대여소 마스터의 ID 정합성 점검

월별 집계 데이터의 대여소 ID와 대여소 마스터의 대여소 ID가 동일한 형식으로 기록되어 있는지 확인한다.

공공자전거 이용정보에는 `05752`와 같이 앞자리에 0이 포함된 ID가 존재할 수 있으나, 대여소 마스터에는 동일한 대여소가 `5752`로 기록될 수 있다. 이를 서로 다른 대여소로 처리하면 정상 GPS 대여소의 수요가 미결합 데이터로 잘못 분류된다.

이에 숫자로 구성된 대여소 ID의 앞자리 0을 제거한 표준 ID를 생성하고 다음 항목을 점검한다.

- 원본 ID 기준 대여소 마스터 결합률
- 표준 ID 기준 대여소 마스터 결합률
- 정상 GPS 대여소의 이용건수
- GPS 미입력 대여소의 이용건수
- 대여소 마스터에 존재하지 않는 이용정보 ID
- ID 표준화로 인해 서로 다른 대여소가 하나의 ID로 중복되는지 여부

이 단계에서는 대여소별 시간당 수요를 모두 메모리에 결합하지 않고, 월별 파일을 차례로 읽어 대여소 단위로 축약한다.

In [15]:
def canonicalize_station_id(
    series: pd.Series,
) -> pd.Series:
    """대여소 ID의 숫자 부분을 추출하고 앞자리 0을 제거한다."""

    station_id = clean_station_id(series)

    canonical_id = (
        station_id
        .str.lstrip("0")
        .replace("", "0")
    )

    return canonical_id


# 정상 GPS 및 비정상 GPS 대여소 파일을 불러온다.
station_master_valid = pd.read_csv(
    clean_station_path,
    encoding="utf-8-sig",
    dtype={"station_id": "string"},
)

station_master_invalid = pd.read_csv(
    invalid_station_path,
    encoding="utf-8-sig",
    dtype={"station_id": "string"},
)


station_master_valid["gps_status"] = "정상 GPS"
station_master_invalid["gps_status"] = "GPS 미입력"


master_columns = [
    "station_id",
    "station_name",
    "latitude",
    "longitude",
    "gps_status",
]


station_master_all = pd.concat(
    [
        station_master_valid[master_columns],
        station_master_invalid[master_columns],
    ],
    ignore_index=True,
)


station_master_all["station_id_raw"] = (
    station_master_all["station_id"]
    .astype("string")
)

station_master_all["station_id_canonical"] = (
    canonicalize_station_id(
        station_master_all["station_id_raw"]
    )
)


# ID 표준화 과정에서 중복 ID가 생성되는지 확인한다.
canonical_master_duplicates = (
    station_master_all.loc[
        station_master_all[
            "station_id_canonical"
        ].duplicated(keep=False)
    ]
    .sort_values("station_id_canonical")
    .reset_index(drop=True)
)


print("[대여소 마스터 ID 표준화]")
print(
    f"전체 대여소 수: "
    f"{len(station_master_all):,}개"
)
print(
    "표준 ID 고유값 수: "
    f"{station_master_all['station_id_canonical'].nunique():,}개"
)
print(
    "표준화 후 중복 ID 행 수: "
    f"{len(canonical_master_duplicates):,}행"
)


if not canonical_master_duplicates.empty:
    print("\n[표준화 후 중복 ID]")
    display(canonical_master_duplicates)

    raise ValueError(
        "대여소 마스터의 ID 표준화 과정에서 "
        "중복 ID가 생성되었다."
    )


# 월별 대여소 시간당 수요 파일을 확인한다.
monthly_station_files = sorted(
    monthly_output_dir.glob(
        "station_hourly_*.csv"
    )
)


if len(monthly_station_files) != 24:
    raise ValueError(
        "월별 대여소 시간당 수요 파일 수가 "
        f"24개가 아니다: {len(monthly_station_files)}개"
    )


monthly_station_summary_records = []


for file_index, path in enumerate(
    monthly_station_files,
    start=1,
):
    year_month_text = (
        path.stem
        .replace("station_hourly_", "")
    )

    monthly_data = pd.read_csv(
        path,
        encoding="utf-8-sig",
        usecols=[
            "datetime",
            "station_id",
            "rental_count",
        ],
        dtype={
            "station_id": "string",
            "rental_count": "int64",
        },
        parse_dates=["datetime"],
    )


    monthly_data["station_id_raw"] = (
        monthly_data["station_id"]
        .astype("string")
    )

    monthly_data["station_id_canonical"] = (
        canonicalize_station_id(
            monthly_data["station_id_raw"]
        )
    )


    monthly_station_summary = (
        monthly_data
        .groupby(
            [
                "station_id_raw",
                "station_id_canonical",
            ],
            as_index=False,
        )
        .agg(
            hourly_row_count=(
                "rental_count",
                "size",
            ),
            rental_count=(
                "rental_count",
                "sum",
            ),
        )
    )

    monthly_station_summary[
        "year_month"
    ] = pd.Period(
        year_month_text,
        freq="M",
    )


    monthly_station_summary_records.append(
        monthly_station_summary
    )


    print(
        f"[{file_index:02d}/24] "
        f"{year_month_text}: "
        f"{len(monthly_data):,}행 → "
        f"{len(monthly_station_summary):,}개 대여소"
    )


    del monthly_data
    del monthly_station_summary
    gc.collect()


station_usage_by_month = pd.concat(
    monthly_station_summary_records,
    ignore_index=True,
)


# 표준 ID 하나에 여러 원본 ID가 연결되는지 확인한다.
station_usage_summary = (
    station_usage_by_month
    .groupby(
        "station_id_canonical",
        as_index=False,
    )
    .agg(
        raw_id_count=(
            "station_id_raw",
            "nunique",
        ),
        raw_station_ids=(
            "station_id_raw",
            lambda values: "|".join(
                sorted(set(values.dropna()))
            ),
        ),
        active_month_count=(
            "year_month",
            "nunique",
        ),
        first_active_month=(
            "year_month",
            "min",
        ),
        last_active_month=(
            "year_month",
            "max",
        ),
        hourly_row_count=(
            "hourly_row_count",
            "sum",
        ),
        rental_count=(
            "rental_count",
            "sum",
        ),
    )
)


usage_canonical_duplicates = (
    station_usage_summary.loc[
        station_usage_summary[
            "raw_id_count"
        ] > 1
    ]
    .copy()
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


master_lookup = (
    station_master_all[
        [
            "station_id_canonical",
            "station_id_raw",
            "station_name",
            "latitude",
            "longitude",
            "gps_status",
        ]
    ]
    .rename(
        columns={
            "station_id_raw": (
                "master_station_id_raw"
            )
        }
    )
)


station_id_reconciliation = (
    station_usage_summary
    .merge(
        master_lookup,
        on="station_id_canonical",
        how="left",
        validate="one_to_one",
    )
)


station_id_reconciliation["match_status"] = (
    station_id_reconciliation[
        "gps_status"
    ]
    .fillna("대여소 마스터 미등록")
)


# 원본 문자열을 그대로 비교한 결합 여부도 계산한다.
master_raw_id_set = set(
    station_master_all[
        "station_id_raw"
    ].dropna()
)

station_id_reconciliation[
    "raw_id_match"
] = (
    station_id_reconciliation[
        "raw_station_ids"
    ]
    .isin(master_raw_id_set)
)

station_id_reconciliation[
    "canonical_id_match"
] = (
    station_id_reconciliation[
        "gps_status"
    ]
    .notna()
)


reconciliation_summary = (
    station_id_reconciliation
    .groupby(
        "match_status",
        as_index=False,
    )
    .agg(
        unique_station_count=(
            "station_id_canonical",
            "nunique",
        ),
        hourly_row_count=(
            "hourly_row_count",
            "sum",
        ),
        rental_count=(
            "rental_count",
            "sum",
        ),
    )
)


total_rental_count = (
    reconciliation_summary[
        "rental_count"
    ].sum()
)


reconciliation_summary[
    "rental_count_share_pct"
] = (
    reconciliation_summary[
        "rental_count"
    ]
    .div(total_rental_count)
    .mul(100)
)


reconciliation_summary = (
    reconciliation_summary
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


unmatched_station_summary = (
    station_id_reconciliation.loc[
        station_id_reconciliation[
            "match_status"
        ].eq("대여소 마스터 미등록")
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


raw_match_station_count = int(
    station_id_reconciliation[
        "raw_id_match"
    ].sum()
)

canonical_match_station_count = int(
    station_id_reconciliation[
        "canonical_id_match"
    ].sum()
)


print("\n[이용정보 대여소 ID 현황]")
print(
    "표준화 전 고유 ID 수: "
    f"{station_usage_by_month['station_id_raw'].nunique():,}개"
)
print(
    "표준화 후 고유 ID 수: "
    f"{station_usage_summary['station_id_canonical'].nunique():,}개"
)
print(
    "여러 원본 ID가 하나로 통합된 표준 ID 수: "
    f"{len(usage_canonical_duplicates):,}개"
)


print("\n[대여소 마스터 결합 결과]")
print(
    "원본 ID 기준 결합 대여소: "
    f"{raw_match_station_count:,}개"
)
print(
    "표준 ID 기준 결합 대여소: "
    f"{canonical_match_station_count:,}개"
)
print(
    "마스터 미등록 대여소: "
    f"{len(unmatched_station_summary):,}개"
)


print("\n[수요 분류 결과]")
display(reconciliation_summary)


if not usage_canonical_duplicates.empty:
    print("\n[여러 원본 ID가 통합된 표준 ID]")
    display(
        usage_canonical_duplicates.head(20)
    )


if not unmatched_station_summary.empty:
    print("\n[마스터 미등록 대여소 상위 20개]")
    display(
        unmatched_station_summary[
            [
                "station_id_canonical",
                "raw_station_ids",
                "active_month_count",
                "first_active_month",
                "last_active_month",
                "hourly_row_count",
                "rental_count",
            ]
        ].head(20)
    )


# 전체 수요 합계가 이전 단계의 검증값과 일치하는지 확인한다.
if total_rental_count != 88_750_388:
    raise ValueError(
        "대여소 ID 정합성 점검의 전체 이용건수 합계가 "
        "88,750,388건과 일치하지 않는다."
    )

[대여소 마스터 ID 표준화]
전체 대여소 수: 3,418개
표준 ID 고유값 수: 3,418개
표준화 후 중복 ID 행 수: 0행
[01/24] 202301: 755,977행 → 2,709개 대여소
[02/24] 202302: 897,582행 → 2,715개 대여소
[03/24] 202303: 1,162,346행 → 2,723개 대여소
[04/24] 202304: 1,129,551행 → 2,721개 대여소
[05/24] 202305: 1,201,550행 → 2,724개 대여소
[06/24] 202306: 1,224,866행 → 2,717개 대여소
[07/24] 202307: 1,128,307행 → 2,733개 대여소
[08/24] 202308: 1,153,894행 → 2,729개 대여소
[09/24] 202309: 1,168,616행 → 2,729개 대여소
[10/24] 202310: 1,303,271행 → 2,734개 대여소
[11/24] 202311: 1,085,399행 → 2,726개 대여소
[12/24] 202312: 850,806행 → 2,730개 대여소
[13/24] 202401: 866,055행 → 2,729개 대여소
[14/24] 202402: 823,783행 → 2,728개 대여소
[15/24] 202403: 1,059,382행 → 2,730개 대여소
[16/24] 202404: 1,202,199행 → 2,730개 대여소
[17/24] 202405: 1,196,745행 → 2,732개 대여소
[18/24] 202406: 1,227,850행 → 2,734개 대여소
[19/24] 202407: 1,109,957행 → 2,733개 대여소
[20/24] 202408: 1,201,740행 → 2,722개 대여소
[21/24] 202409: 1,153,678행 → 2,735개 대여소
[22/24] 202410: 1,231,645행 → 2,744개 대여소
[23/24] 202411: 1,073,234행 → 2,735개 대여소
[24/24] 202412: 

,match_status,unique_station_count,hourly_row_count,rental_count,rental_count_share_pct
0,정상 GPS,1852,17814709,62045868,69.910532
1,대여소 마스터 미등록,958,7747673,24668072,27.794889
2,GPS 미입력,55,582955,2036448,2.294579



[마스터 미등록 대여소 상위 20개]


,station_id_canonical,raw_station_ids,active_month_count,first_active_month,last_active_month,hourly_row_count,rental_count
0,4217,04217,24,2023-01,2024-12,15257,272649
1,3533,03533,24,2023-01,2024-12,15255,159984
2,5052,05052,24,2023-01,2024-12,14352,137004
3,4565,04565,24,2023-01,2024-12,13698,133407
4,4870,04870,24,2023-01,2024-12,15768,133280
5,3798,03798,24,2023-01,2024-12,16145,132345
6,5768,05768,23,2023-02,2024-12,15099,131464
7,5082,05082,24,2023-01,2024-12,14481,123912
8,5858,05858,24,2023-01,2024-12,15383,123763
9,3518,03518,24,2023-01,2024-12,15188,122347


## 15. 대여소 마스터 식별자 의미 검증

표준화된 대여소 ID를 이용해 결합한 결과, 전체 이용정보 대여소 2,865개 중 958개가 대여소 마스터와 연결되지 않았다. 미등록으로 분류된 상위 대여소 대부분이 분석 기간 24개월 동안 지속적으로 운영되었으며 이용건수도 많으므로, 이를 단순한 폐쇄 또는 과거 대여소로 간주할 수 없다.

서울시 대여소 마스터의 `대여소_ID`는 실제 대여소번호가 아니라 `ST-숫자` 형태의 내부 관리 식별자일 가능성이 있다. 반면 공공자전거 이용정보의 `대여소번호`는 대여소명 앞에 표시되는 운영 번호와 대응할 수 있다.

이에 다음 후보 식별자를 이용정보의 대여소번호와 비교한다.

1. 마스터의 `대여소_ID`에서 숫자만 추출한 값
2. `주소2`의 문자열 시작 부분에서 추출한 대여소 번호
3. `주소1` 또는 최종 대여소 표기의 시작 부분에서 추출한 대여소 번호

각 후보별 고유 ID 수, 중복 여부, 이용정보 결합 대여소 수 및 이용건수 커버리지를 비교하여 올바른 결합 키를 확정한다. 이 단계에서는 기존 정제 파일이나 월별 수요 파일을 수정하지 않는다.


In [16]:
def extract_station_number_from_name(
    series: pd.Series,
) -> pd.Series:
    """대여소명 시작 부분에 포함된 운영 대여소 번호를 추출한다."""

    text = (
        series
        .astype("string")
        .str.strip()
    )

    extracted = text.str.extract(
        r"^\s*0*(\d{1,5})\s*(?:[.\-_:]|번|\s)",
        expand=False,
    )

    return extracted.astype("string")


# 원본 마스터 식별자와 주소 정보를 다시 구성한다.
master_identifier_check = pd.DataFrame(
    {
        "master_id_raw": (
            station_raw["대여소_ID"]
            .astype("string")
            .str.strip()
        ),
        "address_primary": (
            station_raw["주소1"]
            .astype("string")
            .str.strip()
        ),
        "address_detail": (
            station_raw["주소2"]
            .astype("string")
            .str.strip()
        ),
        "latitude": pd.to_numeric(
            station_raw["위도"],
            errors="coerce",
        ),
        "longitude": pd.to_numeric(
            station_raw["경도"],
            errors="coerce",
        ),
    }
)


# 기존 방식: 대여소_ID에서 숫자 부분을 추출한다.
master_identifier_check[
    "candidate_master_id"
] = canonicalize_station_id(
    master_identifier_check[
        "master_id_raw"
    ]
)


# 주소2와 주소1의 시작 부분에서 운영 대여소 번호를 추출한다.
master_identifier_check[
    "candidate_address_detail"
] = extract_station_number_from_name(
    master_identifier_check[
        "address_detail"
    ]
)

master_identifier_check[
    "candidate_address_primary"
] = extract_station_number_from_name(
    master_identifier_check[
        "address_primary"
    ]
)


# 주소2를 우선하고, 추출하지 못한 경우 주소1 후보를 사용한다.
master_identifier_check[
    "candidate_address_combined"
] = (
    master_identifier_check[
        "candidate_address_detail"
    ]
    .fillna(
        master_identifier_check[
            "candidate_address_primary"
        ]
    )
)


# 대여소_ID 형식의 분포를 확인한다.
master_identifier_check[
    "master_id_pattern"
] = np.select(
    [
        master_identifier_check[
            "master_id_raw"
        ].str.match(
            r"^ST-\d+$",
            case=False,
            na=False,
        ),
        master_identifier_check[
            "master_id_raw"
        ].str.match(
            r"^\d+$",
            na=False,
        ),
    ],
    [
        "ST-숫자",
        "숫자만",
    ],
    default="기타 형식",
)


master_id_pattern_summary = (
    master_identifier_check
    .groupby(
        "master_id_pattern",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "row_count",
        }
    )
    .sort_values(
        "row_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


# 이용정보의 대여소 ID와 전체 이용건수를 준비한다.
usage_id_set = set(
    station_usage_summary[
        "station_id_canonical"
    ].dropna()
)

usage_demand_lookup = (
    station_usage_summary
    .set_index(
        "station_id_canonical"
    )["rental_count"]
)


def evaluate_master_id_candidate(
    data: pd.DataFrame,
    candidate_column: str,
    candidate_name: str,
) -> dict:
    """마스터 ID 후보와 이용정보 ID 간 결합 성능을 계산한다."""

    candidate = (
        data[candidate_column]
        .dropna()
        .astype("string")
    )

    candidate_unique = set(candidate)

    matched_usage_ids = (
        usage_id_set
        & candidate_unique
    )

    matched_rental_count = float(
        usage_demand_lookup.loc[
            usage_demand_lookup.index.isin(
                matched_usage_ids
            )
        ].sum()
    )

    duplicate_row_count = int(
        data.loc[
            data[candidate_column].notna(),
            candidate_column,
        ]
        .duplicated(keep=False)
        .sum()
    )

    duplicate_id_count = int(
        data.loc[
            data[candidate_column].notna()
            & data[
                candidate_column
            ].duplicated(keep=False),
            candidate_column,
        ]
        .nunique()
    )

    return {
        "candidate_name": candidate_name,
        "source_column": candidate_column,
        "parsed_row_count": int(
            data[candidate_column]
            .notna()
            .sum()
        ),
        "unique_candidate_id_count": len(
            candidate_unique
        ),
        "duplicate_candidate_id_count": (
            duplicate_id_count
        ),
        "duplicate_row_count": (
            duplicate_row_count
        ),
        "matched_usage_station_count": len(
            matched_usage_ids
        ),
        "matched_rental_count": (
            matched_rental_count
        ),
        "matched_rental_share_pct": (
            matched_rental_count
            / 88_750_388
            * 100
        ),
    }


candidate_evaluation = pd.DataFrame(
    [
        evaluate_master_id_candidate(
            data=master_identifier_check,
            candidate_column=(
                "candidate_master_id"
            ),
            candidate_name=(
                "대여소_ID 숫자 추출"
            ),
        ),
        evaluate_master_id_candidate(
            data=master_identifier_check,
            candidate_column=(
                "candidate_address_detail"
            ),
            candidate_name=(
                "주소2 시작 번호"
            ),
        ),
        evaluate_master_id_candidate(
            data=master_identifier_check,
            candidate_column=(
                "candidate_address_primary"
            ),
            candidate_name=(
                "주소1 시작 번호"
            ),
        ),
        evaluate_master_id_candidate(
            data=master_identifier_check,
            candidate_column=(
                "candidate_address_combined"
            ),
            candidate_name=(
                "주소2 우선·주소1 보완"
            ),
        ),
    ]
)


candidate_evaluation = (
    candidate_evaluation
    .sort_values(
        [
            "matched_rental_count",
            "matched_usage_station_count",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print("[마스터 대여소_ID 형식]")
display(master_id_pattern_summary)


print("\n[원본 마스터 표본]")
display(
    master_identifier_check[
        [
            "master_id_raw",
            "candidate_master_id",
            "address_primary",
            "address_detail",
            "candidate_address_detail",
            "candidate_address_primary",
            "latitude",
            "longitude",
        ]
    ].head(30)
)


print("\n[결합 키 후보 비교]")
display(candidate_evaluation)


# 주소 기반 후보가 존재하는 행의 표본을 확인한다.
address_candidate_sample = (
    master_identifier_check.loc[
        master_identifier_check[
            "candidate_address_combined"
        ].notna(),
        [
            "master_id_raw",
            "candidate_master_id",
            "address_primary",
            "address_detail",
            "candidate_address_combined",
            "latitude",
            "longitude",
        ],
    ]
    .head(30)
)


print("\n[주소 기반 대여소번호 추출 표본]")
display(address_candidate_sample)


# 기존 ID 후보와 주소 기반 후보가 서로 다른 행을 확인한다.
identifier_difference = (
    master_identifier_check.loc[
        master_identifier_check[
            "candidate_address_combined"
        ].notna()
        & master_identifier_check[
            "candidate_master_id"
        ].ne(
            master_identifier_check[
                "candidate_address_combined"
            ]
        ),
        [
            "master_id_raw",
            "candidate_master_id",
            "address_primary",
            "address_detail",
            "candidate_address_combined",
            "latitude",
            "longitude",
        ],
    ]
    .reset_index(drop=True)
)


print("\n[내부 관리 ID와 주소 기반 번호가 다른 행]")
print(
    f"행 수: "
    f"{len(identifier_difference):,}행"
)

display(
    identifier_difference.head(30)
)


[마스터 대여소_ID 형식]


,master_id_pattern,row_count
0,ST-숫자,3418



[원본 마스터 표본]


,master_id_raw,candidate_master_id,address_primary,address_detail,candidate_address_detail,candidate_address_primary,latitude,longitude
0,ST-999,999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,<NA>,<NA>,0.000000,0.000000
1,ST-998,998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,<NA>,<NA>,0.000000,0.000000
2,ST-997,997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,<NA>,<NA>,37.534390,126.869598
3,ST-996,996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,<NA>,<NA>,37.524334,126.850548
4,ST-995,995,서울특별시 양천구 중앙로 153 공중화장실,<NA>,<NA>,<NA>,37.510597,126.857323
5,ST-994,994,서울특별시 양천구 목동서로161,SBS방송국,<NA>,<NA>,37.529163,126.872749
6,ST-993,993,서울특별시 양천구 신월로 342-1 구두수선대19,<NA>,<NA>,<NA>,37.521511,126.857384
7,ST-992,992,서울특별시 마포구 마포대로 163,서울신용보증재단,<NA>,<NA>,37.549061,126.954178
8,ST-991,991,서울특별시 마포구 성암로330,DMC첨단산업센터,<NA>,<NA>,37.584503,126.885597
9,ST-990,990,서울특별시 마포구 상암동 1715-29,롯데하이마트 (상암월드컵점),<NA>,<NA>,37.573620,126.898048



[결합 키 후보 비교]


,candidate_name,source_column,parsed_row_count,unique_candidate_id_count,duplicate_candidate_id_count,duplicate_row_count,matched_usage_station_count,matched_rental_count,matched_rental_share_pct
0,대여소_ID 숫자 추출,candidate_master_id,3418,3418,0,0,1907,64082316.0,72.205111
1,주소2 시작 번호,candidate_address_detail,38,33,3,8,12,370418.0,0.417371
2,주소2 우선·주소1 보완,candidate_address_combined,38,33,3,8,12,370418.0,0.417371
3,주소1 시작 번호,candidate_address_primary,0,0,0,0,0,0.0,0.000000



[주소 기반 대여소번호 추출 표본]


,master_id_raw,candidate_master_id,address_primary,address_detail,candidate_address_combined,latitude,longitude
75,ST-930,930,서울특별시 송파구 오금로 지하 499,3번출구,3,37.493343,127.144730
201,ST-817,817,서울특별시 강남구 선릉로 지하 228,8.삼호@ 2동 ( 간선도로),8,37.493759,127.045898
221,ST-8,8,서울특별시 마포구 독막로 4,381-1,381,37.548645,126.912827
331,ST-7,7,서울특별시 마포구 양화로 48,394-95,394,37.550007,126.914825
464,ST-58,58,서울특별시 영등포구 의사당대로 88,27-1,27,37.521362,126.923462
471,ST-573,573,서울특별시 중랑구 용마산로 285,581 번지,581,37.577782,127.090187
497,ST-55,55,서울특별시 영등포구 국제금융로 24,24-3,24,37.524612,126.927834
511,ST-537,537,서울특별시 강서구 곰달래로 51,355-48,355,37.530338,126.838257
553,ST-5,5,서울특별시 마포구 월드컵로 79,378-18,378,37.554951,126.910835
596,ST-46,46,서울특별시 영등포구 국회대로76길 10,13-1,13,37.531239,126.921333



[내부 관리 ID와 주소 기반 번호가 다른 행]
행 수: 36행


,master_id_raw,candidate_master_id,address_primary,address_detail,candidate_address_combined,latitude,longitude
0,ST-930,930,서울특별시 송파구 오금로 지하 499,3번출구,3,37.493343,127.144730
1,ST-817,817,서울특별시 강남구 선릉로 지하 228,8.삼호@ 2동 ( 간선도로),8,37.493759,127.045898
2,ST-8,8,서울특별시 마포구 독막로 4,381-1,381,37.548645,126.912827
3,ST-7,7,서울특별시 마포구 양화로 48,394-95,394,37.550007,126.914825
4,ST-58,58,서울특별시 영등포구 의사당대로 88,27-1,27,37.521362,126.923462
5,ST-573,573,서울특별시 중랑구 용마산로 285,581 번지,581,37.577782,127.090187
6,ST-55,55,서울특별시 영등포구 국제금융로 24,24-3,24,37.524612,126.927834
7,ST-537,537,서울특별시 강서구 곰달래로 51,355-48,355,37.530338,126.838257
8,ST-5,5,서울특별시 마포구 월드컵로 79,378-18,378,37.554951,126.910835
9,ST-46,46,서울특별시 영등포구 국회대로76길 10,13-1,13,37.531239,126.921333


## 16. 숫자 ID 직접 결합의 유효성 검증

대여소 마스터의 `대여소_ID`는 모두 `ST-숫자` 형식으로 구성되어 있으며, 주소 컬럼의 시작 숫자는 대여소번호가 아니라 지번, 출구 번호 및 건물 번호를 의미하는 것으로 확인되었다.

따라서 마스터의 `ST-숫자`에서 추출한 숫자와 이용정보의 `대여소번호`를 직접 연결하는 방식이 실제로 동일한 대여소를 가리키는지 검증해야 한다.

2024년 12월 이용정보에서 대여소번호별 대표 대여소명을 추출하고, 숫자 ID가 동일한 마스터 대여소명과 비교한다. 비교 과정에서는 공백과 문장부호를 제거한 표준 이름을 생성하고 다음 항목을 확인한다.

* 숫자 ID가 동일한 대여소 수
* 대여소명이 정확히 일치하는 경우
* 한쪽 이름이 다른 쪽 이름에 포함되는 경우
* 숫자 ID는 같지만 대여소명이 일치하지 않는 경우
* 대여소명 기준 정확 일치로 연결 가능한 대여소 수

숫자 ID가 같더라도 대여소명이 일치하지 않는다면 두 식별자는 서로 다른 체계로 판단하며, 해당 직접 결합 방식은 폐기한다.


In [17]:
def normalize_station_name_for_match(
    series: pd.Series,
) -> pd.Series:
    """대여소명을 비교 가능한 문자열로 표준화한다."""

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace("\ufffd", "", regex=False)
    )

    # 이용정보 대여소명 앞의 대여소번호를 제거한다.
    normalized = normalized.str.replace(
        r"^\s*0*\d+\s*[.\-_:]?\s*",
        "",
        regex=True,
    )

    # 공백과 문장부호를 제거하고 문자와 숫자만 유지한다.
    normalized = normalized.str.replace(
        r"[^0-9a-z가-힣]",
        "",
        regex=True,
    )

    return normalized.replace("", pd.NA)


# 최신 월인 2024년 12월 파일을 검증 대상으로 선택한다.
validation_file = next(
    path
    for path in bike_usage_files
    if extract_year_month(path)
    == pd.Period("2024-12", freq="M")
)

validation_encoding = detect_csv_encoding(
    validation_file
)


# 대여소 ID별 대여소명 출현 빈도를 청크 단위로 집계한다.
station_name_count_records = []


reader = pd.read_csv(
    validation_file,
    encoding=validation_encoding,
    encoding_errors="replace",
    usecols=[
        "대여소번호",
        "대여소명",
    ],
    dtype="string",
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


for chunk_number, chunk in enumerate(
    reader,
    start=1,
):
    chunk = normalize_columns(chunk)

    chunk["station_id_canonical"] = (
        canonicalize_station_id(
            chunk["대여소번호"]
        )
    )

    chunk["usage_station_name"] = (
        chunk["대여소명"]
        .astype("string")
        .str.strip()
    )

    chunk_name_counts = (
        chunk
        .dropna(
            subset=[
                "station_id_canonical",
                "usage_station_name",
            ]
        )
        .groupby(
            [
                "station_id_canonical",
                "usage_station_name",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size": "name_frequency",
            }
        )
    )

    station_name_count_records.append(
        chunk_name_counts
    )

    print(
        f"청크 {chunk_number:>2}: "
        f"{len(chunk):,}행 → "
        f"{len(chunk_name_counts):,}개 ID·이름 조합"
    )


station_name_counts = (
    pd.concat(
        station_name_count_records,
        ignore_index=True,
    )
    .groupby(
        [
            "station_id_canonical",
            "usage_station_name",
        ],
        as_index=False,
    )["name_frequency"]
    .sum()
)


# 각 대여소번호에서 가장 자주 등장한 이름을 대표 이름으로 선택한다.
latest_usage_station_names = (
    station_name_counts
    .sort_values(
        [
            "station_id_canonical",
            "name_frequency",
            "usage_station_name",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=["station_id_canonical"],
        keep="first",
    )
    .reset_index(drop=True)
)


latest_usage_station_names[
    "usage_name_normalized"
] = normalize_station_name_for_match(
    latest_usage_station_names[
        "usage_station_name"
    ]
)


# 마스터의 대표 이름은 주소2를 우선하고 주소1로 보완한다.
master_name_validation = (
    master_identifier_check[
        [
            "master_id_raw",
            "candidate_master_id",
            "address_primary",
            "address_detail",
            "latitude",
            "longitude",
        ]
    ]
    .copy()
)


master_name_validation[
    "master_station_name"
] = (
    master_name_validation[
        "address_detail"
    ]
    .fillna(
        master_name_validation[
            "address_primary"
        ]
    )
)


master_name_validation[
    "master_name_normalized"
] = normalize_station_name_for_match(
    master_name_validation[
        "master_station_name"
    ]
)


# 전체 기간 수요를 결합하여 불일치 대여소의 중요도를 확인한다.
usage_demand_for_validation = (
    station_usage_summary[
        [
            "station_id_canonical",
            "rental_count",
            "active_month_count",
        ]
    ]
    .copy()
)


latest_usage_station_names = (
    latest_usage_station_names
    .merge(
        usage_demand_for_validation,
        on="station_id_canonical",
        how="left",
        validate="one_to_one",
    )
)


# 기존 숫자 ID 직접 결합 결과를 재구성한다.
direct_id_name_validation = (
    latest_usage_station_names
    .merge(
        master_name_validation,
        left_on="station_id_canonical",
        right_on="candidate_master_id",
        how="inner",
        validate="one_to_one",
    )
)


direct_id_name_validation[
    "exact_name_match"
] = (
    direct_id_name_validation[
        "usage_name_normalized"
    ]
    .eq(
        direct_id_name_validation[
            "master_name_normalized"
        ]
    )
    & direct_id_name_validation[
        "usage_name_normalized"
    ]
    .notna()
)


def contains_name(
    usage_name: object,
    master_name: object,
) -> bool:
    """두 표준 대여소명 중 하나가 다른 하나에 포함되는지 확인한다."""

    if pd.isna(usage_name) or pd.isna(master_name):
        return False

    usage_text = str(usage_name)
    master_text = str(master_name)

    if min(
        len(usage_text),
        len(master_text),
    ) < 4:
        return False

    return (
        usage_text in master_text
        or master_text in usage_text
    )


direct_id_name_validation[
    "containment_name_match"
] = direct_id_name_validation.apply(
    lambda row: contains_name(
        row["usage_name_normalized"],
        row["master_name_normalized"],
    ),
    axis=1,
)


direct_id_name_validation[
    "any_name_match"
] = (
    direct_id_name_validation[
        "exact_name_match"
    ]
    | direct_id_name_validation[
        "containment_name_match"
    ]
)


direct_validation_summary = pd.DataFrame(
    {
        "validation_item": [
            "숫자 ID 직접 결합 대여소",
            "대여소명 정확 일치",
            "대여소명 포함 관계 일치",
            "정확 또는 포함 관계 일치",
            "대여소명 불일치",
        ],
        "station_count": [
            len(direct_id_name_validation),
            int(
                direct_id_name_validation[
                    "exact_name_match"
                ].sum()
            ),
            int(
                direct_id_name_validation[
                    "containment_name_match"
                ].sum()
            ),
            int(
                direct_id_name_validation[
                    "any_name_match"
                ].sum()
            ),
            int(
                (
                    ~direct_id_name_validation[
                        "any_name_match"
                    ]
                ).sum()
            ),
        ],
    }
)


direct_validation_summary[
    "share_pct"
] = (
    direct_validation_summary[
        "station_count"
    ]
    .div(len(direct_id_name_validation))
    .mul(100)
)


direct_id_mismatch_sample = (
    direct_id_name_validation.loc[
        ~direct_id_name_validation[
            "any_name_match"
        ],
        [
            "station_id_canonical",
            "usage_station_name",
            "master_id_raw",
            "master_station_name",
            "rental_count",
            "active_month_count",
            "latitude",
            "longitude",
        ],
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


direct_id_match_sample = (
    direct_id_name_validation.loc[
        direct_id_name_validation[
            "any_name_match"
        ],
        [
            "station_id_canonical",
            "usage_station_name",
            "master_id_raw",
            "master_station_name",
            "exact_name_match",
            "containment_name_match",
            "rental_count",
        ],
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


# 마스터 이름이 유일한 경우만 이름 기준 정확 결합 후보로 사용한다.
master_name_frequency = (
    master_name_validation[
        "master_name_normalized"
    ]
    .value_counts(
        dropna=True,
    )
)


unique_master_names = set(
    master_name_frequency.loc[
        master_name_frequency.eq(1)
    ].index
)


unique_master_name_lookup = (
    master_name_validation.loc[
        master_name_validation[
            "master_name_normalized"
        ].isin(unique_master_names)
    ]
    .copy()
)


exact_name_link_candidates = (
    latest_usage_station_names
    .merge(
        unique_master_name_lookup,
        left_on="usage_name_normalized",
        right_on="master_name_normalized",
        how="inner",
        validate="many_to_one",
    )
)


exact_name_link_summary = pd.DataFrame(
    {
        "validation_item": [
            "2024년 12월 이용 대여소",
            "마스터 이름 보유 대여소",
            "고유한 마스터 표준 이름",
            "이름 정확 일치 대여소",
            "이름 정확 일치 수요",
            "이름 정확 일치 수요 비율",
        ],
        "value": [
            latest_usage_station_names[
                "station_id_canonical"
            ].nunique(),
            master_name_validation[
                "master_name_normalized"
            ].notna().sum(),
            len(unique_master_names),
            exact_name_link_candidates[
                "station_id_canonical"
            ].nunique(),
            exact_name_link_candidates[
                "rental_count"
            ].sum(),
            (
                exact_name_link_candidates[
                    "rental_count"
                ].sum()
                / 88_750_388
                * 100
            ),
        ],
    }
)


print("\n[숫자 ID 직접 결합 이름 검증]")
display(direct_validation_summary)


print("\n[숫자 ID 직접 결합 불일치 상위 30개]")
display(
    direct_id_mismatch_sample.head(30)
)


print("\n[숫자 ID 직접 결합 일치 표본]")
display(
    direct_id_match_sample.head(20)
)


print("\n[대여소명 정확 결합 가능성]")
display(exact_name_link_summary)


print("\n[대여소명 정확 결합 표본]")
display(
    exact_name_link_candidates[
        [
            "station_id_canonical",
            "usage_station_name",
            "master_id_raw",
            "master_station_name",
            "latitude",
            "longitude",
            "rental_count",
        ]
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .head(30)
)


청크  1: 250,000행 → 2,712개 ID·이름 조합
청크  2: 250,000행 → 2,714개 ID·이름 조합
청크  3: 250,000행 → 2,715개 ID·이름 조합
청크  4: 250,000행 → 2,716개 ID·이름 조합
청크  5: 250,000행 → 2,715개 ID·이름 조합
청크  6: 250,000행 → 2,719개 ID·이름 조합
청크  7: 250,000행 → 2,726개 ID·이름 조합
청크  8: 250,000행 → 2,721개 ID·이름 조합
청크  9: 126,133행 → 2,711개 ID·이름 조합

[숫자 ID 직접 결합 이름 검증]


,validation_item,station_count,share_pct
0,숫자 ID 직접 결합 대여소,1827,100.0
1,대여소명 정확 일치,0,0.0
2,대여소명 포함 관계 일치,0,0.0
3,정확 또는 포함 관계 일치,0,0.0
4,대여소명 불일치,1827,100.0



[숫자 ID 직접 결합 불일치 상위 30개]


,station_id_canonical,usage_station_name,master_id_raw,master_station_name,rental_count,active_month_count,latitude,longitude
0,2715,2715.마곡나루역 2번 출구,ST-2715,서울특별시 노원구 서울 노원구 공릉동 547-1,366191,24,37.625336,127.070190
1,502,502. 자양(뚝섬한강공원)역 1번출구 앞,ST-502,서울특별시 강동구 성내동 434-28,267863,24,37.526794,127.130058
2,2728,2728.마곡나루역 3번 출구,ST-2728,서울특별시 송파구 백제고분로 11 신천중학교,235358,24,37.516403,127.077614
3,1210,1210. 롯데월드타워(잠실역2번출구 쪽),ST-1210,길음8골어린이공원 옆,232232,24,37.608978,127.020248
4,2701,2701. 마곡나루역 5번출구 뒤편,ST-2701,서울특별시 노원구 서울 노원구 덕릉로83길 5,229472,24,37.659557,127.075256
5,1153,"1153. 발산역 1번, 9번 인근 대여소",ST-1153,한강 현대아파트 건너편,216509,24,37.505928,126.969231
6,207,207. 여의나루역 1번출구 앞,ST-207,서강대역 2번출구 앞,199741,24,37.552956,126.934341
7,230,230. 영등포구청역 1번출구,ST-230,서대문역 8번출구 앞,192521,24,37.564777,126.966148
8,2102,2102. 봉림교 교통섬,ST-2102,서울특별시 도봉구 삼양로 590,188934,24,37.655685,127.013382
9,1911,1911. 구로디지털단지역 앞,ST-1911,서울특별시 서초구 신반포로 143 경남쇼핑센터,172696,24,37.504787,126.999924



[숫자 ID 직접 결합 일치 표본]


,station_id_canonical,usage_station_name,master_id_raw,master_station_name,exact_name_match,containment_name_match,rental_count



[대여소명 정확 결합 가능성]


,validation_item,value
0,2024년 12월 이용 대여소,2.734000e+03
1,마스터 이름 보유 대여소,3.405000e+03
2,고유한 마스터 표준 이름,3.213000e+03
3,이름 정확 일치 대여소,6.660000e+02
4,이름 정확 일치 수요,2.241492e+07
5,이름 정확 일치 수요 비율,2.525614e+01



[대여소명 정확 결합 표본]


,station_id_canonical,usage_station_name,master_id_raw,master_station_name,latitude,longitude,rental_count
237,207,207. 여의나루역 1번출구 앞,ST-73,여의나루역 1번출구 앞,37.527157,126.931900,199741
377,2622,2622. 올림픽공원역 3번출구,ST-1720,올림픽공원역 3번출구,37.516258,127.130592,145023
6,1160,1160. 양천향교역 7번출구앞,ST-1251,양천향교역 7번출구 앞,37.567680,126.840897,144099
385,272,272. 당산육갑문,ST-426,당산육갑문,37.535339,126.903679,141912
600,765,765. 오목교역 3번출구,ST-1731,오목교역 3번출구,37.524776,126.875481,138293
495,5052,5052. 마곡역 7번출구,ST-1064,마곡역7번출구,37.558311,126.826523,137004
256,2177,2177. 신대방역 2번 출구,ST-1256,신대방역 2번출구,0.000000,0.000000,134455
375,2620,2620. 송파나루역 4번 출구옆,ST-1730,송파나루역 4번 출구옆,37.509979,127.112312,134382
537,583,583. 청계천 생태교실 앞,ST-376,청계천 생태교실 앞,37.567970,127.046890,126556
433,3518,3518. 군자역 7번출구뒤,ST-1263,군자역 7번출구 뒤,37.556862,127.079140,122347


## 17. 시점별 대여소 정보 파일 구조 점검

2023년과 2024년의 반기별 공공자전거 대여소 정보 파일을 이용해 실제 대여소번호와 GPS 좌표를 연결한다.

파일별로 CSV와 Excel 형식이 혼재할 수 있으며, 기준 시점에 따라 컬럼명과 시트 구성이 다를 수 있다. 따라서 데이터를 통합하기 전에 다음 사항을 확인한다.

- 다운로드된 파일 수와 파일명
- 파일 형식과 용량
- CSV 파일의 문자 인코딩
- Excel 파일의 시트 구성
- 파일별 컬럼명과 표본 데이터
- 대여소번호, 대여소명, 위도 및 경도 후보 컬럼

이 단계에서는 파일 구조만 확인하며 정제 또는 결합은 수행하지 않는다.

In [18]:
station_history_dir = (
    ROOT / "data" / "raw" / "station_history"
)

station_history_dir.mkdir(
    parents=True,
    exist_ok=True,
)


supported_station_suffixes = {
    ".csv",
    ".xlsx",
    ".xls",
}


station_history_files = sorted(
    [
        path
        for path in station_history_dir.iterdir()
        if (
            path.is_file()
            and path.suffix.lower()
            in supported_station_suffixes
        )
    ],
    key=lambda path: path.name,
)


print("[시점별 대여소 정보 파일 목록]")
print(f"폴더: {station_history_dir.relative_to(ROOT)}")
print(f"인식된 파일 수: {len(station_history_files)}개")


file_inventory_records = []


for path in station_history_files:
    file_inventory_records.append(
        {
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "file_size_mb": round(
                path.stat().st_size / (1024**2),
                2,
            ),
        }
    )


station_history_inventory = pd.DataFrame(
    file_inventory_records
)

display(station_history_inventory)


if len(station_history_files) != 4:
    print(
        "\n주의: 예상 파일 수는 4개이지만 "
        f"현재 {len(station_history_files)}개가 인식되었다."
    )


station_history_structure_records = []
station_history_samples = {}


for path in station_history_files:
    suffix = path.suffix.lower()

    print("\n" + "=" * 80)
    print(f"[파일 구조 점검] {path.name}")
    print("=" * 80)


    if suffix == ".csv":
        sample = read_table(
            path,
            nrows=5,
        )

        encoding = sample.attrs.get(
            "source_encoding",
            "unknown",
        )

        sheet_names = None
        selected_sheet = None


    elif suffix in {".xlsx", ".xls"}:
        excel_file = pd.ExcelFile(path)

        sheet_names = excel_file.sheet_names
        selected_sheet = sheet_names[0]

        sample = pd.read_excel(
            path,
            sheet_name=selected_sheet,
            nrows=5,
            dtype="string",
        )

        encoding = "excel"


    else:
        continue


    sample = normalize_columns(sample)

    station_history_samples[path.name] = (
        sample.copy()
    )


    station_history_structure_records.append(
        {
            "file_name": path.name,
            "file_type": suffix,
            "encoding": encoding,
            "sheet_count": (
                len(sheet_names)
                if sheet_names is not None
                else pd.NA
            ),
            "sheet_names": (
                " | ".join(sheet_names)
                if sheet_names is not None
                else pd.NA
            ),
            "selected_sheet": (
                selected_sheet
                if selected_sheet is not None
                else pd.NA
            ),
            "column_count": len(sample.columns),
            "columns": " | ".join(
                map(str, sample.columns)
            ),
        }
    )


    print(f"파일 형식: {suffix}")
    print(f"문자 인코딩: {encoding}")

    if sheet_names is not None:
        print(
            "시트 목록: "
            + " | ".join(sheet_names)
        )
        print(
            f"점검 시트: {selected_sheet}"
        )

    print(
        f"컬럼 수: {len(sample.columns)}개"
    )

    print("\n[컬럼 목록]")

    for column_number, column in enumerate(
        sample.columns,
        start=1,
    ):
        print(
            f"{column_number:>2}. {column}"
        )

    print("\n[표본 데이터]")
    display(sample)


station_history_structure = pd.DataFrame(
    station_history_structure_records
)


print("\n[파일별 구조 요약]")
display(
    station_history_structure[
        [
            "file_name",
            "file_type",
            "encoding",
            "sheet_count",
            "selected_sheet",
            "column_count",
            "columns",
        ]
    ]
)

[시점별 대여소 정보 파일 목록]
폴더: data\raw\station_history
인식된 파일 수: 4개


,file_name,extension,file_size_mb
0,공공자전거 대여소 정보(23.06월 기준).xlsx,.xlsx,0.27
1,공공자전거 대여소 정보(23.12월 기준).xlsx,.xlsx,0.28
2,공공자전거 대여소 정보(24.12월 기준).xlsx,.xlsx,0.28
3,공공자전거 대여소 정보(24.6월 기준).xlsx,.xlsx,0.28



[파일 구조 점검] 공공자전거 대여소 정보(23.06월 기준).xlsx
파일 형식: .xlsx
문자 인코딩: excel
시트 목록: 대여소현황
점검 시트: 대여소현황
컬럼 수: 10개

[컬럼 목록]
 1. 대여소번호
 2. 보관소(대여소)명
 3. 소재지(위치)
 4. Unnamed:_3
 5. Unnamed:_4
 6. Unnamed:_5
 7. 설치시기
 8. 설치형태
 9. Unnamed:_8
10. 운영방식

[표본 데이터]


,대여소번호,보관소(대여소)명,소재지(위치),Unnamed:_3,Unnamed:_4,Unnamed:_5,설치시기,설치형태,Unnamed:_8,운영방식
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,LCD,QR,<NA>
1,<NA>,<NA>,자치구,상세주소,위도,경도,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,거치 대수,거치 대수,<NA>
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.57579422,126.9714508,2015-10-07 00:00:00,<NA>,20,QR



[파일 구조 점검] 공공자전거 대여소 정보(23.12월 기준).xlsx
파일 형식: .xlsx
문자 인코딩: excel
시트 목록: 대여소현황
점검 시트: 대여소현황
컬럼 수: 10개

[컬럼 목록]
 1. 대여소번호
 2. 보관소(대여소)명
 3. 소재지(위치)
 4. Unnamed:_3
 5. Unnamed:_4
 6. Unnamed:_5
 7. 설치시기
 8. 설치형태
 9. Unnamed:_8
10. 운영방식

[표본 데이터]


...

,대여소번호,보관소(대여소)명,소재지(위치),Unnamed:_3,Unnamed:_4,Unnamed:_5,설치시기,설치형태,Unnamed:_8,운영방식
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,LCD,QR,<NA>
1,<NA>,<NA>,자치구,상세주소,위도,경도,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,거치 대수,거치 대수,<NA>
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.57579422,126.97145081,2015-10-07 12:03:46,<NA>,20,QR



[파일 구조 점검] 공공자전거 대여소 정보(24.6월 기준).xlsx
파일 형식: .xlsx
문자 인코딩: excel
시트 목록: 대여소현황
점검 시트: 대여소현황
컬럼 수: 10개

[컬럼 목록]
 1. 대여소번호
 2. 보관소(대여소)명
 3. 소재지(위치)
 4. Unnamed:_3
 5. Unnamed:_4
 6. Unnamed:_5
 7. 설치시기
 8. 설치형태
 9. Unnamed:_8
10. 운영방식

[표본 데이터]


,대여소번호,보관소(대여소)명,소재지(위치),Unnamed:_3,Unnamed:_4,Unnamed:_5,설치시기,설치형태,Unnamed:_8,운영방식
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,LCD,QR,<NA>
1,<NA>,<NA>,자치구,상세주소,위도,경도,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,거치 대수,거치 대수,<NA>
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130 경복궁역 7번출구 앞,37.57579422,126.97145081,2015-10-07 12:03:46,<NA>,20,QR



[파일별 구조 요약]


,file_name,file_type,encoding,sheet_count,selected_sheet,column_count,columns
0,공공자전거 대여소 정보(23.06월 기준).xlsx,.xlsx,excel,1,대여소현황,10,대여소번호 | 보관소(대여소)명 | 소재지(위치) | Unnamed:_3 | Unn...
1,공공자전거 대여소 정보(23.12월 기준).xlsx,.xlsx,excel,1,대여소현황,10,대여소번호 | 보관소(대여소)명 | 소재지(위치) | Unnamed:_3 | Unn...
2,공공자전거 대여소 정보(24.12월 기준).xlsx,.xlsx,excel,1,대여소현황,10,대여소번호 | 보관소(대여소)명 | 소재지(위치) | Unnamed:_3 | Unn...
3,공공자전거 대여소 정보(24.6월 기준).xlsx,.xlsx,excel,1,대여소현황,10,대여소번호 | 보관소(대여소)명 | 소재지(위치) | Unnamed:_3 | Unn...


## 18. 시점별 대여소 정보 표준화 및 결합 가능성 검증

2023년 6월·12월과 2024년 6월·12월 기준 대여소 정보 파일은 동일한 다중 헤더 구조를 가진다. 첫 네 행은 하위 헤더와 안내 정보이며, 실제 대여소 데이터는 대여소번호가 입력된 행부터 시작한다.

각 파일을 다음과 같은 공통 구조로 표준화한다.

- 대여소번호
- 대여소명
- 자치구
- 상세주소
- 위도
- 경도
- 설치일자
- LCD 거치대 수
- QR 거치대 수
- 운영방식
- 기준 시점

표준화 후 파일별 대여소 수, 중복 대여소번호, 결측치, 비정상 GPS 좌표를 점검한다. 또한 네 시점의 합집합을 이용정보 대여소번호와 연결하여 대여소 수와 이용건수 기준 GPS 확보 비율을 확인한다.

동일한 대여소번호의 좌표가 시점에 따라 달라질 수 있으므로 이 단계에서는 하나의 대표 좌표를 확정하지 않는다.

In [19]:
def extract_station_snapshot_period(
    path: Path,
) -> pd.Period:
    """파일명에서 대여소 정보 기준 연월을 추출한다."""

    match = re.search(
        r"\((\d{2})[.](\d{1,2})월\s*기준\)",
        path.name,
    )

    if match is None:
        raise ValueError(
            f"파일명에서 기준 연월을 추출하지 못했다: "
            f"{path.name}"
        )

    year = 2000 + int(match.group(1))
    month = int(match.group(2))

    return pd.Period(
        f"{year}-{month:02d}",
        freq="M",
    )


def normalize_history_station_id(
    series: pd.Series,
) -> pd.Series:
    """Excel 대여소번호를 앞자리 0이 없는 문자열 ID로 변환한다."""

    raw_id = (
        series
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
    )

    numeric_id = pd.to_numeric(
        raw_id,
        errors="coerce",
    )

    integer_mask = (
        numeric_id.notna()
        & numeric_id.mod(1).eq(0)
    )

    normalized_id = pd.Series(
        pd.NA,
        index=series.index,
        dtype="string",
    )

    normalized_id.loc[integer_mask] = (
        numeric_id.loc[integer_mask]
        .astype("int64")
        .astype("string")
    )

    return normalized_id


def read_station_history_file(
    path: Path,
) -> tuple[pd.DataFrame, dict]:
    """시점별 대여소 정보 Excel 파일을 공통 구조로 변환한다."""

    snapshot_period = (
        extract_station_snapshot_period(path)
    )

    raw = pd.read_excel(
        path,
        sheet_name="대여소현황",
        header=0,
        dtype="string",
    )

    raw = normalize_columns(raw)


    required_columns = [
        "대여소번호",
        "보관소(대여소)명",
        "소재지(위치)",
        "Unnamed:_3",
        "Unnamed:_4",
        "Unnamed:_5",
        "설치시기",
        "설치형태",
        "Unnamed:_8",
        "운영방식",
    ]


    missing_columns = sorted(
        set(required_columns)
        - set(raw.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{path.name}에 필수 컬럼이 없다: "
            f"{missing_columns}"
        )


    raw_station_id = (
        raw["대여소번호"]
        .astype("string")
        .str.strip()
    )

    station_id = (
        normalize_history_station_id(
            raw_station_id
        )
    )


    # 안내 행과 다중 헤더 행은 대여소번호가 없으므로 제외한다.
    data_row_mask = station_id.notna()


    data = pd.DataFrame(
        {
            "snapshot_period": (
                snapshot_period
            ),
            "snapshot_date": (
                snapshot_period.end_time.normalize()
            ),
            "source_file": path.name,
            "station_id_raw": (
                raw_station_id.loc[
                    data_row_mask
                ]
            ),
            "station_id": (
                station_id.loc[
                    data_row_mask
                ]
            ),
            "station_name": (
                raw.loc[
                    data_row_mask,
                    "보관소(대여소)명",
                ]
                .astype("string")
                .str.strip()
            ),
            "district": (
                raw.loc[
                    data_row_mask,
                    "소재지(위치)",
                ]
                .astype("string")
                .str.strip()
            ),
            "address": (
                raw.loc[
                    data_row_mask,
                    "Unnamed:_3",
                ]
                .astype("string")
                .str.strip()
            ),
            "latitude": pd.to_numeric(
                raw.loc[
                    data_row_mask,
                    "Unnamed:_4",
                ],
                errors="coerce",
            ),
            "longitude": pd.to_numeric(
                raw.loc[
                    data_row_mask,
                    "Unnamed:_5",
                ],
                errors="coerce",
            ),
            "installation_date": (
                pd.to_datetime(
                    raw.loc[
                        data_row_mask,
                        "설치시기",
                    ],
                    errors="coerce",
                )
            ),
            "lcd_rack_count": (
                pd.to_numeric(
                    raw.loc[
                        data_row_mask,
                        "설치형태",
                    ],
                    errors="coerce",
                )
            ),
            "qr_rack_count": (
                pd.to_numeric(
                    raw.loc[
                        data_row_mask,
                        "Unnamed:_8",
                    ],
                    errors="coerce",
                )
            ),
            "operation_method": (
                raw.loc[
                    data_row_mask,
                    "운영방식",
                ]
                .astype("string")
                .str.strip()
            ),
        }
    ).reset_index(drop=True)


    data["valid_gps"] = (
        data["latitude"].between(
            37.4,
            37.8,
        )
        & data["longitude"].between(
            126.7,
            127.3,
        )
    )


    duplicate_station_mask = (
        data["station_id"]
        .duplicated(keep=False)
    )


    quality_record = {
        "snapshot_period": str(
            snapshot_period
        ),
        "source_file": path.name,
        "raw_excel_row_count": len(raw),
        "station_row_count": len(data),
        "unique_station_count": int(
            data["station_id"].nunique()
        ),
        "duplicate_station_id_count": int(
            data.loc[
                duplicate_station_mask,
                "station_id",
            ].nunique()
        ),
        "duplicate_station_row_count": int(
            duplicate_station_mask.sum()
        ),
        "station_name_missing": int(
            (
                data["station_name"].isna()
                | data["station_name"].eq("")
            ).sum()
        ),
        "district_missing": int(
            (
                data["district"].isna()
                | data["district"].eq("")
            ).sum()
        ),
        "address_missing": int(
            (
                data["address"].isna()
                | data["address"].eq("")
            ).sum()
        ),
        "latitude_missing": int(
            data["latitude"].isna().sum()
        ),
        "longitude_missing": int(
            data["longitude"].isna().sum()
        ),
        "valid_gps_count": int(
            data["valid_gps"].sum()
        ),
        "invalid_gps_count": int(
            (~data["valid_gps"]).sum()
        ),
        "latitude_min": (
            data["latitude"].min()
        ),
        "latitude_max": (
            data["latitude"].max()
        ),
        "longitude_min": (
            data["longitude"].min()
        ),
        "longitude_max": (
            data["longitude"].max()
        ),
    }


    return data, quality_record


# 기준 시점 순서로 파일을 정렬한다.
station_history_files_sorted = sorted(
    station_history_files,
    key=extract_station_snapshot_period,
)


station_history_frames = []
station_history_quality_records = []


for file_index, path in enumerate(
    station_history_files_sorted,
    start=1,
):
    history_data, quality_record = (
        read_station_history_file(path)
    )

    station_history_frames.append(
        history_data
    )

    station_history_quality_records.append(
        quality_record
    )


    print(
        f"[{file_index:02d}/04] "
        f"{quality_record['snapshot_period']}: "
        f"{quality_record['station_row_count']:,}행, "
        f"고유 ID "
        f"{quality_record['unique_station_count']:,}개, "
        f"정상 GPS "
        f"{quality_record['valid_gps_count']:,}개"
    )


station_history_all = (
    pd.concat(
        station_history_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "snapshot_date",
            "station_id",
        ]
    )
    .reset_index(drop=True)
)


station_history_quality = pd.DataFrame(
    station_history_quality_records
)


print("\n[파일별 대여소 정보 품질]")
display(station_history_quality)


# 동일 기준 시점에서 중복된 대여소번호를 확인한다.
snapshot_duplicate_rows = (
    station_history_all.loc[
        station_history_all.duplicated(
            subset=[
                "snapshot_period",
                "station_id",
            ],
            keep=False,
        )
    ]
    .sort_values(
        [
            "snapshot_period",
            "station_id",
        ]
    )
    .reset_index(drop=True)
)


print("\n[기준 시점 내 중복 대여소번호]")
print(
    f"중복 행 수: "
    f"{len(snapshot_duplicate_rows):,}행"
)

if not snapshot_duplicate_rows.empty:
    display(snapshot_duplicate_rows)


# 좌표는 소수점 다섯째 자리에서 비교한다.
station_history_all[
    "coordinate_key"
] = pd.NA

valid_coordinate_mask = (
    station_history_all["valid_gps"]
)

station_history_all.loc[
    valid_coordinate_mask,
    "coordinate_key",
] = (
    station_history_all.loc[
        valid_coordinate_mask,
        "latitude",
    ]
    .round(5)
    .astype("string")
    + "|"
    + station_history_all.loc[
        valid_coordinate_mask,
        "longitude",
    ]
    .round(5)
    .astype("string")
)


history_station_summary = (
    station_history_all
    .groupby(
        "station_id",
        as_index=False,
    )
    .agg(
        snapshot_count=(
            "snapshot_period",
            "nunique",
        ),
        first_snapshot=(
            "snapshot_period",
            "min",
        ),
        last_snapshot=(
            "snapshot_period",
            "max",
        ),
        station_name_count=(
            "station_name",
            "nunique",
        ),
        valid_gps_snapshot_count=(
            "valid_gps",
            "sum",
        ),
        coordinate_count=(
            "coordinate_key",
            "nunique",
        ),
    )
)


history_station_summary[
    "has_valid_gps"
] = (
    history_station_summary[
        "valid_gps_snapshot_count"
    ] > 0
)


history_union_summary = pd.DataFrame(
    {
        "summary_item": [
            "네 파일 전체 행 수",
            "전체 고유 대여소번호",
            "4개 기준 시점 모두 존재",
            "유효 GPS가 한 번 이상 존재",
            "유효 GPS가 한 번도 없음",
            "이름이 시점별로 변경된 대여소",
            "좌표가 시점별로 변경된 대여소",
        ],
        "value": [
            len(station_history_all),
            history_station_summary[
                "station_id"
            ].nunique(),
            int(
                history_station_summary[
                    "snapshot_count"
                ].eq(4).sum()
            ),
            int(
                history_station_summary[
                    "has_valid_gps"
                ].sum()
            ),
            int(
                (
                    ~history_station_summary[
                        "has_valid_gps"
                    ]
                ).sum()
            ),
            int(
                history_station_summary[
                    "station_name_count"
                ].gt(1).sum()
            ),
            int(
                history_station_summary[
                    "coordinate_count"
                ].gt(1).sum()
            ),
        ],
    }
)


print("\n[네 시점 대여소 정보 통합 현황]")
display(history_union_summary)


# 이용정보 대여소와 역사 대여소 정보의 결합 가능성을 확인한다.
usage_history_reconciliation = (
    station_usage_summary[
        [
            "station_id_canonical",
            "raw_station_ids",
            "active_month_count",
            "first_active_month",
            "last_active_month",
            "hourly_row_count",
            "rental_count",
        ]
    ]
    .rename(
        columns={
            "station_id_canonical": (
                "station_id"
            )
        }
    )
    .merge(
        history_station_summary,
        on="station_id",
        how="left",
        validate="one_to_one",
    )
)


usage_history_reconciliation[
    "history_match"
] = (
    usage_history_reconciliation[
        "snapshot_count"
    ].notna()
)

usage_history_reconciliation[
    "gps_match"
] = (
    usage_history_reconciliation[
        "has_valid_gps"
    ]
    .fillna(False)
    .astype(bool)
)


usage_history_reconciliation[
    "match_status"
] = np.select(
    [
        usage_history_reconciliation[
            "gps_match"
        ],
        usage_history_reconciliation[
            "history_match"
        ],
    ],
    [
        "유효 GPS 결합 가능",
        "대여소 정보 존재·유효 GPS 없음",
    ],
    default="대여소 정보 미등록",
)


history_reconciliation_summary = (
    usage_history_reconciliation
    .groupby(
        "match_status",
        as_index=False,
    )
    .agg(
        unique_station_count=(
            "station_id",
            "nunique",
        ),
        hourly_row_count=(
            "hourly_row_count",
            "sum",
        ),
        rental_count=(
            "rental_count",
            "sum",
        ),
    )
)


history_reconciliation_summary[
    "station_share_pct"
] = (
    history_reconciliation_summary[
        "unique_station_count"
    ]
    .div(
        usage_history_reconciliation[
            "station_id"
        ].nunique()
    )
    .mul(100)
)


history_reconciliation_summary[
    "rental_count_share_pct"
] = (
    history_reconciliation_summary[
        "rental_count"
    ]
    .div(
        usage_history_reconciliation[
            "rental_count"
        ].sum()
    )
    .mul(100)
)


history_reconciliation_summary = (
    history_reconciliation_summary
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n[이용정보와 시점별 대여소 정보 결합 결과]")
display(history_reconciliation_summary)


unmatched_history_stations = (
    usage_history_reconciliation.loc[
        ~usage_history_reconciliation[
            "history_match"
        ]
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


no_valid_gps_stations = (
    usage_history_reconciliation.loc[
        usage_history_reconciliation[
            "history_match"
        ]
        & ~usage_history_reconciliation[
            "gps_match"
        ]
    ]
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


if not unmatched_history_stations.empty:
    print(
        "\n[시점별 대여소 정보 미등록 상위 20개]"
    )

    display(
        unmatched_history_stations[
            [
                "station_id",
                "raw_station_ids",
                "active_month_count",
                "first_active_month",
                "last_active_month",
                "rental_count",
            ]
        ].head(20)
    )


if not no_valid_gps_stations.empty:
    print(
        "\n[유효 GPS가 없는 대여소 상위 20개]"
    )

    display(
        no_valid_gps_stations[
            [
                "station_id",
                "raw_station_ids",
                "active_month_count",
                "first_active_month",
                "last_active_month",
                "snapshot_count",
                "rental_count",
            ]
        ].head(20)
    )


total_reconciled_rental_count = (
    history_reconciliation_summary[
        "rental_count"
    ].sum()
)


if total_reconciled_rental_count != 88_750_388:
    raise ValueError(
        "대여소 이력 결합 검증의 전체 이용건수가 "
        "88,750,388건과 일치하지 않는다."
    )

[01/04] 2023-06: 2,749행, 고유 ID 2,749개, 정상 GPS 2,749개
[02/04] 2023-12: 2,762행, 고유 ID 2,762개, 정상 GPS 2,762개
[03/04] 2024-06: 2,763행, 고유 ID 2,763개, 정상 GPS 2,763개
[04/04] 2024-12: 2,766행, 고유 ID 2,766개, 정상 GPS 2,766개

[파일별 대여소 정보 품질]


,snapshot_period,source_file,raw_excel_row_count,station_row_count,unique_station_count,duplicate_station_id_count,duplicate_station_row_count,station_name_missing,district_missing,address_missing,latitude_missing,longitude_missing,valid_gps_count,invalid_gps_count,latitude_min,latitude_max,longitude_min,longitude_max
0,2023-06,공공자전거 대여소 정보(23.06월 기준).xlsx,2753,2749,2749,0,0,0,0,0,0,0,2749,0,37.430977,37.691013,126.798599,127.180756
1,2023-12,공공자전거 대여소 정보(23.12월 기준).xlsx,2766,2762,2762,0,0,0,0,2,0,0,2762,0,37.430977,37.691013,126.798599,127.180756
2,2024-06,공공자전거 대여소 정보(24.6월 기준).xlsx,2767,2763,2763,0,0,0,0,2,0,0,2763,0,37.430977,37.691013,126.798599,127.180756
3,2024-12,공공자전거 대여소 정보(24.12월 기준).xlsx,2770,2766,2766,0,0,0,0,2,0,0,2766,0,37.430977,37.691013,126.798599,127.180756



[기준 시점 내 중복 대여소번호]
중복 행 수: 0행

[네 시점 대여소 정보 통합 현황]


,summary_item,value
0,네 파일 전체 행 수,11040
1,전체 고유 대여소번호,2865
2,4개 기준 시점 모두 존재,2651
3,유효 GPS가 한 번 이상 존재,2865
4,유효 GPS가 한 번도 없음,0
5,이름이 시점별로 변경된 대여소,45
6,좌표가 시점별로 변경된 대여소,47



[이용정보와 시점별 대여소 정보 결합 결과]


,match_status,unique_station_count,hourly_row_count,rental_count,station_share_pct,rental_count_share_pct
0,유효 GPS 결합 가능,2847,26127479,88708877,99.371728,99.953227
1,대여소 정보 미등록,18,17858,41511,0.628272,0.046773



[시점별 대여소 정보 미등록 상위 20개]


,station_id,raw_station_ids,active_month_count,first_active_month,last_active_month,rental_count
0,3578,03578,6,2023-01,2023-06,7585
1,1413,01413,6,2023-01,2023-06,6206
2,972,00972,5,2023-01,2023-05,5234
3,492,00492,4,2023-01,2023-04,4338
4,1249,01249,4,2023-01,2023-04,4139
5,4034,04034,5,2023-01,2023-05,4103
6,3107,03107,6,2023-01,2023-06,3065
7,4093,04093,5,2023-01,2023-05,1969
8,3011,03011,2,2023-01,2023-02,1130
9,3403,03403,5,2023-01,2023-05,858


## 19. 시점별 대여소 좌표 변경 규모 점검

네 개 기준 시점의 대여소 정보를 통합한 결과, 47개 대여소에서 시점별 좌표 차이가 확인되었다.

좌표 차이는 단순한 소수점 정밀도 수정일 수도 있고, 대여소의 실제 이전이나 재설치일 수도 있다. 모든 시점에 동일한 대표 좌표를 일괄 적용하면 실제 위치가 변경된 대여소의 과거 수요가 잘못된 공간 군집에 포함될 수 있다.

이에 동일 대여소번호 내 모든 유효 좌표 조합의 최대 하버사인 거리를 계산하고 다음 구간으로 분류한다.

- 10m 미만
- 10m 이상 50m 미만
- 50m 이상 200m 미만
- 200m 이상 1km 미만
- 1km 이상

좌표 변경 규모를 확인한 후 월별 수요에 해당 반기의 대여소 좌표를 연결할지, 최신 좌표 하나를 대표 좌표로 사용할지 결정한다.

In [20]:
def haversine_distance_m(
    latitude_1: float,
    longitude_1: float,
    latitude_2: float,
    longitude_2: float,
) -> float:
    """두 위경도 좌표 사이의 하버사인 거리를 미터 단위로 계산한다."""

    earth_radius_m = 6_371_000

    lat_1 = np.radians(latitude_1)
    lon_1 = np.radians(longitude_1)
    lat_2 = np.radians(latitude_2)
    lon_2 = np.radians(longitude_2)

    delta_lat = lat_2 - lat_1
    delta_lon = lon_2 - lon_1

    haversine_value = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat_1)
        * np.cos(lat_2)
        * np.sin(delta_lon / 2) ** 2
    )

    central_angle = 2 * np.arcsin(
        np.sqrt(haversine_value)
    )

    return float(
        earth_radius_m * central_angle
    )


valid_station_history = (
    station_history_all.loc[
        station_history_all["valid_gps"]
    ]
    .copy()
    .sort_values(
        [
            "station_id",
            "snapshot_date",
        ]
    )
)


coordinate_movement_records = []


for station_id, group in valid_station_history.groupby(
    "station_id",
    sort=True,
):
    group = (
        group
        .sort_values("snapshot_date")
        .reset_index(drop=True)
    )

    coordinate_rows = (
        group
        .drop_duplicates(
            subset=[
                "latitude",
                "longitude",
            ],
            keep="last",
        )
        .sort_values("snapshot_date")
        .reset_index(drop=True)
    )

    coordinate_count = len(coordinate_rows)

    max_distance_m = 0.0
    maximum_pair = None


    for first_index in range(coordinate_count):
        for second_index in range(
            first_index + 1,
            coordinate_count,
        ):
            first_row = coordinate_rows.iloc[
                first_index
            ]

            second_row = coordinate_rows.iloc[
                second_index
            ]

            distance_m = haversine_distance_m(
                first_row["latitude"],
                first_row["longitude"],
                second_row["latitude"],
                second_row["longitude"],
            )

            if distance_m > max_distance_m:
                max_distance_m = distance_m
                maximum_pair = (
                    first_row,
                    second_row,
                )


    latest_row = group.iloc[-1]


    movement_record = {
        "station_id": station_id,
        "latest_station_name": (
            latest_row["station_name"]
        ),
        "snapshot_count": int(
            group["snapshot_period"].nunique()
        ),
        "coordinate_count": coordinate_count,
        "max_distance_m": max_distance_m,
        "latest_snapshot": (
            latest_row["snapshot_period"]
        ),
        "latest_latitude": (
            latest_row["latitude"]
        ),
        "latest_longitude": (
            latest_row["longitude"]
        ),
    }


    if maximum_pair is None:
        movement_record.update(
            {
                "first_pair_snapshot": pd.NA,
                "first_pair_latitude": pd.NA,
                "first_pair_longitude": pd.NA,
                "second_pair_snapshot": pd.NA,
                "second_pair_latitude": pd.NA,
                "second_pair_longitude": pd.NA,
            }
        )

    else:
        first_row, second_row = maximum_pair

        movement_record.update(
            {
                "first_pair_snapshot": (
                    first_row["snapshot_period"]
                ),
                "first_pair_latitude": (
                    first_row["latitude"]
                ),
                "first_pair_longitude": (
                    first_row["longitude"]
                ),
                "second_pair_snapshot": (
                    second_row["snapshot_period"]
                ),
                "second_pair_latitude": (
                    second_row["latitude"]
                ),
                "second_pair_longitude": (
                    second_row["longitude"]
                ),
            }
        )


    coordinate_movement_records.append(
        movement_record
    )


station_coordinate_movement = pd.DataFrame(
    coordinate_movement_records
)


station_coordinate_movement = (
    station_coordinate_movement
    .merge(
        station_usage_summary[
            [
                "station_id_canonical",
                "active_month_count",
                "rental_count",
            ]
        ],
        left_on="station_id",
        right_on="station_id_canonical",
        how="left",
        validate="one_to_one",
    )
    .drop(
        columns=["station_id_canonical"]
    )
)


changed_station_coordinates = (
    station_coordinate_movement.loc[
        station_coordinate_movement[
            "coordinate_count"
        ].gt(1)
    ]
    .copy()
)


changed_station_coordinates[
    "movement_category"
] = pd.cut(
    changed_station_coordinates[
        "max_distance_m"
    ],
    bins=[
        -0.001,
        10,
        50,
        200,
        1_000,
        np.inf,
    ],
    labels=[
        "10m 미만",
        "10m 이상 50m 미만",
        "50m 이상 200m 미만",
        "200m 이상 1km 미만",
        "1km 이상",
    ],
    right=False,
)


coordinate_change_summary = (
    changed_station_coordinates
    .groupby(
        "movement_category",
        observed=True,
        as_index=False,
    )
    .agg(
        station_count=(
            "station_id",
            "nunique",
        ),
        rental_count=(
            "rental_count",
            "sum",
        ),
        maximum_distance_m=(
            "max_distance_m",
            "max",
        ),
    )
)


coordinate_change_summary[
    "station_share_pct"
] = (
    coordinate_change_summary[
        "station_count"
    ]
    .div(
        station_coordinate_movement[
            "station_id"
        ].nunique()
    )
    .mul(100)
)


coordinate_change_summary[
    "rental_count_share_pct"
] = (
    coordinate_change_summary[
        "rental_count"
    ]
    .div(88_750_388)
    .mul(100)
)


coordinate_change_summary = (
    coordinate_change_summary
    .sort_values(
        "maximum_distance_m"
    )
    .reset_index(drop=True)
)


changed_station_coordinates = (
    changed_station_coordinates
    .sort_values(
        [
            "max_distance_m",
            "rental_count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("[좌표 변경 대여소 현황]")
print(
    f"전체 GPS 보유 대여소: "
    f"{len(station_coordinate_movement):,}개"
)
print(
    f"좌표가 한 종류인 대여소: "
    f"{(
        station_coordinate_movement[
            'coordinate_count'
        ].eq(1)
    ).sum():,}개"
)
print(
    f"좌표가 변경된 대여소: "
    f"{len(changed_station_coordinates):,}개"
)
print(
    "좌표 변경 대여소 수요: "
    f"{changed_station_coordinates['rental_count'].sum():,.0f}건"
)
print(
    "좌표 변경 대여소 수요 비율: "
    f"{(
        changed_station_coordinates[
            'rental_count'
        ].sum()
        / 88_750_388
        * 100
    ):,.4f}%"
)


print("\n[좌표 변경 거리 구간]")
display(coordinate_change_summary)


print("\n[좌표 변경 거리 상위 20개]")
display(
    changed_station_coordinates[
        [
            "station_id",
            "latest_station_name",
            "snapshot_count",
            "coordinate_count",
            "max_distance_m",
            "movement_category",
            "first_pair_snapshot",
            "first_pair_latitude",
            "first_pair_longitude",
            "second_pair_snapshot",
            "second_pair_latitude",
            "second_pair_longitude",
            "active_month_count",
            "rental_count",
        ]
    ].head(20)
)


if len(changed_station_coordinates) != 47:
    print(
        "\n주의: 앞 단계에서 확인한 좌표 변경 대여소 "
        "47개와 현재 계산 결과가 일치하지 않는다."
    )

[좌표 변경 대여소 현황]
전체 GPS 보유 대여소: 2,865개
좌표가 한 종류인 대여소: 427개
좌표가 변경된 대여소: 2,438개
좌표 변경 대여소 수요: 78,109,521건
좌표 변경 대여소 수요 비율: 88.0103%

[좌표 변경 거리 구간]


,movement_category,station_count,rental_count,maximum_distance_m,station_share_pct,rental_count_share_pct
0,10m 미만,2400,76933515.0,8.907826,83.769634,86.685272
1,10m 이상 50m 미만,24,771510.0,49.146874,0.837696,0.869303
2,50m 이상 200m 미만,12,358679.0,184.798608,0.418848,0.404144
3,200m 이상 1km 미만,2,45817.0,401.781124,0.069808,0.051625



[좌표 변경 거리 상위 20개]


,station_id,latest_station_name,snapshot_count,coordinate_count,max_distance_m,movement_category,first_pair_snapshot,first_pair_latitude,first_pair_longitude,second_pair_snapshot,second_pair_latitude,second_pair_longitude,active_month_count,rental_count
0,3988,엘아르베뉴 102동 앞,4,2,401.781124,200m 이상 1km 미만,2023-06,37.470947,126.887711,2024-12,37.474525,126.887077,20.0,19216.0
1,1277,성내7교,4,2,216.118916,200m 이상 1km 미만,2023-06,37.503757,127.137093,2024-12,37.505627,127.136421,24.0,26601.0
2,3652,한영외국어고등학교,4,2,184.798608,50m 이상 200m 미만,2023-06,37.548748,127.156403,2024-12,37.547092,127.156219,24.0,7917.0
3,4008,하계 어린이 공원,4,2,152.865193,50m 이상 200m 미만,2023-06,37.632145,127.069824,2024-12,37.633415,127.06916,24.0,35496.0
4,4107,장안동근린공원 입구,4,2,118.895102,50m 이상 200m 미만,2023-06,37.577396,127.071854,2024-12,37.576332,127.071724,22.0,17288.0
5,594,중랑물재생센터(서울새활용플라자),4,2,101.772084,50m 이상 200m 미만,2023-06,37.558365,127.056908,2024-12,37.558304,127.05806,24.0,19746.0
6,2921,중계역 2번출구,4,2,100.723851,50m 이상 200m 미만,2023-06,37.645489,127.064583,2024-12,37.64587,127.06562,24.0,60893.0
7,2314,청담나들목입구,4,2,83.213196,50m 이상 200m 미만,2023-06,37.521275,127.061035,2024-12,37.521015,127.06015,24.0,11784.0
8,222,시범아파트버스정류장 옆,4,2,77.805443,50m 이상 200m 미만,2023-06,37.519436,126.937599,2024-12,37.51902,126.93689,24.0,55448.0
9,2520,서초1교 (진흥아파트 방면),4,2,73.223700,50m 이상 200m 미만,2023-06,37.496193,127.021011,2024-12,37.495934,127.021774,24.0,11296.0



주의: 앞 단계에서 확인한 좌표 변경 대여소 47개와 현재 계산 결과가 일치하지 않는다.


## 20. 최신 유효 좌표 기반 분석용 대여소 마스터 재구축

원본 위·경도 값을 완전히 동일한지 비교하면 2,438개 대여소에서 좌표 차이가 확인된다. 그러나 이 중 2,400개는 최대 좌표 차이가 10m 미만이며, 대부분 데이터 제공 시점에 따른 좌표 정밀도 수정으로 판단한다.

10m 이상의 좌표 차이가 존재하는 대여소는 38개이며, 전체 수요에서 차지하는 비율은 약 1.33%이다. 최대 이동 거리도 약 402m로 서울 전역을 세 개 권역으로 구분하는 공간 군집화에 미치는 영향은 제한적이다.

본 분석에서는 대여소별로 가장 최근 기준 시점의 유효 GPS 좌표를 대표 좌표로 사용한다. 이를 통해 모든 분석 기간에 동일한 대여소-좌표 매핑을 적용하고 공간 군집의 해석 일관성을 확보한다.

기존 `ST-숫자` 내부 관리 ID 기반 마스터는 잘못된 결합 키를 사용한 결과이므로 별도 레거시 파일로 보존한다. 새로운 분석용 마스터는 실제 대여소번호를 기준으로 재구축한다.

시점별 대여소 정보에 존재하지 않는 18개 대여소는 임의로 GPS를 추정하지 않는다. 해당 대여소는 전체 수요의 약 0.047%이며, 별도 제외 파일에 기록한 후 공간 군집화에서 제외한다.

In [21]:
import shutil


MEANINGFUL_MOVEMENT_THRESHOLD_M = 10.0


# 1. 기존 내부 관리 ID 기반 마스터를 레거시 파일로 보존한다.

clean_station_path = (
    processed_dir / "station_master_clean.csv"
)

invalid_station_path = (
    processed_dir / "station_master_invalid_gps.csv"
)

legacy_clean_station_path = (
    processed_dir
    / "station_master_internal_id_legacy.csv"
)

legacy_invalid_station_path = (
    processed_dir
    / "station_master_internal_id_invalid_gps_legacy.csv"
)


if (
    clean_station_path.exists()
    and not legacy_clean_station_path.exists()
):
    shutil.copy2(
        clean_station_path,
        legacy_clean_station_path,
    )

    print(
        "기존 마스터 백업: "
        f"{legacy_clean_station_path.relative_to(ROOT)}"
    )


if (
    invalid_station_path.exists()
    and not legacy_invalid_station_path.exists()
):
    shutil.copy2(
        invalid_station_path,
        legacy_invalid_station_path,
    )

    print(
        "기존 제외 마스터 백업: "
        f"{legacy_invalid_station_path.relative_to(ROOT)}"
    )


# 2. 좌표 변경 기준을 재정의한다.

exact_coordinate_change_count = int(
    station_coordinate_movement[
        "coordinate_count"
    ]
    .gt(1)
    .sum()
)


meaningful_movement_stations = (
    station_coordinate_movement.loc[
        station_coordinate_movement[
            "max_distance_m"
        ]
        .ge(
            MEANINGFUL_MOVEMENT_THRESHOLD_M
        )
        .fillna(False)
    ]
    .copy()
    .sort_values(
        "max_distance_m",
        ascending=False,
    )
    .reset_index(drop=True)
)


meaningful_movement_rental_count = int(
    meaningful_movement_stations[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)


maximum_movement_distance = (
    meaningful_movement_stations[
        "max_distance_m"
    ].max()
)


print("\n[좌표 변경 기준 재정의]")
print(
    f"원본 좌표값이 다른 대여소: "
    f"{exact_coordinate_change_count:,}개"
)
print(
    f"10m 이상 이동 대여소: "
    f"{len(meaningful_movement_stations):,}개"
)
print(
    "10m 이상 이동 대여소 수요: "
    f"{meaningful_movement_rental_count:,}건"
)
print(
    "10m 이상 이동 대여소 수요 비율: "
    f"{meaningful_movement_rental_count / 88_750_388 * 100:.4f}%"
)
print(
    "최대 좌표 이동 거리: "
    f"{maximum_movement_distance:,.2f}m"
)


# 3. 각 대여소의 가장 최근 유효 GPS를 선택한다.

latest_valid_station_history = (
    station_history_all.loc[
        station_history_all[
            "valid_gps"
        ].fillna(False)
    ]
    .sort_values(
        [
            "station_id",
            "snapshot_date",
        ]
    )
    .drop_duplicates(
        subset=["station_id"],
        keep="last",
    )
    .reset_index(drop=True)
)


if (
    latest_valid_station_history[
        "station_id"
    ].duplicated().any()
):
    raise ValueError(
        "최신 유효 대여소 정보에 중복 ID가 존재한다."
    )


print("\n[최신 유효 GPS 정보]")
print(
    "최신 유효 GPS 대여소 수: "
    f"{len(latest_valid_station_history):,}개"
)


# 4. 이용정보 기준 대여소 목록을 준비한다.

usage_station_base = (
    station_usage_summary[
        [
            "station_id_canonical",
            "raw_station_ids",
            "active_month_count",
            "first_active_month",
            "last_active_month",
            "hourly_row_count",
            "rental_count",
        ]
    ]
    .rename(
        columns={
            "station_id_canonical": "station_id",
        }
    )
    .copy()
)


if (
    usage_station_base[
        "station_id"
    ].duplicated().any()
):
    raise ValueError(
        "이용정보 기준 대여소 목록에 중복 ID가 존재한다."
    )


# 5. 이용정보와 최신 대여소 정보를 결합한다.

station_master_rebuild = (
    usage_station_base
    .merge(
        latest_valid_station_history[
            [
                "station_id",
                "station_name",
                "district",
                "address",
                "latitude",
                "longitude",
                "installation_date",
                "lcd_rack_count",
                "qr_rack_count",
                "operation_method",
                "snapshot_period",
                "snapshot_date",
                "source_file",
            ]
        ],
        on="station_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        history_station_summary[
            [
                "station_id",
                "snapshot_count",
                "first_snapshot",
                "last_snapshot",
                "station_name_count",
                "valid_gps_snapshot_count",
                "coordinate_count",
            ]
        ],
        on="station_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        station_coordinate_movement[
            [
                "station_id",
                "max_distance_m",
            ]
        ],
        on="station_id",
        how="left",
        validate="one_to_one",
    )
)


station_master_rebuild[
    "meaningful_coordinate_movement"
] = (
    station_master_rebuild[
        "max_distance_m"
    ]
    .ge(
        MEANINGFUL_MOVEMENT_THRESHOLD_M
    )
    .fillna(False)
    .astype(bool)
)


# 위도 또는 경도가 결측이면 반드시 False로 처리한다.
latitude_valid = (
    station_master_rebuild[
        "latitude"
    ]
    .between(
        37.4,
        37.8,
    )
    .fillna(False)
)


longitude_valid = (
    station_master_rebuild[
        "longitude"
    ]
    .between(
        126.7,
        127.3,
    )
    .fillna(False)
)


station_master_rebuild[
    "gps_available"
] = (
    latitude_valid
    & longitude_valid
).astype(bool)


# 6. 분석용 마스터와 제외 대여소를 분리한다.

station_master_clean = (
    station_master_rebuild.loc[
        station_master_rebuild[
            "gps_available"
        ].eq(True)
    ]
    .copy()
    .sort_values("station_id")
    .reset_index(drop=True)
)


station_master_excluded = (
    station_master_rebuild.loc[
        station_master_rebuild[
            "gps_available"
        ].eq(False)
    ]
    .copy()
    .sort_values(
        "rental_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n[GPS 분류 검증]")
print(
    station_master_rebuild[
        "gps_available"
    ].value_counts(
        dropna=False
    )
)

print(
    "분석용 대여소 수: "
    f"{len(station_master_clean):,}개"
)

print(
    "제외 대여소 수: "
    f"{len(station_master_excluded):,}개"
)

print(
    "분류 합계: "
    f"{len(station_master_clean) + len(station_master_excluded):,}개"
)


if (
    station_master_rebuild[
        "gps_available"
    ].isna().any()
):
    raise ValueError(
        "gps_available 변수에 결측값이 존재한다."
    )


if (
    len(station_master_clean)
    + len(station_master_excluded)
    != len(station_master_rebuild)
):
    raise ValueError(
        "GPS 보유 및 미보유 대여소 분류 결과가 "
        "전체 대여소 수와 일치하지 않는다."
    )


# 7. 저장에 사용할 컬럼을 정리한다.

station_master_clean = (
    station_master_clean[
        [
            "station_id",
            "station_name",
            "district",
            "address",
            "latitude",
            "longitude",
            "installation_date",
            "lcd_rack_count",
            "qr_rack_count",
            "operation_method",
            "snapshot_period",
            "snapshot_date",
            "source_file",
            "snapshot_count",
            "first_snapshot",
            "last_snapshot",
            "station_name_count",
            "valid_gps_snapshot_count",
            "coordinate_count",
            "max_distance_m",
            "meaningful_coordinate_movement",
            "raw_station_ids",
            "active_month_count",
            "first_active_month",
            "last_active_month",
            "hourly_row_count",
            "rental_count",
        ]
    ]
)


station_master_excluded[
    "exclusion_reason"
] = (
    "시점별 공식 대여소 정보 미등록"
)


station_master_excluded = (
    station_master_excluded[
        [
            "station_id",
            "raw_station_ids",
            "active_month_count",
            "first_active_month",
            "last_active_month",
            "hourly_row_count",
            "rental_count",
            "exclusion_reason",
        ]
    ]
)


# 8. 분석용 마스터의 기본 품질을 검증한다.

if (
    station_master_clean[
        "station_id"
    ].duplicated().any()
):
    raise ValueError(
        "분석용 대여소 마스터에 중복 ID가 존재한다."
    )


if (
    station_master_excluded[
        "station_id"
    ].duplicated().any()
):
    raise ValueError(
        "제외 대여소 목록에 중복 ID가 존재한다."
    )


if (
    station_master_clean[
        [
            "station_id",
            "latitude",
            "longitude",
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "분석용 대여소 마스터의 핵심 변수에 "
        "결측치가 존재한다."
    )


if not station_master_clean[
    "latitude"
].between(
    37.4,
    37.8,
).all():
    raise ValueError(
        "분석용 마스터에 서울 범위 밖 위도가 존재한다."
    )


if not station_master_clean[
    "longitude"
].between(
    126.7,
    127.3,
).all():
    raise ValueError(
        "분석용 마스터에 서울 범위 밖 경도가 존재한다."
    )


matched_station_count = len(
    station_master_clean
)

excluded_station_count = len(
    station_master_excluded
)


matched_rental_count = int(
    station_master_clean[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)

excluded_rental_count = int(
    station_master_excluded[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)


total_station_count = (
    matched_station_count
    + excluded_station_count
)

total_rental_count = (
    matched_rental_count
    + excluded_rental_count
)


if total_station_count != 2_865:
    raise ValueError(
        "분석용 및 제외 대여소 수 합계가 "
        f"2,865개와 일치하지 않는다: "
        f"{total_station_count:,}개"
    )


if total_rental_count != 88_750_388:
    raise ValueError(
        "분석용 및 제외 대여소의 이용건수 합계가 "
        f"88,750,388건과 일치하지 않는다: "
        f"{total_rental_count:,}건"
    )


if matched_station_count != 2_847:
    raise ValueError(
        "유효 GPS 대여소 수가 예상값 "
        f"2,847개와 일치하지 않는다: "
        f"{matched_station_count:,}개"
    )


if excluded_station_count != 18:
    raise ValueError(
        "공간 분석 제외 대여소 수가 예상값 "
        f"18개와 일치하지 않는다: "
        f"{excluded_station_count:,}개"
    )


# 9. 결과 파일을 저장한다.

coordinate_audit_path = (
    processed_dir
    / "station_coordinate_movement_audit.csv"
)

station_master_quality_path = (
    processed_dir
    / "station_master_quality_summary.csv"
)


station_master_clean.to_csv(
    clean_station_path,
    index=False,
    encoding="utf-8-sig",
)


station_master_excluded.to_csv(
    invalid_station_path,
    index=False,
    encoding="utf-8-sig",
)


station_coordinate_movement.to_csv(
    coordinate_audit_path,
    index=False,
    encoding="utf-8-sig",
)


station_master_quality_summary = pd.DataFrame(
    {
        "quality_item": [
            "전체 이용정보 대여소",
            "유효 GPS 결합 대여소",
            "공간 분석 제외 대여소",
            "유효 GPS 결합 이용건수",
            "공간 분석 제외 이용건수",
            "유효 GPS 수요 커버리지",
            "10m 이상 좌표 이동 대여소",
            "10m 이상 좌표 이동 대여소 수요",
            "10m 이상 좌표 이동 대여소 수요 비율",
            "최대 좌표 이동 거리",
            "대표 좌표 선정 기준",
        ],
        "value": [
            total_station_count,
            matched_station_count,
            excluded_station_count,
            matched_rental_count,
            excluded_rental_count,
            (
                matched_rental_count
                / total_rental_count
                * 100
            ),
            len(
                meaningful_movement_stations
            ),
            meaningful_movement_rental_count,
            (
                meaningful_movement_rental_count
                / total_rental_count
                * 100
            ),
            maximum_movement_distance,
            "대여소별 최신 기준 시점의 유효 GPS",
        ],
    }
)


station_master_quality_summary.to_csv(
    station_master_quality_path,
    index=False,
    encoding="utf-8-sig",
)


# 10. 최종 결과를 출력한다.

print("\n[분석용 대여소 마스터 재구축 결과]")
print(
    f"전체 이용 대여소: "
    f"{total_station_count:,}개"
)
print(
    f"유효 GPS 대여소: "
    f"{matched_station_count:,}개"
)
print(
    f"공간 분석 제외 대여소: "
    f"{excluded_station_count:,}개"
)
print(
    f"유효 GPS 이용건수: "
    f"{matched_rental_count:,}건"
)
print(
    f"제외 이용건수: "
    f"{excluded_rental_count:,}건"
)
print(
    "수요 커버리지: "
    f"{matched_rental_count / total_rental_count * 100:.6f}%"
)
print(
    "10m 이상 이동 대여소: "
    f"{station_master_clean['meaningful_coordinate_movement'].sum():,}개"
)


print("\n[저장 파일]")
print(
    clean_station_path.relative_to(ROOT)
)
print(
    invalid_station_path.relative_to(ROOT)
)
print(
    coordinate_audit_path.relative_to(ROOT)
)
print(
    station_master_quality_path.relative_to(ROOT)
)


print("\n[분석용 대여소 마스터 표본]")
display(
    station_master_clean.head(10)
)


print("\n[공간 분석 제외 대여소]")
display(
    station_master_excluded
)


[좌표 변경 기준 재정의]
원본 좌표값이 다른 대여소: 2,438개
10m 이상 이동 대여소: 38개
10m 이상 이동 대여소 수요: 1,176,006건
10m 이상 이동 대여소 수요 비율: 1.3251%
최대 좌표 이동 거리: 401.78m

[최신 유효 GPS 정보]
최신 유효 GPS 대여소 수: 2,865개

[GPS 분류 검증]
gps_available
True     2847
False      18
Name: count, dtype: int64
분석용 대여소 수: 2,847개
제외 대여소 수: 18개
분류 합계: 2,865개

[분석용 대여소 마스터 재구축 결과]
전체 이용 대여소: 2,865개
유효 GPS 대여소: 2,847개
공간 분석 제외 대여소: 18개
유효 GPS 이용건수: 88,708,877건
제외 이용건수: 41,511건
수요 커버리지: 99.953227%
10m 이상 이동 대여소: 38개

[저장 파일]
data\processed\station_master_clean.csv
data\processed\station_master_invalid_gps.csv
data\processed\station_coordinate_movement_audit.csv
data\processed\station_master_quality_summary.csv

[분석용 대여소 마스터 표본]


,station_id,station_name,district,address,latitude,longitude,installation_date,lcd_rack_count,qr_rack_count,operation_method,...,valid_gps_snapshot_count,coordinate_count,max_distance_m,meaningful_coordinate_movement,raw_station_ids,active_month_count,first_active_month,last_active_month,hourly_row_count,rental_count
0,1001,광진교 남단 사거리(천호공원 방면),강동구,서울특별시 강동구 구천면로 171,37.541794,127.124748,2017-04-19 10:11:51,15,<NA>,LCD,...,4,1.0,0.002645,False,01001,24,2023-01,2024-12,12630,40736
1,1002,해공공원(천호동),강동구,서울특별시 강동구 올림픽로 702,37.545265,127.125938,2017-04-19 14:00:02,10,<NA>,LCD,...,4,1.0,0.001763,False,01002,24,2023-01,2024-12,14109,58417
2,1003,해공도서관앞,강동구,서울특별시 강동구 올림픽로 702,37.543957,127.125488,2017-04-19 14:16:45,20,<NA>,LCD,...,4,1.0,0.001763,False,01003,24,2023-01,2024-12,11062,28356
3,1004,삼성광나루아파트 버스정류장,강동구,서울특별시 강동구 상암로3길 77,37.553329,127.128868,2017-04-19 14:39:27,10,<NA>,LCD,...,4,1.0,0.000000,False,01004,24,2023-01,2024-12,12229,40642
4,1006,롯데캐슬 115동앞,강동구,서울특별시 강동구 양재대로 1665,37.554867,127.142799,2017-04-19 14:43:13,10,<NA>,LCD,...,4,1.0,0.001763,False,01006,24,2023-01,2024-12,4501,6012
5,1007,암사동 선사유적지,강동구,서울특별시 강동구 올림픽로 875,37.559559,127.130875,2017-04-19 14:43:55,14,<NA>,LCD,...,4,1.0,0.002644,False,01007,24,2023-01,2024-12,2876,4249
6,1008,암사역 3번출구(국민은행앞),강동구,서울특별시 강동구 올림픽로 지하 776,37.549355,127.127083,2017-04-19 14:44:50,9,<NA>,LCD,...,4,1.0,0.001763,False,01008,24,2023-01,2024-12,14817,85179
7,1009,천호역4번출구(현대백화점),강동구,서울특별시 강동구 천호대로 지하 997,37.538666,127.124245,2017-04-19 14:46:04,20,<NA>,LCD,...,4,1.0,0.000882,False,01009,24,2023-01,2024-12,15187,130694
8,1010,강동세무서,강동구,서울특별시 강동구 천호대로 1139,37.534508,127.138405,2017-04-19 14:47:29,20,<NA>,LCD,...,4,1.0,0.004409,False,01010,24,2023-01,2024-12,15295,79214
9,1011,LIGA 아파트 앞,강동구,서울특별시 강동구 천호대로 1120,37.534744,127.135284,2017-04-19 14:48:48,20,<NA>,LCD,...,4,1.0,0.001764,False,01011,24,2023-01,2024-12,15308,89594



[공간 분석 제외 대여소]


,station_id,raw_station_ids,active_month_count,first_active_month,last_active_month,hourly_row_count,rental_count,exclusion_reason
0,3578,03578,6,2023-01,2023-06,2546,7585,시점별 공식 대여소 정보 미등록
1,1413,01413,6,2023-01,2023-06,2365,6206,시점별 공식 대여소 정보 미등록
2,972,00972,5,2023-01,2023-05,2093,5234,시점별 공식 대여소 정보 미등록
3,492,00492,4,2023-01,2023-04,1560,4338,시점별 공식 대여소 정보 미등록
4,1249,01249,4,2023-01,2023-04,1471,4139,시점별 공식 대여소 정보 미등록
5,4034,04034,5,2023-01,2023-05,1771,4103,시점별 공식 대여소 정보 미등록
6,3107,03107,6,2023-01,2023-06,1723,3065,시점별 공식 대여소 정보 미등록
7,4093,04093,5,2023-01,2023-05,1241,1969,시점별 공식 대여소 정보 미등록
8,3011,03011,2,2023-01,2023-02,643,1130,시점별 공식 대여소 정보 미등록
9,3403,03403,5,2023-01,2023-05,556,858,시점별 공식 대여소 정보 미등록


## 21. 공간 분석용 월별 시간대 수요 데이터 구축

시점별 공식 대여소 정보와 연결 가능한 2,847개 대여소만을 대상으로 공간 분석용 시간대별 수요 데이터를 구축한다.

공식 대여소 정보에 존재하지 않는 18개 대여소는 전체 수요의 약 0.047%를 차지한다. 해당 대여소에 임의의 GPS 좌표를 부여할 경우 잘못된 공간 군집에 포함될 위험이 있으므로 공간 분석에서는 제외한다.

기존 월별 시간대 수요 파일의 대여소번호는 앞자리에 0이 포함된 문자열 형식이다. 이를 분석용 마스터와 동일하게 앞자리 0이 없는 표준 대여소번호로 변환한다.

각 월별 파일을 다음 두 집단으로 분리한다.

- 유효 GPS가 연결된 공간 분석 대상 대여소
- 공식 대여소 정보에 존재하지 않는 공간 분석 제외 대여소

공간 분석 제외 행은 별도의 감사 파일에 저장한다. 전체 이용건수, 행 수 및 대여소 수가 분리 전후에 보존되는지 검증한다.

In [22]:
# 1. 입력 및 출력 경로를 설정한다.

spatial_monthly_dir = (
    processed_dir
    / "station_hourly_spatial_monthly"
)

spatial_monthly_dir.mkdir(
    parents=True,
    exist_ok=True,
)


spatial_quality_path = (
    processed_dir
    / "spatial_demand_quality_by_month.csv"
)

spatial_excluded_audit_path = (
    processed_dir
    / "station_hourly_excluded_from_spatial.csv"
)


monthly_station_files = sorted(
    monthly_output_dir.glob(
        "station_hourly_*.csv"
    )
)


if len(monthly_station_files) != 24:
    raise ValueError(
        "입력 월별 시간대 수요 파일이 "
        f"24개가 아니다: {len(monthly_station_files)}개"
    )


# 이전 실행 결과가 남아 있을 경우 월별 출력 파일만 정리한다.
for old_output_path in (
    spatial_monthly_dir.glob(
        "station_hourly_*.csv"
    )
):
    old_output_path.unlink()


# 2. 분석 대상 및 제외 대여소 ID 집합을 구성한다.

valid_station_id_set = set(
    station_master_clean[
        "station_id"
    ]
    .astype("string")
    .dropna()
)


excluded_station_id_set = set(
    station_master_excluded[
        "station_id"
    ]
    .astype("string")
    .dropna()
)


usage_station_id_set = set(
    station_usage_summary[
        "station_id_canonical"
    ]
    .astype("string")
    .dropna()
)


if (
    valid_station_id_set
    & excluded_station_id_set
):
    raise ValueError(
        "분석 대상 대여소와 제외 대여소 ID가 중복된다."
    )


classified_station_id_set = (
    valid_station_id_set
    | excluded_station_id_set
)


unclassified_station_ids = (
    usage_station_id_set
    - classified_station_id_set
)

unused_master_station_ids = (
    classified_station_id_set
    - usage_station_id_set
)


print("[대여소 ID 분류 검증]")
print(
    f"이용정보 전체 대여소: "
    f"{len(usage_station_id_set):,}개"
)
print(
    f"공간 분석 대상 대여소: "
    f"{len(valid_station_id_set):,}개"
)
print(
    f"공간 분석 제외 대여소: "
    f"{len(excluded_station_id_set):,}개"
)
print(
    f"미분류 대여소: "
    f"{len(unclassified_station_ids):,}개"
)
print(
    f"이용정보에 없는 분류 ID: "
    f"{len(unused_master_station_ids):,}개"
)


if unclassified_station_ids:
    raise ValueError(
        "공간 분석 대상 또는 제외 대상으로 "
        "분류되지 않은 이용정보 대여소가 존재한다: "
        f"{sorted(unclassified_station_ids)[:20]}"
    )


if unused_master_station_ids:
    raise ValueError(
        "분석 대상·제외 대여소 목록에 "
        "이용정보에 존재하지 않는 ID가 포함되어 있다: "
        f"{sorted(unused_master_station_ids)[:20]}"
    )


# 3. 월별 파일을 공간 분석 대상과 제외 대상으로 분리한다.

spatial_quality_records = []
excluded_monthly_frames = []


for file_index, input_path in enumerate(
    monthly_station_files,
    start=1,
):
    year_month_text = (
        input_path.stem
        .replace(
            "station_hourly_",
            "",
        )
    )


    monthly_data = pd.read_csv(
        input_path,
        encoding="utf-8-sig",
        dtype={
            "station_id": "string",
            "rental_count": "int64",
        },
        parse_dates=["datetime"],
    )


    required_columns = {
        "datetime",
        "station_id",
        "rental_count",
    }


    missing_columns = sorted(
        required_columns
        - set(monthly_data.columns)
    )


    if missing_columns:
        raise KeyError(
            f"{input_path.name}에 필수 컬럼이 없다: "
            f"{missing_columns}"
        )


    monthly_data = monthly_data[
        [
            "datetime",
            "station_id",
            "rental_count",
        ]
    ].copy()


    monthly_data[
        "station_id"
    ] = canonicalize_station_id(
        monthly_data[
            "station_id"
        ]
    )


    if monthly_data[
        "station_id"
    ].isna().any():
        raise ValueError(
            f"{input_path.name}의 대여소번호 표준화 후 "
            "결측치가 발생했다."
        )


    valid_row_mask = (
        monthly_data[
            "station_id"
        ].isin(valid_station_id_set)
    )


    excluded_row_mask = (
        monthly_data[
            "station_id"
        ].isin(excluded_station_id_set)
    )


    unclassified_row_mask = ~(
        valid_row_mask
        | excluded_row_mask
    )


    if unclassified_row_mask.any():
        unknown_ids = sorted(
            monthly_data.loc[
                unclassified_row_mask,
                "station_id",
            ]
            .dropna()
            .unique()
            .tolist()
        )

        raise ValueError(
            f"{input_path.name}에 미분류 대여소가 존재한다: "
            f"{unknown_ids[:20]}"
        )


    spatial_monthly = (
        monthly_data.loc[
            valid_row_mask,
            [
                "datetime",
                "station_id",
                "rental_count",
            ],
        ]
        .copy()
        .sort_values(
            [
                "datetime",
                "station_id",
            ]
        )
        .reset_index(drop=True)
    )


    excluded_monthly = (
        monthly_data.loc[
            excluded_row_mask,
            [
                "datetime",
                "station_id",
                "rental_count",
            ],
        ]
        .copy()
        .sort_values(
            [
                "datetime",
                "station_id",
            ]
        )
        .reset_index(drop=True)
    )


    excluded_monthly[
        "year_month"
    ] = year_month_text

    excluded_monthly[
        "source_file"
    ] = input_path.name


    if not excluded_monthly.empty:
        excluded_monthly_frames.append(
            excluded_monthly
        )


    input_row_count = len(
        monthly_data
    )

    spatial_row_count = len(
        spatial_monthly
    )

    excluded_row_count = len(
        excluded_monthly
    )


    input_rental_count = int(
        monthly_data[
            "rental_count"
        ].sum()
    )

    spatial_rental_count = int(
        spatial_monthly[
            "rental_count"
        ].sum()
    )

    excluded_rental_count = int(
        excluded_monthly[
            "rental_count"
        ].sum()
    )


    if (
        spatial_row_count
        + excluded_row_count
        != input_row_count
    ):
        raise ValueError(
            f"{input_path.name}의 행 수가 "
            "분리 전후에 보존되지 않았다."
        )


    if (
        spatial_rental_count
        + excluded_rental_count
        != input_rental_count
    ):
        raise ValueError(
            f"{input_path.name}의 이용건수가 "
            "분리 전후에 보존되지 않았다."
        )


    output_path = (
        spatial_monthly_dir
        / input_path.name
    )


    spatial_monthly.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )


    spatial_quality_records.append(
        {
            "year_month": year_month_text,
            "input_file": input_path.name,
            "output_file": output_path.name,
            "input_row_count": (
                input_row_count
            ),
            "spatial_row_count": (
                spatial_row_count
            ),
            "excluded_row_count": (
                excluded_row_count
            ),
            "input_station_count": int(
                monthly_data[
                    "station_id"
                ].nunique()
            ),
            "spatial_station_count": int(
                spatial_monthly[
                    "station_id"
                ].nunique()
            ),
            "excluded_station_count": int(
                excluded_monthly[
                    "station_id"
                ].nunique()
            ),
            "input_rental_count": (
                input_rental_count
            ),
            "spatial_rental_count": (
                spatial_rental_count
            ),
            "excluded_rental_count": (
                excluded_rental_count
            ),
            "spatial_rental_share_pct": (
                spatial_rental_count
                / input_rental_count
                * 100
            ),
        }
    )


    print(
        f"[{file_index:02d}/24] "
        f"{year_month_text}: "
        f"{input_row_count:,}행 → "
        f"분석 {spatial_row_count:,}행 / "
        f"제외 {excluded_row_count:,}행, "
        f"수요 보존 "
        f"{input_rental_count:,}건"
    )


    del monthly_data
    del spatial_monthly
    del excluded_monthly
    gc.collect()


# 4. 월별 품질표와 제외 행 감사 파일을 구성한다.

spatial_demand_quality = pd.DataFrame(
    spatial_quality_records
)


if excluded_monthly_frames:
    station_hourly_excluded = (
        pd.concat(
            excluded_monthly_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "datetime",
                "station_id",
            ]
        )
        .reset_index(drop=True)
    )

else:
    station_hourly_excluded = pd.DataFrame(
        columns=[
            "datetime",
            "station_id",
            "rental_count",
            "year_month",
            "source_file",
        ]
    )


spatial_demand_quality.to_csv(
    spatial_quality_path,
    index=False,
    encoding="utf-8-sig",
)


station_hourly_excluded.to_csv(
    spatial_excluded_audit_path,
    index=False,
    encoding="utf-8-sig",
)


# 5. 전체 기간 합계를 검증한다.

total_input_rows = int(
    spatial_demand_quality[
        "input_row_count"
    ].sum()
)

total_spatial_rows = int(
    spatial_demand_quality[
        "spatial_row_count"
    ].sum()
)

total_excluded_rows = int(
    spatial_demand_quality[
        "excluded_row_count"
    ].sum()
)


total_input_rental_count = int(
    spatial_demand_quality[
        "input_rental_count"
    ].sum()
)

total_spatial_rental_count = int(
    spatial_demand_quality[
        "spatial_rental_count"
    ].sum()
)

total_excluded_rental_count = int(
    spatial_demand_quality[
        "excluded_rental_count"
    ].sum()
)


generated_spatial_files = sorted(
    spatial_monthly_dir.glob(
        "station_hourly_*.csv"
    )
)


EXPECTED_INPUT_ROWS = 26_145_337
EXPECTED_SPATIAL_ROWS = 26_127_479
EXPECTED_EXCLUDED_ROWS = 17_858

EXPECTED_INPUT_RENTAL_COUNT = 88_750_388
EXPECTED_SPATIAL_RENTAL_COUNT = 88_708_877
EXPECTED_EXCLUDED_RENTAL_COUNT = 41_511


if len(generated_spatial_files) != 24:
    raise ValueError(
        "공간 분석용 월별 파일이 "
        f"24개가 아니다: {len(generated_spatial_files)}개"
    )


if total_input_rows != EXPECTED_INPUT_ROWS:
    raise ValueError(
        "전체 입력 행 수가 예상값과 다르다: "
        f"{total_input_rows:,}행"
    )


if total_spatial_rows != EXPECTED_SPATIAL_ROWS:
    raise ValueError(
        "공간 분석 대상 행 수가 예상값과 다르다: "
        f"{total_spatial_rows:,}행"
    )


if total_excluded_rows != EXPECTED_EXCLUDED_ROWS:
    raise ValueError(
        "공간 분석 제외 행 수가 예상값과 다르다: "
        f"{total_excluded_rows:,}행"
    )


if (
    total_spatial_rows
    + total_excluded_rows
    != total_input_rows
):
    raise ValueError(
        "전체 행 수가 분리 전후에 보존되지 않았다."
    )


if (
    total_input_rental_count
    != EXPECTED_INPUT_RENTAL_COUNT
):
    raise ValueError(
        "전체 입력 이용건수가 예상값과 다르다: "
        f"{total_input_rental_count:,}건"
    )


if (
    total_spatial_rental_count
    != EXPECTED_SPATIAL_RENTAL_COUNT
):
    raise ValueError(
        "공간 분석 이용건수가 예상값과 다르다: "
        f"{total_spatial_rental_count:,}건"
    )


if (
    total_excluded_rental_count
    != EXPECTED_EXCLUDED_RENTAL_COUNT
):
    raise ValueError(
        "공간 분석 제외 이용건수가 예상값과 다르다: "
        f"{total_excluded_rental_count:,}건"
    )


if (
    total_spatial_rental_count
    + total_excluded_rental_count
    != total_input_rental_count
):
    raise ValueError(
        "전체 이용건수가 분리 전후에 보존되지 않았다."
    )


# 6. 최종 결과를 출력한다.

print("\n[공간 분석용 수요 데이터 구축 결과]")
print(
    f"생성된 월별 파일: "
    f"{len(generated_spatial_files):,}개"
)
print(
    f"전체 입력 행 수: "
    f"{total_input_rows:,}행"
)
print(
    f"공간 분석 대상 행 수: "
    f"{total_spatial_rows:,}행"
)
print(
    f"공간 분석 제외 행 수: "
    f"{total_excluded_rows:,}행"
)
print(
    f"전체 입력 이용건수: "
    f"{total_input_rental_count:,}건"
)
print(
    f"공간 분석 대상 이용건수: "
    f"{total_spatial_rental_count:,}건"
)
print(
    f"공간 분석 제외 이용건수: "
    f"{total_excluded_rental_count:,}건"
)
print(
    "공간 분석 수요 커버리지: "
    f"{total_spatial_rental_count / total_input_rental_count * 100:.6f}%"
)
print(
    "행 수 보존 여부: "
    f"{total_spatial_rows + total_excluded_rows == total_input_rows}"
)
print(
    "이용건수 보존 여부: "
    f"{total_spatial_rental_count + total_excluded_rental_count == total_input_rental_count}"
)


print("\n[저장 파일 및 폴더]")
print(
    spatial_monthly_dir.relative_to(ROOT)
)
print(
    spatial_quality_path.relative_to(ROOT)
)
print(
    spatial_excluded_audit_path.relative_to(ROOT)
)


print("\n[월별 공간 분석 데이터 품질]")
display(spatial_demand_quality)


print("\n[공간 분석 제외 행 표본]")
display(
    station_hourly_excluded.head(20)
)

[대여소 ID 분류 검증]
이용정보 전체 대여소: 2,865개
공간 분석 대상 대여소: 2,847개
공간 분석 제외 대여소: 18개
미분류 대여소: 0개
이용정보에 없는 분류 ID: 0개
[01/24] 202301: 755,977행 → 분석 752,615행 / 제외 3,362행, 수요 보존 1,571,434건
[02/24] 202302: 897,582행 → 분석 893,812행 / 제외 3,770행, 수요 보존 2,231,457건
[03/24] 202303: 1,162,346행 → 분석 1,158,142행 / 제외 4,204행, 수요 보존 3,885,426건
[04/24] 202304: 1,129,551행 → 분석 1,126,112행 / 제외 3,439행, 수요 보존 4,081,995건
[05/24] 202305: 1,201,550행 → 분석 1,199,497행 / 제외 2,053행, 수요 보존 4,952,798건
[06/24] 202306: 1,224,866행 → 분석 1,224,443행 / 제외 423행, 수요 보존 4,933,508건
[07/24] 202307: 1,128,307행 → 분석 1,128,244행 / 제외 63행, 수요 보존 3,931,894건
[08/24] 202308: 1,153,894행 → 분석 1,153,861행 / 제외 33행, 수요 보존 3,935,325건
[09/24] 202309: 1,168,616행 → 분석 1,168,556행 / 제외 60행, 수요 보존 4,548,903건
[10/24] 202310: 1,303,271행 → 분석 1,303,240행 / 제외 31행, 수요 보존 5,293,068건
[11/24] 202311: 1,085,399행 → 분석 1,085,397행 / 제외 2행, 수요 보존 3,433,810건
[12/24] 202312: 850,806행 → 분석 850,800행 / 제외 6행, 수요 보존 2,104,399건
[13/24] 202401: 866,055행 → 분석 866,054행 / 제외 1행, 수요 보존

,year_month,input_file,output_file,input_row_count,spatial_row_count,excluded_row_count,input_station_count,spatial_station_count,excluded_station_count,input_rental_count,spatial_rental_count,excluded_rental_count,spatial_rental_share_pct
0,202301,station_hourly_202301.csv,station_hourly_202301.csv,755977,752615,3362,2709,2694,15,1571434,1565736,5698,99.637401
1,202302,station_hourly_202302.csv,station_hourly_202302.csv,897582,893812,3770,2715,2700,15,2231457,2223894,7563,99.661073
2,202303,station_hourly_202303.csv,station_hourly_202303.csv,1162346,1158142,4204,2723,2709,14,3885426,3874617,10809,99.721807
3,202304,station_hourly_202304.csv,station_hourly_202304.csv,1129551,1126112,3439,2721,2709,12,4081995,4073053,8942,99.780940
4,202305,station_hourly_202305.csv,station_hourly_202305.csv,1201550,1199497,2053,2724,2716,8,4952798,4946583,6215,99.874515
5,202306,station_hourly_202306.csv,station_hourly_202306.csv,1224866,1224443,423,2717,2712,5,4933508,4932262,1246,99.974744
6,202307,station_hourly_202307.csv,station_hourly_202307.csv,1128307,1128244,63,2733,2732,1,3931894,3931806,88,99.997762
7,202308,station_hourly_202308.csv,station_hourly_202308.csv,1153894,1153861,33,2729,2728,1,3935325,3935272,53,99.998653
8,202309,station_hourly_202309.csv,station_hourly_202309.csv,1168616,1168556,60,2729,2728,1,4548903,4548821,82,99.998197
9,202310,station_hourly_202310.csv,station_hourly_202310.csv,1303271,1303240,31,2734,2732,2,5293068,5293021,47,99.999112



[공간 분석 제외 행 표본]


,datetime,station_id,rental_count,year_month,source_file
0,2023-01-01 00:00:00,1413,1,202301,station_hourly_202301.csv
1,2023-01-01 00:00:00,3011,1,202301,station_hourly_202301.csv
2,2023-01-01 00:00:00,3578,2,202301,station_hourly_202301.csv
3,2023-01-01 00:00:00,4093,1,202301,station_hourly_202301.csv
4,2023-01-01 00:00:00,492,1,202301,station_hourly_202301.csv
5,2023-01-01 01:00:00,1249,1,202301,station_hourly_202301.csv
6,2023-01-01 01:00:00,1746,1,202301,station_hourly_202301.csv
7,2023-01-01 01:00:00,4239,1,202301,station_hourly_202301.csv
8,2023-01-01 01:00:00,972,1,202301,station_hourly_202301.csv
9,2023-01-01 02:00:00,3011,2,202301,station_hourly_202301.csv


## 22. 전처리 최종 산출물 무결성 검증

전처리 과정에서 생성한 모든 핵심 산출물을 다시 불러와 최종 무결성을 검증한다.

최종 검증 항목은 다음과 같다.

- 원본 집계 및 공간 분석용 월별 파일이 각각 24개인지 확인한다.
- 실제 대여소번호 기반 분석용 마스터와 제외 목록의 대여소 수를 확인한다.
- 대여소번호 중복, 핵심 변수 결측 및 비정상 GPS를 확인한다.
- 월별 파일에서 대여소번호와 시간대 조합의 중복 여부를 확인한다.
- 이용건수가 결측·음수·비정수 값을 포함하지 않는지 확인한다.
- 각 월별 파일의 관측 시점이 파일명에 표시된 연월과 일치하는지 확인한다.
- 전체 행 수와 이용건수가 공간 분석 대상 및 제외 데이터로 정확히 분리되는지 확인한다.
- 분석용 대여소 마스터와 공간 분석용 수요 데이터의 대여소번호 집합이 일치하는지 확인한다.
- 2023년 12월 수동 교정 내역이 6건 보존되었는지 확인한다.

모든 검증 결과는 별도 CSV 파일로 저장한다. 하나라도 실패하면 노트북 실행을 중단하여 잘못된 데이터가 후속 공간 군집화 단계로 전달되지 않도록 한다.

In [23]:
# 1. 최종 검증 기준값과 경로를 설정한다.

EXPECTED_MONTH_COUNT = 24

EXPECTED_TOTAL_STATION_COUNT = 2_865
EXPECTED_SPATIAL_STATION_COUNT = 2_847
EXPECTED_EXCLUDED_STATION_COUNT = 18

EXPECTED_INPUT_ROW_COUNT = 26_145_337
EXPECTED_SPATIAL_ROW_COUNT = 26_127_479
EXPECTED_EXCLUDED_ROW_COUNT = 17_858

EXPECTED_INPUT_RENTAL_COUNT = 88_750_388
EXPECTED_SPATIAL_RENTAL_COUNT = 88_708_877
EXPECTED_EXCLUDED_RENTAL_COUNT = 41_511

EXPECTED_CORRECTION_COUNT = 6


monthly_output_dir = (
    processed_dir
    / "station_hourly_monthly"
)

spatial_monthly_dir = (
    processed_dir
    / "station_hourly_spatial_monthly"
)


clean_station_path = (
    processed_dir
    / "station_master_clean.csv"
)

excluded_station_path = (
    processed_dir
    / "station_master_invalid_gps.csv"
)

bike_quality_path = (
    processed_dir
    / "bike_usage_quality_by_month.csv"
)

manual_correction_path = (
    processed_dir
    / "bike_usage_manual_corrections.csv"
)

station_coordinate_audit_path = (
    processed_dir
    / "station_coordinate_movement_audit.csv"
)

station_master_quality_path = (
    processed_dir
    / "station_master_quality_summary.csv"
)

spatial_quality_path = (
    processed_dir
    / "spatial_demand_quality_by_month.csv"
)

spatial_excluded_audit_path = (
    processed_dir
    / "station_hourly_excluded_from_spatial.csv"
)

final_validation_path = (
    processed_dir
    / "preprocessing_final_validation.csv"
)

final_summary_path = (
    processed_dir
    / "preprocessing_final_summary.csv"
)

monthly_final_audit_path = (
    processed_dir
    / "preprocessing_monthly_final_audit.csv"
)


expected_year_months = [
    period.strftime("%Y%m")
    for period in pd.period_range(
        "2023-01",
        "2024-12",
        freq="M",
    )
]


expected_period_texts = [
    period.strftime("%Y-%m")
    for period in pd.period_range(
        "2023-01",
        "2024-12",
        freq="M",
    )
]


expected_monthly_file_names = [
    f"station_hourly_{year_month}.csv"
    for year_month in expected_year_months
]


# 2. 검증 보조 함수를 정의한다.

validation_records = []


def add_validation(
    validation_item: str,
    passed: bool,
    actual,
    expected,
    detail: str = "",
) -> None:
    """최종 검증 결과를 한 행으로 기록한다."""

    validation_records.append(
        {
            "validation_item": validation_item,
            "passed": bool(passed),
            "actual": str(actual),
            "expected": str(expected),
            "detail": detail,
        }
    )


def canonicalize_station_id_for_audit(
    series: pd.Series,
) -> pd.Series:
    """감사용 대여소번호를 앞자리 0이 없는 문자열로 변환한다."""

    raw_id = (
        series
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
    )

    numeric_id = pd.to_numeric(
        raw_id,
        errors="coerce",
    )

    integer_mask = (
        numeric_id.notna()
        & numeric_id.mod(1).eq(0)
    )

    canonical_id = pd.Series(
        pd.NA,
        index=series.index,
        dtype="string",
    )

    canonical_id.loc[integer_mask] = (
        numeric_id.loc[integer_mask]
        .astype("int64")
        .astype("string")
    )

    return canonical_id


def parse_boolean_series(
    series: pd.Series,
) -> pd.Series:
    """CSV에서 불러온 불리언 값을 안전하게 변환한다."""

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return (
        normalized
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


# 3. 필수 산출물 존재 여부를 확인한다.

required_artifacts = {
    "분석용 대여소 마스터": clean_station_path,
    "공간 분석 제외 대여소": excluded_station_path,
    "이용정보 월별 품질표": bike_quality_path,
    "2023년 12월 수동 교정 내역": manual_correction_path,
    "대여소 좌표 변경 감사표": station_coordinate_audit_path,
    "대여소 마스터 품질표": station_master_quality_path,
    "공간 분석 월별 품질표": spatial_quality_path,
    "공간 분석 제외 행 감사표": spatial_excluded_audit_path,
}


artifact_inventory_records = []


for artifact_name, artifact_path in required_artifacts.items():
    exists = artifact_path.exists()

    artifact_inventory_records.append(
        {
            "artifact_name": artifact_name,
            "relative_path": str(
                artifact_path.relative_to(ROOT)
            ),
            "exists": exists,
            "file_size_mb": (
                round(
                    artifact_path.stat().st_size
                    / (1024**2),
                    3,
                )
                if exists
                else pd.NA
            ),
        }
    )

    add_validation(
        validation_item=(
            f"필수 산출물 존재: {artifact_name}"
        ),
        passed=exists,
        actual=exists,
        expected=True,
        detail=str(
            artifact_path.relative_to(ROOT)
        ),
    )


artifact_inventory = pd.DataFrame(
    artifact_inventory_records
)


missing_artifacts = [
    artifact_name
    for artifact_name, artifact_path
    in required_artifacts.items()
    if not artifact_path.exists()
]


if missing_artifacts:
    preliminary_validation = pd.DataFrame(
        validation_records
    )

    preliminary_validation.to_csv(
        final_validation_path,
        index=False,
        encoding="utf-8-sig",
    )

    raise FileNotFoundError(
        "필수 전처리 산출물이 존재하지 않는다: "
        f"{missing_artifacts}"
    )


print("[필수 산출물 목록]")
display(artifact_inventory)


# 4. 월별 파일 수와 파일명을 확인한다.

monthly_input_files = sorted(
    monthly_output_dir.glob(
        "station_hourly_*.csv"
    ),
    key=lambda path: path.name,
)


monthly_spatial_files = sorted(
    spatial_monthly_dir.glob(
        "station_hourly_*.csv"
    ),
    key=lambda path: path.name,
)


input_file_names = [
    path.name
    for path in monthly_input_files
]


spatial_file_names = [
    path.name
    for path in monthly_spatial_files
]


add_validation(
    validation_item="원본 집계 월별 파일 수",
    passed=(
        len(monthly_input_files)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(monthly_input_files),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="공간 분석용 월별 파일 수",
    passed=(
        len(monthly_spatial_files)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(monthly_spatial_files),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="원본 집계 월별 파일명",
    passed=(
        input_file_names
        == expected_monthly_file_names
    ),
    actual=input_file_names,
    expected=expected_monthly_file_names,
)


add_validation(
    validation_item="공간 분석용 월별 파일명",
    passed=(
        spatial_file_names
        == expected_monthly_file_names
    ),
    actual=spatial_file_names,
    expected=expected_monthly_file_names,
)


if (
    input_file_names
    != expected_monthly_file_names
    or spatial_file_names
    != expected_monthly_file_names
):
    preliminary_validation = pd.DataFrame(
        validation_records
    )

    preliminary_validation.to_csv(
        final_validation_path,
        index=False,
        encoding="utf-8-sig",
    )

    raise ValueError(
        "월별 파일명 또는 파일 수가 예상 범위와 다르다."
    )


# 5. 대여소 마스터와 제외 목록을 검증한다.

station_master_clean_final = pd.read_csv(
    clean_station_path,
    encoding="utf-8-sig",
    dtype={
        "station_id": "string",
        "rental_count": "Int64",
    },
)


station_master_excluded_final = pd.read_csv(
    excluded_station_path,
    encoding="utf-8-sig",
    dtype={
        "station_id": "string",
        "rental_count": "Int64",
    },
)


station_master_clean_final[
    "station_id"
] = canonicalize_station_id_for_audit(
    station_master_clean_final[
        "station_id"
    ]
)


station_master_excluded_final[
    "station_id"
] = canonicalize_station_id_for_audit(
    station_master_excluded_final[
        "station_id"
    ]
)


clean_station_ids = set(
    station_master_clean_final[
        "station_id"
    ].dropna()
)


excluded_station_ids = set(
    station_master_excluded_final[
        "station_id"
    ].dropna()
)


master_union_ids = (
    clean_station_ids
    | excluded_station_ids
)


master_intersection_ids = (
    clean_station_ids
    & excluded_station_ids
)


clean_master_rental_count = int(
    station_master_clean_final[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)


excluded_master_rental_count = int(
    station_master_excluded_final[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)


valid_master_gps = (
    station_master_clean_final[
        "latitude"
    ]
    .between(
        37.4,
        37.8,
    )
    .fillna(False)
    & station_master_clean_final[
        "longitude"
    ]
    .between(
        126.7,
        127.3,
    )
    .fillna(False)
)


add_validation(
    validation_item="분석용 대여소 마스터 행 수",
    passed=(
        len(station_master_clean_final)
        == EXPECTED_SPATIAL_STATION_COUNT
    ),
    actual=len(station_master_clean_final),
    expected=EXPECTED_SPATIAL_STATION_COUNT,
)


add_validation(
    validation_item="공간 분석 제외 대여소 행 수",
    passed=(
        len(station_master_excluded_final)
        == EXPECTED_EXCLUDED_STATION_COUNT
    ),
    actual=len(station_master_excluded_final),
    expected=EXPECTED_EXCLUDED_STATION_COUNT,
)


add_validation(
    validation_item="분석용 대여소 ID 중복",
    passed=(
        not station_master_clean_final[
            "station_id"
        ].duplicated().any()
    ),
    actual=int(
        station_master_clean_final[
            "station_id"
        ].duplicated().sum()
    ),
    expected=0,
)


add_validation(
    validation_item="제외 대여소 ID 중복",
    passed=(
        not station_master_excluded_final[
            "station_id"
        ].duplicated().any()
    ),
    actual=int(
        station_master_excluded_final[
            "station_id"
        ].duplicated().sum()
    ),
    expected=0,
)


add_validation(
    validation_item="분석용·제외 대여소 ID 교집합",
    passed=(
        len(master_intersection_ids) == 0
    ),
    actual=len(master_intersection_ids),
    expected=0,
)


add_validation(
    validation_item="전체 이용 대여소 ID 합집합",
    passed=(
        len(master_union_ids)
        == EXPECTED_TOTAL_STATION_COUNT
    ),
    actual=len(master_union_ids),
    expected=EXPECTED_TOTAL_STATION_COUNT,
)


add_validation(
    validation_item="분석용 마스터 유효 GPS",
    passed=bool(
        valid_master_gps.all()
    ),
    actual=int(
        valid_master_gps.sum()
    ),
    expected=EXPECTED_SPATIAL_STATION_COUNT,
)


add_validation(
    validation_item="분석용 마스터 핵심 변수 결측",
    passed=(
        not station_master_clean_final[
            [
                "station_id",
                "latitude",
                "longitude",
            ]
        ]
        .isna()
        .any()
        .any()
    ),
    actual=int(
        station_master_clean_final[
            [
                "station_id",
                "latitude",
                "longitude",
            ]
        ]
        .isna()
        .sum()
        .sum()
    ),
    expected=0,
)


add_validation(
    validation_item="분석용 마스터 이용건수 합계",
    passed=(
        clean_master_rental_count
        == EXPECTED_SPATIAL_RENTAL_COUNT
    ),
    actual=clean_master_rental_count,
    expected=EXPECTED_SPATIAL_RENTAL_COUNT,
)


add_validation(
    validation_item="제외 마스터 이용건수 합계",
    passed=(
        excluded_master_rental_count
        == EXPECTED_EXCLUDED_RENTAL_COUNT
    ),
    actual=excluded_master_rental_count,
    expected=EXPECTED_EXCLUDED_RENTAL_COUNT,
)


# 6. 월별 파일을 다시 읽어 무결성을 검증한다.

total_input_rows = 0
total_spatial_rows = 0

total_input_rental_count = 0
total_spatial_rental_count = 0

input_duplicate_key_count = 0
spatial_duplicate_key_count = 0

input_missing_core_count = 0
spatial_missing_core_count = 0

input_negative_rental_count = 0
spatial_negative_rental_count = 0

input_non_integer_count = 0
spatial_non_integer_count = 0

input_month_mismatch_count = 0
spatial_month_mismatch_count = 0

input_station_id_union = set()
spatial_station_id_union = set()

input_period_union = set()
spatial_period_union = set()

input_datetime_min = None
input_datetime_max = None

spatial_datetime_min = None
spatial_datetime_max = None

monthly_audit_records = []


for file_index, (
    input_path,
    spatial_path,
    expected_year_month,
    expected_period_text,
) in enumerate(
    zip(
        monthly_input_files,
        monthly_spatial_files,
        expected_year_months,
        expected_period_texts,
    ),
    start=1,
):
    input_data = pd.read_csv(
        input_path,
        encoding="utf-8-sig",
        usecols=[
            "datetime",
            "station_id",
            "rental_count",
        ],
        dtype={
            "station_id": "string",
        },
    )


    spatial_data = pd.read_csv(
        spatial_path,
        encoding="utf-8-sig",
        usecols=[
            "datetime",
            "station_id",
            "rental_count",
        ],
        dtype={
            "station_id": "string",
        },
    )


    input_data[
        "datetime"
    ] = pd.to_datetime(
        input_data[
            "datetime"
        ],
        errors="coerce",
    )


    spatial_data[
        "datetime"
    ] = pd.to_datetime(
        spatial_data[
            "datetime"
        ],
        errors="coerce",
    )


    input_data[
        "rental_count"
    ] = pd.to_numeric(
        input_data[
            "rental_count"
        ],
        errors="coerce",
    )


    spatial_data[
        "rental_count"
    ] = pd.to_numeric(
        spatial_data[
            "rental_count"
        ],
        errors="coerce",
    )


    input_data[
        "station_id_canonical"
    ] = canonicalize_station_id_for_audit(
        input_data[
            "station_id"
        ]
    )


    spatial_data[
        "station_id_canonical"
    ] = canonicalize_station_id_for_audit(
        spatial_data[
            "station_id"
        ]
    )


    input_core_missing = int(
        input_data[
            [
                "datetime",
                "station_id_canonical",
                "rental_count",
            ]
        ]
        .isna()
        .sum()
        .sum()
    )


    spatial_core_missing = int(
        spatial_data[
            [
                "datetime",
                "station_id_canonical",
                "rental_count",
            ]
        ]
        .isna()
        .sum()
        .sum()
    )


    input_duplicate_count = int(
        input_data.duplicated(
            subset=[
                "datetime",
                "station_id_canonical",
            ],
            keep=False,
        ).sum()
    )


    spatial_duplicate_count = int(
        spatial_data.duplicated(
            subset=[
                "datetime",
                "station_id_canonical",
            ],
            keep=False,
        ).sum()
    )


    input_negative_count = int(
        input_data[
            "rental_count"
        ]
        .lt(0)
        .fillna(False)
        .sum()
    )


    spatial_negative_count = int(
        spatial_data[
            "rental_count"
        ]
        .lt(0)
        .fillna(False)
        .sum()
    )


    input_non_integer = int(
        (
            input_data[
                "rental_count"
            ].notna()
            & ~input_data[
                "rental_count"
            ].mod(1).eq(0)
        ).sum()
    )


    spatial_non_integer = int(
        (
            spatial_data[
                "rental_count"
            ].notna()
            & ~spatial_data[
                "rental_count"
            ].mod(1).eq(0)
        ).sum()
    )


    input_period = (
        input_data[
            "datetime"
        ]
        .dt.to_period("M")
        .astype("string")
    )


    spatial_period = (
        spatial_data[
            "datetime"
        ]
        .dt.to_period("M")
        .astype("string")
    )


    # 먼저 불리언 Series의 합계를 계산한 뒤 int로 변환한다.
    input_month_mismatch = int(
        (
            input_period.notna()
            & input_period.ne(
                expected_period_text
            )
        ).sum()
    )


    spatial_month_mismatch = int(
        (
            spatial_period.notna()
            & spatial_period.ne(
                expected_period_text
            )
        ).sum()
    )


    input_row_count = len(
        input_data
    )


    spatial_row_count = len(
        spatial_data
    )


    input_month_rental_count = int(
        input_data[
            "rental_count"
        ]
        .fillna(0)
        .sum()
    )


    spatial_month_rental_count = int(
        spatial_data[
            "rental_count"
        ]
        .fillna(0)
        .sum()
    )


    input_month_station_ids = set(
        input_data[
            "station_id_canonical"
        ]
        .dropna()
        .tolist()
    )


    spatial_month_station_ids = set(
        spatial_data[
            "station_id_canonical"
        ]
        .dropna()
        .tolist()
    )


    unknown_spatial_station_ids = (
        spatial_month_station_ids
        - clean_station_ids
    )


    total_input_rows += (
        input_row_count
    )

    total_spatial_rows += (
        spatial_row_count
    )

    total_input_rental_count += (
        input_month_rental_count
    )

    total_spatial_rental_count += (
        spatial_month_rental_count
    )


    input_duplicate_key_count += (
        input_duplicate_count
    )

    spatial_duplicate_key_count += (
        spatial_duplicate_count
    )

    input_missing_core_count += (
        input_core_missing
    )

    spatial_missing_core_count += (
        spatial_core_missing
    )

    input_negative_rental_count += (
        input_negative_count
    )

    spatial_negative_rental_count += (
        spatial_negative_count
    )

    input_non_integer_count += (
        input_non_integer
    )

    spatial_non_integer_count += (
        spatial_non_integer
    )

    input_month_mismatch_count += (
        input_month_mismatch
    )

    spatial_month_mismatch_count += (
        spatial_month_mismatch
    )


    input_station_id_union.update(
        input_month_station_ids
    )

    spatial_station_id_union.update(
        spatial_month_station_ids
    )


    input_period_union.update(
        input_period
        .dropna()
        .tolist()
    )

    spatial_period_union.update(
        spatial_period
        .dropna()
        .tolist()
    )


    current_input_min = (
        input_data[
            "datetime"
        ].min()
    )

    current_input_max = (
        input_data[
            "datetime"
        ].max()
    )

    current_spatial_min = (
        spatial_data[
            "datetime"
        ].min()
    )

    current_spatial_max = (
        spatial_data[
            "datetime"
        ].max()
    )


    if pd.notna(current_input_min):
        input_datetime_min = (
            current_input_min
            if input_datetime_min is None
            else min(
                input_datetime_min,
                current_input_min,
            )
        )


    if pd.notna(current_input_max):
        input_datetime_max = (
            current_input_max
            if input_datetime_max is None
            else max(
                input_datetime_max,
                current_input_max,
            )
        )


    if pd.notna(current_spatial_min):
        spatial_datetime_min = (
            current_spatial_min
            if spatial_datetime_min is None
            else min(
                spatial_datetime_min,
                current_spatial_min,
            )
        )


    if pd.notna(current_spatial_max):
        spatial_datetime_max = (
            current_spatial_max
            if spatial_datetime_max is None
            else max(
                spatial_datetime_max,
                current_spatial_max,
            )
        )


    monthly_audit_records.append(
        {
            "year_month": (
                expected_year_month
            ),
            "input_file": (
                input_path.name
            ),
            "spatial_file": (
                spatial_path.name
            ),
            "input_row_count": (
                input_row_count
            ),
            "spatial_row_count": (
                spatial_row_count
            ),
            "input_rental_count": (
                input_month_rental_count
            ),
            "spatial_rental_count": (
                spatial_month_rental_count
            ),
            "input_station_count": (
                len(input_month_station_ids)
            ),
            "spatial_station_count": (
                len(spatial_month_station_ids)
            ),
            "input_duplicate_key_count": (
                input_duplicate_count
            ),
            "spatial_duplicate_key_count": (
                spatial_duplicate_count
            ),
            "input_missing_core_count": (
                input_core_missing
            ),
            "spatial_missing_core_count": (
                spatial_core_missing
            ),
            "input_negative_rental_count": (
                input_negative_count
            ),
            "spatial_negative_rental_count": (
                spatial_negative_count
            ),
            "input_non_integer_count": (
                input_non_integer
            ),
            "spatial_non_integer_count": (
                spatial_non_integer
            ),
            "input_month_mismatch_count": (
                input_month_mismatch
            ),
            "spatial_month_mismatch_count": (
                spatial_month_mismatch
            ),
            "unknown_spatial_station_count": (
                len(
                    unknown_spatial_station_ids
                )
            ),
        }
    )


    print(
        f"[{file_index:02d}/24] "
        f"{expected_year_month}: "
        f"입력 {input_row_count:,}행 / "
        f"공간 분석 {spatial_row_count:,}행"
    )


    del input_data
    del spatial_data
    gc.collect()


monthly_final_audit = pd.DataFrame(
    monthly_audit_records
)


# 7. 제외 행 감사 파일과 품질 보고서를 검증한다.

spatial_excluded_audit = pd.read_csv(
    spatial_excluded_audit_path,
    encoding="utf-8-sig",
    dtype={
        "station_id": "string",
    },
)


spatial_excluded_audit[
    "station_id"
] = canonicalize_station_id_for_audit(
    spatial_excluded_audit[
        "station_id"
    ]
)


spatial_excluded_audit[
    "rental_count"
] = pd.to_numeric(
    spatial_excluded_audit[
        "rental_count"
    ],
    errors="coerce",
)


excluded_audit_row_count = len(
    spatial_excluded_audit
)


excluded_audit_rental_count = int(
    spatial_excluded_audit[
        "rental_count"
    ]
    .fillna(0)
    .sum()
)


excluded_audit_station_ids = set(
    spatial_excluded_audit[
        "station_id"
    ]
    .dropna()
    .tolist()
)


spatial_quality_final = pd.read_csv(
    spatial_quality_path,
    encoding="utf-8-sig",
    dtype={
        "year_month": "string",
    },
)


bike_quality_final = pd.read_csv(
    bike_quality_path,
    encoding="utf-8-sig",
)


manual_corrections_final = pd.read_csv(
    manual_correction_path,
    encoding="utf-8-sig",
)


add_validation(
    validation_item="전체 원본 집계 행 수",
    passed=(
        total_input_rows
        == EXPECTED_INPUT_ROW_COUNT
    ),
    actual=total_input_rows,
    expected=EXPECTED_INPUT_ROW_COUNT,
)


add_validation(
    validation_item="전체 공간 분석 행 수",
    passed=(
        total_spatial_rows
        == EXPECTED_SPATIAL_ROW_COUNT
    ),
    actual=total_spatial_rows,
    expected=EXPECTED_SPATIAL_ROW_COUNT,
)


add_validation(
    validation_item="전체 공간 분석 제외 행 수",
    passed=(
        excluded_audit_row_count
        == EXPECTED_EXCLUDED_ROW_COUNT
    ),
    actual=excluded_audit_row_count,
    expected=EXPECTED_EXCLUDED_ROW_COUNT,
)


add_validation(
    validation_item="전체 행 수 분리 보존",
    passed=(
        total_spatial_rows
        + excluded_audit_row_count
        == total_input_rows
    ),
    actual=(
        total_spatial_rows
        + excluded_audit_row_count
    ),
    expected=total_input_rows,
)


add_validation(
    validation_item="전체 원본 집계 이용건수",
    passed=(
        total_input_rental_count
        == EXPECTED_INPUT_RENTAL_COUNT
    ),
    actual=total_input_rental_count,
    expected=EXPECTED_INPUT_RENTAL_COUNT,
)


add_validation(
    validation_item="전체 공간 분석 이용건수",
    passed=(
        total_spatial_rental_count
        == EXPECTED_SPATIAL_RENTAL_COUNT
    ),
    actual=total_spatial_rental_count,
    expected=EXPECTED_SPATIAL_RENTAL_COUNT,
)


add_validation(
    validation_item="전체 공간 분석 제외 이용건수",
    passed=(
        excluded_audit_rental_count
        == EXPECTED_EXCLUDED_RENTAL_COUNT
    ),
    actual=excluded_audit_rental_count,
    expected=EXPECTED_EXCLUDED_RENTAL_COUNT,
)


add_validation(
    validation_item="전체 이용건수 분리 보존",
    passed=(
        total_spatial_rental_count
        + excluded_audit_rental_count
        == total_input_rental_count
    ),
    actual=(
        total_spatial_rental_count
        + excluded_audit_rental_count
    ),
    expected=total_input_rental_count,
)


add_validation(
    validation_item="원본 월별 키 중복",
    passed=(
        input_duplicate_key_count == 0
    ),
    actual=input_duplicate_key_count,
    expected=0,
)


add_validation(
    validation_item="공간 분석 월별 키 중복",
    passed=(
        spatial_duplicate_key_count == 0
    ),
    actual=spatial_duplicate_key_count,
    expected=0,
)


add_validation(
    validation_item="원본 핵심 변수 결측",
    passed=(
        input_missing_core_count == 0
    ),
    actual=input_missing_core_count,
    expected=0,
)


add_validation(
    validation_item="공간 분석 핵심 변수 결측",
    passed=(
        spatial_missing_core_count == 0
    ),
    actual=spatial_missing_core_count,
    expected=0,
)


add_validation(
    validation_item="원본 음수 이용건수",
    passed=(
        input_negative_rental_count == 0
    ),
    actual=input_negative_rental_count,
    expected=0,
)


add_validation(
    validation_item="공간 분석 음수 이용건수",
    passed=(
        spatial_negative_rental_count == 0
    ),
    actual=spatial_negative_rental_count,
    expected=0,
)


add_validation(
    validation_item="원본 비정수 이용건수",
    passed=(
        input_non_integer_count == 0
    ),
    actual=input_non_integer_count,
    expected=0,
)


add_validation(
    validation_item="공간 분석 비정수 이용건수",
    passed=(
        spatial_non_integer_count == 0
    ),
    actual=spatial_non_integer_count,
    expected=0,
)


add_validation(
    validation_item="원본 파일명·관측월 불일치",
    passed=(
        input_month_mismatch_count == 0
    ),
    actual=input_month_mismatch_count,
    expected=0,
)


add_validation(
    validation_item="공간 분석 파일명·관측월 불일치",
    passed=(
        spatial_month_mismatch_count == 0
    ),
    actual=spatial_month_mismatch_count,
    expected=0,
)


add_validation(
    validation_item="원본 이용 대여소 ID 수",
    passed=(
        len(input_station_id_union)
        == EXPECTED_TOTAL_STATION_COUNT
    ),
    actual=len(input_station_id_union),
    expected=EXPECTED_TOTAL_STATION_COUNT,
)


add_validation(
    validation_item="공간 분석 대여소 ID 수",
    passed=(
        len(spatial_station_id_union)
        == EXPECTED_SPATIAL_STATION_COUNT
    ),
    actual=len(spatial_station_id_union),
    expected=EXPECTED_SPATIAL_STATION_COUNT,
)


add_validation(
    validation_item="원본 대여소 ID와 마스터 합집합 일치",
    passed=(
        input_station_id_union
        == master_union_ids
    ),
    actual=len(
        input_station_id_union
        ^ master_union_ids
    ),
    expected=0,
    detail="대칭차집합 ID 수",
)


add_validation(
    validation_item="공간 분석 대여소 ID와 분석용 마스터 일치",
    passed=(
        spatial_station_id_union
        == clean_station_ids
    ),
    actual=len(
        spatial_station_id_union
        ^ clean_station_ids
    ),
    expected=0,
    detail="대칭차집합 ID 수",
)


add_validation(
    validation_item="제외 행 대여소 ID와 제외 마스터 일치",
    passed=(
        excluded_audit_station_ids
        == excluded_station_ids
    ),
    actual=len(
        excluded_audit_station_ids
        ^ excluded_station_ids
    ),
    expected=0,
    detail="대칭차집합 ID 수",
)


add_validation(
    validation_item="원본 관측월 수",
    passed=(
        len(input_period_union)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(input_period_union),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="공간 분석 관측월 수",
    passed=(
        len(spatial_period_union)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(spatial_period_union),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="원본 관측월 목록",
    passed=(
        sorted(input_period_union)
        == expected_period_texts
    ),
    actual=sorted(input_period_union),
    expected=expected_period_texts,
)


add_validation(
    validation_item="공간 분석 관측월 목록",
    passed=(
        sorted(spatial_period_union)
        == expected_period_texts
    ),
    actual=sorted(spatial_period_union),
    expected=expected_period_texts,
)


add_validation(
    validation_item="이용정보 품질표 월 수",
    passed=(
        len(bike_quality_final)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(bike_quality_final),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="공간 분석 품질표 월 수",
    passed=(
        len(spatial_quality_final)
        == EXPECTED_MONTH_COUNT
    ),
    actual=len(spatial_quality_final),
    expected=EXPECTED_MONTH_COUNT,
)


add_validation(
    validation_item="2023년 12월 수동 교정 행 수",
    passed=(
        len(manual_corrections_final)
        == EXPECTED_CORRECTION_COUNT
    ),
    actual=len(manual_corrections_final),
    expected=EXPECTED_CORRECTION_COUNT,
)


# 8. 최종 검증 보고서와 요약표를 저장한다.

final_validation = pd.DataFrame(
    validation_records
)


final_validation[
    "status"
] = np.where(
    final_validation[
        "passed"
    ],
    "PASS",
    "FAIL",
)


final_validation = final_validation[
    [
        "validation_item",
        "status",
        "passed",
        "actual",
        "expected",
        "detail",
    ]
]


all_validation_passed = bool(
    final_validation[
        "passed"
    ].all()
)


failed_validation_count = int(
    (
        ~final_validation[
            "passed"
        ]
    ).sum()
)


if (
    "meaningful_coordinate_movement"
    in station_master_clean_final.columns
):
    meaningful_movement_count = int(
        parse_boolean_series(
            station_master_clean_final[
                "meaningful_coordinate_movement"
            ]
        ).sum()
    )

else:
    meaningful_movement_count = pd.NA


preprocessing_final_summary = pd.DataFrame(
    {
        "summary_item": [
            "분석 기간",
            "원천 이용정보 행 수",
            "유효 원천 이용정보 행 수",
            "월별 집계 파일 수",
            "전체 집계 행 수",
            "전체 이용건수",
            "공간 분석용 월별 파일 수",
            "공간 분석 대상 대여소",
            "공간 분석 제외 대여소",
            "공간 분석 대상 행 수",
            "공간 분석 제외 행 수",
            "공간 분석 대상 이용건수",
            "공간 분석 제외 이용건수",
            "공간 분석 수요 커버리지",
            "10m 이상 좌표 이동 대여소",
            "수동 교정 행 수",
            "최종 검증 항목 수",
            "최종 검증 실패 항목 수",
            "최종 검증 결과",
        ],
        "value": [
            "2023-01-01 ~ 2024-12-31",
            79_670_647,
            79_670_647,
            len(monthly_input_files),
            total_input_rows,
            total_input_rental_count,
            len(monthly_spatial_files),
            len(clean_station_ids),
            len(excluded_station_ids),
            total_spatial_rows,
            excluded_audit_row_count,
            total_spatial_rental_count,
            excluded_audit_rental_count,
            (
                total_spatial_rental_count
                / total_input_rental_count
                * 100
            ),
            meaningful_movement_count,
            len(manual_corrections_final),
            len(final_validation),
            failed_validation_count,
            (
                "PASS"
                if all_validation_passed
                else "FAIL"
            ),
        ],
    }
)


final_validation.to_csv(
    final_validation_path,
    index=False,
    encoding="utf-8-sig",
)


preprocessing_final_summary.to_csv(
    final_summary_path,
    index=False,
    encoding="utf-8-sig",
)


monthly_final_audit.to_csv(
    monthly_final_audit_path,
    index=False,
    encoding="utf-8-sig",
)


# 9. 최종 결과를 출력한다.

print("\n[전처리 최종 검증 결과]")
print(
    f"전체 검증 항목: "
    f"{len(final_validation):,}개"
)
print(
    f"통과 항목: "
    f"{int(final_validation['passed'].sum()):,}개"
)
print(
    f"실패 항목: "
    f"{failed_validation_count:,}개"
)
print(
    f"최종 결과: "
    f"{'PASS' if all_validation_passed else 'FAIL'}"
)


print("\n[관측 기간]")
print(
    f"원본 집계: "
    f"{input_datetime_min} "
    f"~ {input_datetime_max}"
)
print(
    f"공간 분석: "
    f"{spatial_datetime_min} "
    f"~ {spatial_datetime_max}"
)


print("\n[전처리 최종 요약]")
display(
    preprocessing_final_summary
)


print("\n[검증 항목 전체]")
display(
    final_validation
)


print("\n[저장 파일]")
print(
    final_validation_path.relative_to(ROOT)
)
print(
    final_summary_path.relative_to(ROOT)
)
print(
    monthly_final_audit_path.relative_to(ROOT)
)


if not all_validation_passed:
    failed_items = (
        final_validation.loc[
            ~final_validation[
                "passed"
            ],
            [
                "validation_item",
                "actual",
                "expected",
                "detail",
            ],
        ]
        .reset_index(drop=True)
    )

    print("\n[실패 검증 항목]")
    display(
        failed_items
    )

    raise ValueError(
        "전처리 최종 무결성 검증에 실패했다. "
        f"실패 항목 수: "
        f"{failed_validation_count}개"
    )


print(
    "\n모든 전처리 산출물의 "
    "무결성 검증을 통과했다."
)

print(
    "01_data_quality_preprocessing.ipynb "
    "실행이 완료되었다."
)

[필수 산출물 목록]


,artifact_name,relative_path,exists,file_size_mb
0,분석용 대여소 마스터,data\processed\station_master_clean.csv,True,0.822
1,공간 분석 제외 대여소,data\processed\station_master_invalid_gps.csv,True,0.002
2,이용정보 월별 품질표,data\processed\bike_usage_quality_by_month.csv,True,0.007
3,2023년 12월 수동 교정 내역,data\processed\bike_usage_manual_corrections.csv,True,0.002
4,대여소 좌표 변경 감사표,data\processed\station_coordinate_movement_aud...,True,0.430
5,대여소 마스터 품질표,data\processed\station_master_quality_summary.csv,True,0.001
6,공간 분석 월별 품질표,data\processed\spatial_demand_quality_by_month...,True,0.003
7,공간 분석 제외 행 감사표,data\processed\station_hourly_excluded_from_sp...,True,1.036


[01/24] 202301: 입력 755,977행 / 공간 분석 752,615행
[02/24] 202302: 입력 897,582행 / 공간 분석 893,812행
[03/24] 202303: 입력 1,162,346행 / 공간 분석 1,158,142행
[04/24] 202304: 입력 1,129,551행 / 공간 분석 1,126,112행
[05/24] 202305: 입력 1,201,550행 / 공간 분석 1,199,497행
[06/24] 202306: 입력 1,224,866행 / 공간 분석 1,224,443행
[07/24] 202307: 입력 1,128,307행 / 공간 분석 1,128,244행
[08/24] 202308: 입력 1,153,894행 / 공간 분석 1,153,861행
[09/24] 202309: 입력 1,168,616행 / 공간 분석 1,168,556행
[10/24] 202310: 입력 1,303,271행 / 공간 분석 1,303,240행
[11/24] 202311: 입력 1,085,399행 / 공간 분석 1,085,397행
[12/24] 202312: 입력 850,806행 / 공간 분석 850,800행
[13/24] 202401: 입력 866,055행 / 공간 분석 866,054행
[14/24] 202402: 입력 823,783행 / 공간 분석 823,776행
[15/24] 202403: 입력 1,059,382행 / 공간 분석 1,059,376행
[16/24] 202404: 입력 1,202,199행 / 공간 분석 1,202,199행
[17/24] 202405: 입력 1,196,745행 / 공간 분석 1,196,745행
[18/24] 202406: 입력 1,227,850행 / 공간 분석 1,227,850행
[19/24] 202407: 입력 1,109,957행 / 공간 분석 1,109,957행
[20/24] 202408: 입력 1,201,740행 / 공간 분석 1,201,740행
[21/24] 202409: 입력 1,153,678행 / 공간 분석 1,

,summary_item,value
0,분석 기간,2023-01-01 ~ 2024-12-31
1,원천 이용정보 행 수,79670647
2,유효 원천 이용정보 행 수,79670647
3,월별 집계 파일 수,24
4,전체 집계 행 수,26145337
5,전체 이용건수,88750388
6,공간 분석용 월별 파일 수,24
7,공간 분석 대상 대여소,2847
8,공간 분석 제외 대여소,18
9,공간 분석 대상 행 수,26127479



[검증 항목 전체]


,validation_item,status,passed,actual,expected,detail
0,필수 산출물 존재: 분석용 대여소 마스터,PASS,True,True,True,data\processed\station_master_clean.csv
1,필수 산출물 존재: 공간 분석 제외 대여소,PASS,True,True,True,data\processed\station_master_invalid_gps.csv
2,필수 산출물 존재: 이용정보 월별 품질표,PASS,True,True,True,data\processed\bike_usage_quality_by_month.csv
3,필수 산출물 존재: 2023년 12월 수동 교정 내역,PASS,True,True,True,data\processed\bike_usage_manual_corrections.csv
4,필수 산출물 존재: 대여소 좌표 변경 감사표,PASS,True,True,True,data\processed\station_coordinate_movement_aud...
5,필수 산출물 존재: 대여소 마스터 품질표,PASS,True,True,True,data\processed\station_master_quality_summary.csv
6,필수 산출물 존재: 공간 분석 월별 품질표,PASS,True,True,True,data\processed\spatial_demand_quality_by_month...
7,필수 산출물 존재: 공간 분석 제외 행 감사표,PASS,True,True,True,data\processed\station_hourly_excluded_from_sp...
8,원본 집계 월별 파일 수,PASS,True,24,24,
9,공간 분석용 월별 파일 수,PASS,True,24,24,



[저장 파일]
data\processed\preprocessing_final_validation.csv
data\processed\preprocessing_final_summary.csv
data\processed\preprocessing_monthly_final_audit.csv

모든 전처리 산출물의 무결성 검증을 통과했다.
01_data_quality_preprocessing.ipynb 실행이 완료되었다.
